# Part 01

## IMPORT LIBRARIES

In [ ]:
import os
import sys
import torch
import platform
import psutil
import warnings

## MAKE THE FILE DIRECTORY

In [ ]:
# Go one level up from notebooks/
BASE_DIR = os.path.abspath("..")

folders = [
    "notebooks",
    "src/data",
    "src/models",
    "src/utils",
    "src/training",
    "data/raw",
    "data/processed",
    "models",
    "outputs",
    "logs"
]

for folder in folders:
    os.makedirs(os.path.join(BASE_DIR, folder), exist_ok=True)

# Create requirements.txt if it doesn't exist
req_path = os.path.join(BASE_DIR, "requirements.txt")
if not os.path.exists(req_path):
    open(req_path, "w").close()

print("Project structure created successfully.")

## SET PROJECT ROOT & ENVIRONMENT

In [ ]:
# Notebook is in `StutterDetection/notebooks/`
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
print(f"📂 Project Root set to: {PROJECT_ROOT}")

# Add the src directory to Python path for module imports
SRC_PATH = os.path.join(PROJECT_ROOT, 'src')
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

# Create necessary directories (if they don't exist)
dirs_to_create = [
    os.path.join(PROJECT_ROOT, 'data', 'raw'),
    os.path.join(PROJECT_ROOT, 'data', 'processed'),
    os.path.join(PROJECT_ROOT, 'models', 'checkpoints'),
    os.path.join(PROJECT_ROOT, 'outputs', 'visualizations'),
    os.path.join(PROJECT_ROOT, 'logs'),
]

for dir_path in dirs_to_create:
    os.makedirs(dir_path, exist_ok=True)
    print(f"   Created/Verified: {dir_path}")

## HARDWARE CHECK (CPU on M4 MacBook)

In [ ]:
print(f"\nHARDWARE & PLATFORM CHECK:")
print(f"System: {platform.system()} {platform.machine()}")
print(f"Processor: {platform.processor()}")

print(f"\nPyTorch Device Configuration:")
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"MPS (Apple Silicon GPU) Available: {torch.backends.mps.is_available()}")

# Set device - Prioritize MPS (Apple GPU) for M4, else CPU.
# Use CPU for maximum compatibility during initial transfer.
# Later experiment with `device = torch.device('mps')`.
device = torch.device('cpu')
print(f"Using device: {device}")

# Display memory info (approximate for CPU)
ram_gb = psutil.virtual_memory().total / (1024**3)
print(f"Available RAM: {ram_gb:.2f} GB")

## IMPORT LIBRARIES (Local Environment)

In [ ]:
print("\nImporting core libraries...")

warnings.filterwarnings('ignore')

try:
    # Standard libraries
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    from pathlib import Path
    import json
    from datetime import datetime
    import random
    import glob
    from typing import List, Dict, Tuple, Optional

    # Audio processing
    import librosa
    import soundfile as sf

    # Deep learning
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
    import torchaudio

    # Hugging Face Transformers (for Wav2Vec2)
    from transformers import Wav2Vec2Model, Wav2Vec2Processor, Wav2Vec2FeatureExtractor

    # Scikit-learn
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import (
        precision_recall_fscore_support,
        classification_report,
        confusion_matrix,
        roc_auc_score
    )

    # Progress bar
    from tqdm.auto import tqdm

    print("All core libraries imported successfully.")

except ImportError as e:
    print(f"Critical Import Error: {e}")
    print("\nPlease ensure your local environment has the required packages.")
    # Optionally, re-raise to stop execution
    raise

## SET RANDOM SEEDS FOR REPRODUCIBILITY

In [ ]:
def set_seed(seed=42):
    # Set all random seeds for reproducibility
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    # If using CUDA or MPS, seed them as well
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Random seed set to {seed}")

set_seed(42)

## CONFIGURATION CLASS (ADAPTED FOR LOCAL)

In [ ]:
class Config:
    """
    Central configuration for the entire project.
    All paths are relative to the PROJECT_ROOT.
    """

    # Paths (Updated for local structure)
    PROJECT_ROOT = PROJECT_ROOT
    DATA_RAW = os.path.join(PROJECT_ROOT, 'data', 'raw')
    DATA_PROCESSED = os.path.join(PROJECT_ROOT, 'data', 'processed')
    CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, 'models', 'checkpoints')
    OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'outputs')
    LOG_DIR = os.path.join(PROJECT_ROOT, 'logs')

    # Dataset Path
    DATASET_PATH = DATA_RAW

    # Audio Processing
    SAMPLE_RATE = 16000  # Wav2Vec2 requirement
    CLIP_DURATION = 3.0  # seconds
    MAX_AUDIO_LENGTH = int(SAMPLE_RATE * CLIP_DURATION)

    # Model Parameters
    WAV2VEC2_MODEL = "facebook/wav2vec2-base-960h"
    HIDDEN_DIM = 768  # Wav2Vec2 base output dimension

    # STUTTER_TYPES and NUM_CLASSES will be populated later from data
    STUTTER_TYPES = []  # e.g., ['blocks', 'prolongations', 'repetitions']
    NUM_CLASSES = 0

    # Training Parameters (ADJUST FOR CPU)
    BATCH_SIZE = 4  # Reduced for CPU memory. Start small.
    LEARNING_RATE = 1e-4
    NUM_EPOCHS_STAGE1 = 5  # Reduced for initial testing
    NUM_EPOCHS_STAGE2 = 3
    WARMUP_STEPS = 50

    # Data Splits
    TRAIN_RATIO = 0.70
    VAL_RATIO = 0.15
    TEST_RATIO = 0.15

    # MIL & Pseudo-labeling
    MIL_POOL_TYPE = 'max'  # 'max', 'average', 'attention'
    CONFIDENCE_THRESHOLD = 0.8

    # Device
    DEVICE = device  # Use the device which set earlier ('cpu')

    def __init__(self):
        """Optional: Create directories on config init."""
        for attr in [self.DATA_RAW, self.DATA_PROCESSED, self.CHECKPOINT_DIR,
                     self.OUTPUT_DIR, self.LOG_DIR]:
            Path(attr).mkdir(parents=True, exist_ok=True)

    def save(self, path=None):
        """Save config to JSON file."""
        if path is None:
            path = os.path.join(self.CHECKPOINT_DIR, 'config.json')
        
        # Collect ALL attributes (class + instance)
        config_dict = {}
        
        # Get class attributes (excluding private/methods)
        for attr in dir(self):
            if not attr.startswith('_') and not callable(getattr(self, attr)):
                config_dict[attr] = getattr(self, attr)
        
        # Also get instance attributes
        config_dict.update(self.__dict__)
        
        # Handle non-serializable objects
        if 'DEVICE' in config_dict:
            config_dict['DEVICE'] = str(config_dict['DEVICE'])
        
        with open(path, 'w') as f:
            json.dump(config_dict, f, indent=2)
        print(f"Config saved to {path}")

    @classmethod
    def load(cls, path):
        """Load config from JSON file."""
        with open(path, 'r') as f:
            config_dict = json.load(f)
        config = cls()
        for k, v in config_dict.items():
            # Handle device string conversion
            if k == 'DEVICE':
                v = torch.device(v)
            setattr(config, k, v)
        return config

# Initialize config
config = Config()
print("\nConfiguration initialized")
print(f"   Dataset path set to: {config.DATASET_PATH}")
print(f"   Working directory: {os.getcwd()}")

## SIMPLE LOGGER (Adapted)

In [ ]:
class Logger:
    """Simple logger for tracking progress."""

    def __init__(self, log_dir):
        self.log_dir = Path(log_dir)
        self.log_dir.mkdir(parents=True, exist_ok=True)
        log_filename = f"log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
        self.log_file = self.log_dir / log_filename

    def log(self, message, level="INFO"):
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        log_message = f"[{timestamp}] [{level}] {message}"
        print(log_message)
        with open(self.log_file, 'a') as f:
            f.write(log_message + '\n')

logger = Logger(config.LOG_DIR)
logger.log("Environment setup completed successfully on MacBook.")
logger.log(f"Device in use: {config.DEVICE}")
logger.log(f"Batch size: {config.BATCH_SIZE}")

## DISPLAY FINAL SYSTEM INFO

In [ ]:
print("\n" + "="*60)
print("LOCAL SYSTEM INFORMATION")
print("="*60)
print(f"Project Root: {PROJECT_ROOT}")
print(f"PyTorch Version: {torch.__version__}")
print(f"Torchaudio Version: {torchaudio.__version__}")
print(f"Active Device: {config.DEVICE}")
print(f"NumPy Version: {np.__version__}")
print(f"Pandas Version: {pd.__version__}")
print(f"Librosa Version: {librosa.__version__}")
print(f"Transformers Version: {Wav2Vec2Processor.__module__.split('.')[0]}")
print("="*60)

# Part 02

## Dataset Exploration & Organization

In [ ]:
STUTTER_CLASS_MAPPING = {
    'Block': 'block',
    'Prolongation': 'prolongation', 
    'SoundRep': 'sound_repetition',
    'WordRep': 'word_repetition',
    'Interjection': 'interjection'
}

# Non-stutter classes to exclude
NON_STUTTER_CLASSES = ['Music', 'NoSpeech', 'NoStutteredWords', 
                       'PoorAudioQuality', 'NaturalPause', 'DifficultToUnderstand']

## Verify Dataset Path from Config

In [ ]:
# Use the path from our Config class
DATASET_PATH = config.DATASET_PATH
print(f"Dataset path from config: {DATASET_PATH}")

if not os.path.exists(DATASET_PATH):
    print(f"CRITICAL ERROR: Dataset path not found: {DATASET_PATH}")
    raise FileNotFoundError(f"Dataset not found at {DATASET_PATH}")
else:
    print(f"Dataset directory exists.")
    # List what's inside
    items = os.listdir(DATASET_PATH)
    print(f"   Found {len(items)} items in directory (first 10): {items[:10]}")

## Explore Dataset Structure

In [ ]:
def explore_dataset_structure(dataset_path):
    print("\n" + "="*60)
    print("DATASET STRUCTURE EXPLORATION")
    print("="*60)

    # Get all subdirectories (these are your classes)
    # Filter for directories only
    all_items = os.listdir(dataset_path)
    class_folders = []
    for item in all_items:
        item_path = os.path.join(dataset_path, item)
        if os.path.isdir(item_path):
            class_folders.append(item)

    class_folders.sort()

    if not class_folders:
        print("No class folders found! Check dataset structure.")
        return None, None

    print(f"\nFound {len(class_folders)} stutter event classes:")

    dataset_info = {}
    total_files = 0

    for idx, class_name in enumerate(class_folders):
        class_path = os.path.join(dataset_path, class_name)

        # Get all .wav files in this folder
        # Using glob for pattern matching
        wav_pattern = os.path.join(class_path, "*.wav")
        wav_files = glob.glob(wav_pattern)
        num_files = len(wav_files)
        total_files += num_files

        dataset_info[class_name] = {
            'path': class_path,
            'num_files': num_files,
            'files': wav_files[:100]  # Store only first 100 paths to avoid memory bloat
        }

        print(f"   {idx+1:2d}. {class_name:20s} - {num_files:5d} files")

    print(f"\nTotal .wav files found: {total_files:,}")

    # Simple quality check
    if total_files == 0:
        print("WARNING: No .wav files found in class folders.")
        print("Check if files have .wav extension or dataset is incomplete.")

    return class_folders, dataset_info

# Run exploration
class_folders, dataset_info = explore_dataset_structure(DATASET_PATH)

if class_folders:
    # Update config with discovered classes
    config.STUTTER_TYPES = class_folders
    config.NUM_CLASSES = len(class_folders)

    print(f"\n Configuration updated:")
    print(f"   STUTTER_TYPES: {config.STUTTER_TYPES}")
    print(f"   NUM_CLASSES: {config.NUM_CLASSES}")
    config.save()  # Save config to disk
else:
    print("Failed to load dataset structure. Exiting exploration.")
    # raise ValueError("Dataset structure invalid.")

def filter_stutter_classes_only(dataset_info):
    """
    Filter to keep only actual stuttering event classes.
    Based on SEP-28k documentation and PPRS.
    """
    STUTTER_CLASSES = ['block', 'prolongation', 'sound_repetition', 'word_repetition', 'interjection']
    
    # Map folder names to standardized names (case-insensitive)
    folder_to_standard = {}
    for folder_name in dataset_info.keys():
        folder_lower = folder_name.lower()
        for stutter_class in STUTTER_CLASSES:
            if stutter_class in folder_lower or folder_lower in stutter_class:
                folder_to_standard[folder_name] = stutter_class
                break
    
    print(f"\nIdentifying stutter classes...")
    print(f"Looking for: {STUTTER_CLASSES}")
    
    filtered_info = {}
    for folder_name, standard_name in folder_to_standard.items():
        filtered_info[standard_name] = dataset_info[folder_name]
        print(f"  ✓ {folder_name} → {standard_name}")
    
    # Check for missing classes
    missing = set(STUTTER_CLASSES) - set(filtered_info.keys())
    if missing:
        print(f"\nMissing expected stutter classes: {missing}")
    
    return filtered_info

# Modify the exploration section:
if class_folders:
    # Filter to stutter classes only
    print("\n" + "="*60)
    print("FILTERING TO STUTTERING EVENTS ONLY")
    print("="*60)
    
    stutter_dataset_info = filter_stutter_classes_only(dataset_info)
    
    # Update config with discovered STUTTER classes
    config.STUTTER_TYPES = list(stutter_dataset_info.keys())
    config.NUM_CLASSES = len(config.STUTTER_TYPES)
    
    print(f"\nConfiguration updated (Stutter Events Only):")
    print(f"   STUTTER_TYPES: {config.STUTTER_TYPES}")
    print(f"   NUM_CLASSES: {config.NUM_CLASSES}")
    
    # Use the filtered dataset_info for subsequent steps
    dataset_info = stutter_dataset_info

## Sample Audio Analysis

In [ ]:
def analyze_sample_audios(dataset_info, num_samples=2):
    """
    Load and analyze a few random audio files per class
    to verify integrity and understand properties.
    """
    print("\n" + "="*60)
    print("SAMPLE AUDIO ANALYSIS (First 3 Classes)")
    print("="*60)

    # Limit to first 3 classes for brevity
    classes_to_check = list(dataset_info.keys())[:3]

    for class_name in classes_to_check:
        info = dataset_info[class_name]
        print(f"\nClass: {class_name} ({info['num_files']} files)")

        # Get sample files (first few, not random, for reproducibility)
        sample_files = info['files'][:min(num_samples, len(info['files']))]

        for i, audio_file in enumerate(sample_files):
            try:
                # Load audio with librosa
                audio, sr = librosa.load(audio_file, sr=None, mono=True)
                duration = len(audio) / sr

                print(f"   Sample {i+1}: {os.path.basename(audio_file)}")
                print(f"      • Sample Rate: {sr} Hz")
                print(f"      • Duration: {duration:.3f} seconds")
                print(f"      • Shape: {audio.shape}")
                print(f"      • Min/Max: [{audio.min():.3f}, {audio.max():.3f}]")
                print(f"      • Mean Abs: {np.abs(audio).mean():.3f}")

                # Quick check if duration matches expected (3 seconds)
                expected_duration = config.CLIP_DURATION
                if abs(duration - expected_duration) > 0.5:
                    print(f"      Duration mismatch! Expected ~{expected_duration}s")

            except Exception as e:
                print(f"   Error loading {os.path.basename(audio_file)}: {str(e)[:100]}")

if dataset_info:
    analyze_sample_audios(dataset_info, num_samples=2)

## Create Dataset DataFrame

In [ ]:
def create_dataset_dataframe(dataset_info, save_csv=True):
    """
    Create a pandas DataFrame with all audio files and labels.
    (Master reference for the dataset).
    """
    print("\n" + "="*60)
    print("CREATING DATASET DATAFRAME")
    print("="*60)

    data = []
    for class_name, info in dataset_info.items():
        # Stored only first 100 files in info['files'] to avoid memory issues
        # Get all files for the complete DataFrame
        class_path = info['path']
        wav_pattern = os.path.join(class_path, "*.wav")
        all_wav_files = glob.glob(wav_pattern)

        for file_path in all_wav_files:
            data.append({
                'file_path': file_path,
                'filename': os.path.basename(file_path),
                'class': class_name,
                'class_label': config.STUTTER_TYPES.index(class_name)  # Integer label
            })

    df = pd.DataFrame(data)

    print(f"Created DataFrame with {len(df):,} audio files")
    print(f"   Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

    # Display class distribution
    print("\nCLASS DISTRIBUTION:")
    class_dist = df['class'].value_counts().sort_index()
    for class_name, count in class_dist.items():
        percentage = (count / len(df)) * 100
        print(f"   {class_name:20s}: {count:6d} files ({percentage:5.1f}%)")

    # Check for duplicates
    duplicate_files = df['filename'].duplicated().sum()
    if duplicate_files > 0:
        print(f"Found {duplicate_files} duplicate filenames (may be in different folders)")

    # Save to CSV for reference
    if save_csv:
        csv_path = os.path.join(config.DATA_RAW, 'sep28k_dataset_index.csv')
        df.to_csv(csv_path, index=False)
        print(f"\nSaved dataset index to: {csv_path}")

        # Also save a sample for quick inspection
        sample_csv = os.path.join(config.DATA_RAW, 'sep28k_sample_100.csv')
        df.sample(min(100, len(df))).to_csv(sample_csv, index=False)
        print(f"Saved 100-sample subset to: {sample_csv}")

    return df

if dataset_info:
    dataset_df = create_dataset_dataframe(dataset_info, save_csv=True)

    # Display informative sample
    print("\nDATASET DATAFRAME SAMPLE (first 10 rows):")
    print(dataset_df.head(10))

    # Basic statistics
    print("\nDATASET SUMMARY:")
    print(f"   Total files: {len(dataset_df):,}")
    print(f"   Classes: {len(dataset_df['class'].unique())}")
    print(f"   Unique speakers (estimated from filenames): {dataset_df['filename'].apply(lambda x: x.split('_')[0]).nunique() if '_' in dataset_df['filename'].iloc[0] else 'N/A'}")

    # Store in config for later use
    config.dataset_df_path = os.path.join(config.DATA_RAW, 'sep28k_dataset_index.csv')
else:
    print("Cannot create DataFrame without dataset_info.")
    dataset_df = None

## Quick Visualization of Class Distribution

In [ ]:
def visualize_class_distribution(df):
    """Create a simple bar plot of class distribution."""
    if df is None:
        print("No DataFrame to visualize.")
        return

    print("\n" + "="*60)
    print("VISUALIZING CLASS DISTRIBUTION")
    print("="*60)

    try:
        plt.figure(figsize=(12, 6))

        # Bar plot
        class_counts = df['class'].value_counts().sort_index()
        bars = plt.bar(class_counts.index, class_counts.values, color='skyblue', edgecolor='black')

        # Add counts on top of bars
        for bar in bars:
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2., height + 5,
                    f'{int(height):,}', ha='center', va='bottom', fontsize=9)

        plt.title('SEP-28k Dataset: Class Distribution', fontsize=14, fontweight='bold')
        plt.xlabel('Stutter Type', fontsize=12)
        plt.ylabel('Number of Audio Files', fontsize=12)
        plt.xticks(rotation=45, ha='right')
        plt.grid(axis='y', alpha=0.3, linestyle='--')

        # Add statistics text box
        stats_text = f"Total Files: {len(df):,}\nClasses: {len(class_counts)}"
        plt.text(0.02, 0.98, stats_text, transform=plt.gca().transAxes,
                fontsize=10, verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

        plt.tight_layout()

        # Save figure
        viz_path = os.path.join(config.OUTPUT_DIR, 'visualizations', 'class_distribution.png')
        plt.savefig(viz_path, dpi=150, bbox_inches='tight')
        print(f"Saved visualization to: {viz_path}")

        plt.show()

    except Exception as e:
        print(f"Could not create visualization: {e}")

# Run visualization if DataFrame exists
if dataset_df is not None:
    visualize_class_distribution(dataset_df)

## Final Dataset Status

In [ ]:
print("\n" + "="*60)
print("DATASET EXPLORATION COMPLETE")
print("="*60)
print(f"Dataset Location: {DATASET_PATH}")
print(f"Total Files Found: {len(dataset_df) if dataset_df is not None else 'N/A'}")
print(f"Classes Identified: {config.STUTTER_TYPES}")
print(f"Configuration Saved: {os.path.join(config.CHECKPOINT_DIR, 'config.json')}")
print(f"DataFrame Saved: {os.path.join(config.DATA_RAW, 'sep28k_dataset_index.csv')}")

logger.log(f"Dataset exploration completed. Found {config.NUM_CLASSES} classes with {len(dataset_df) if dataset_df is not None else 0} files.")

# Part 03

## DATA PREPROCESSING

In [ ]:
# Prepare SEP-28k data for model training with proper splits and normalization

print("\n" + "="*60)
print("VERIFYING DATASET FROM PREVIOUS PHASE")
print("="*60)

# Check if there is the dataset from Part 2
if 'dataset_df' not in locals() or dataset_df is None:
    print("ERROR: dataset_df not found. Please run Part 2 first.")
    raise ValueError("Dataset not loaded. Run Part 2: Dataset Exploration.")

print(f"Dataset loaded: {len(dataset_df):,} files")
print(f"Classes found: {config.STUTTER_TYPES}")
print(f"NUM_CLASSES: {config.NUM_CLASSES}")

# Create a working copy
df_working = dataset_df.copy()
print(f"Created working copy with {len(df_working):,} entries")

## ADD NEGATIVE/FLUENT CLASS IF MISSING

In [ ]:
print("\n" + "="*60)
print("CHECKING FOR FLUENT/NEGATIVE CLASS")
print("="*60)

# Check if there is a fluent/negative class
fluent_classes = ['no_stuttered_words', 'fluent', 'normal']
has_fluent_class = any(cls in [c.lower() for c in df_working['class'].unique()] 
                       for cls in fluent_classes)

if not has_fluent_class:
    print("No fluent/negative class detected.")
    print("   For binary/multi-class classification, you might want:")
    print("   1. Add 'no_stuttered_words' folder from SEP-28k")
    print("   2. Or treat one stutter type as negative during MIL training")
    
    # For now, proceed with just stutter classes
    # Noted this limitation
    logger.log("Warning: No fluent/negative class in dataset. Using stutter-only classification.")
else:
    print("Found fluent/negative class in dataset")

## EXTRACT SPEAKER INFORMATION (IMPROVED)

In [ ]:
def extract_speaker_info_sep28k(df):
    """
    Extract speaker/episode ID from SEP-28k filenames for speaker-disjoint splits.
    Handles different filename formats in SEP-28k.
    """
    print("\n" + "="*60)
    print("EXTRACTING SPEAKER INFORMATION")
    print("="*60)
    
    def parse_sep28k_filename(filename):
        """Parse speaker ID from SEP-28k filename with multiple format support"""
        try:
            # Remove .wav extension
            name = filename.replace('.wav', '')
            
            # SEP-28k has various formats:
            # 1. Podcast_Episode_Clip.wav
            # 2. Podcast_Episode_Speaker_Clip.wav
            # 3. Other variations
            
            parts = name.split('_')
            
            if len(parts) >= 3:
                # Most common: Podcast_Episode_Clip
                # Use Podcast_Episode as speaker ID for disjoint splits
                speaker_id = f"{parts[0]}_{parts[1]}"
            elif len(parts) == 2:
                speaker_id = f"{parts[0]}_unknown"
            else:
                speaker_id = "unknown_speaker"
            
            return speaker_id.lower()
            
        except Exception as e:
            print(f"   Warning parsing {filename}: {e}")
            return "unknown_speaker"
    
    # Apply parsing
    df['speaker_id'] = df['filename'].apply(parse_sep28k_filename)
    
    # Statistics
    num_speakers = df['speaker_id'].nunique()
    print(f"Found {num_speakers} unique speakers/episodes")
    
    # Check distribution
    print("\nSPEAKER DISTRIBUTION:")
    speaker_counts = df['speaker_id'].value_counts()
    
    # Show top 5 and bottom 5 speakers
    print("   Top 5 speakers by clip count:")
    for speaker, count in speaker_counts.head(5).items():
        percentage = (count / len(df)) * 100
        print(f"     {speaker:30s}: {count:4d} clips ({percentage:5.1f}%)")
    
    print("\n   Speakers with only 1 clip:")
    single_clip_speakers = sum(speaker_counts == 1)
    print(f"     {single_clip_speakers} speakers have only 1 clip")
    
    # Check for unknown speakers
    unknown_count = sum(df['speaker_id'] == 'unknown_speaker')
    if unknown_count > 0:
        print(f"Warning: {unknown_count} files have 'unknown_speaker' ID")
    
    # Save speaker statistics
    speaker_stats = {
        'total_speakers': int(num_speakers),
        'speakers_with_single_clip': int(single_clip_speakers),
        'unknown_speakers': int(unknown_count),
        'top_speakers': speaker_counts.head(20).to_dict()
    }
    
    stats_path = os.path.join(config.DATA_PROCESSED, 'speaker_stats.json')
    with open(stats_path, 'w') as f:
        json.dump(speaker_stats, f, indent=2)
    print(f"Saved speaker statistics to: {stats_path}")
    
    return df

# Apply speaker extraction
df_working = extract_speaker_info_sep28k(df_working)

## CREATE SPEAKER-DISJOINT SPLITS (OPTIMIZED)

In [ ]:
def create_speaker_disjoint_splits(df, config):
    """
    Create train/val/test splits ensuring no speaker overlap.
    """
    print("\n" + "="*60)
    print("CREATING SPEAKER-DISJOINT SPLITS")
    print("="*60)
    
    # Get unique speakers sorted by clip count (descending)
    speaker_counts = df['speaker_id'].value_counts()
    sorted_speakers = speaker_counts.index.tolist()
    
    print(f"Total unique speakers: {len(sorted_speakers)}")
    print(f"Split ratios: Train={config.TRAIN_RATIO:.0%}, "
          f"Val={config.VAL_RATIO:.0%}, Test={config.TEST_RATIO:.0%}")
    
    # Calculate number of speakers per split
    total_speakers = len(sorted_speakers)
    train_speaker_count = int(total_speakers * config.TRAIN_RATIO)
    val_speaker_count = int(total_speakers * config.VAL_RATIO)
    test_speaker_count = total_speakers - train_speaker_count - val_speaker_count
    
    print(f"Target speaker counts: Train={train_speaker_count}, "
          f"Val={val_speaker_count}, Test={test_speaker_count}")
    
    # Assign speakers to splits (simple round-robin for balance)
    train_speakers = []
    val_speakers = []
    test_speakers = []
    
    for i, speaker in enumerate(sorted_speakers):
        if len(train_speakers) < train_speaker_count:
            train_speakers.append(speaker)
        elif len(val_speakers) < val_speaker_count:
            val_speakers.append(speaker)
        else:
            test_speakers.append(speaker)
    
    print(f"\nActual speaker distribution:")
    print(f"   Train: {len(train_speakers)} speakers")
    print(f"   Val:   {len(val_speakers)} speakers")
    print(f"   Test:  {len(test_speakers)} speakers")
    
    # Assign splits to dataframe
    df['split'] = 'unassigned'
    
    # Use vectorized assignment for speed
    train_mask = df['speaker_id'].isin(train_speakers)
    val_mask = df['speaker_id'].isin(val_speakers)
    test_mask = df['speaker_id'].isin(test_speakers)
    
    df.loc[train_mask, 'split'] = 'train'
    df.loc[val_mask, 'split'] = 'val'
    df.loc[test_mask, 'split'] = 'test'
    
    # Verify no speaker overlap
    train_set = set(train_speakers)
    val_set = set(val_speakers)
    test_set = set(test_speakers)
    
    assert len(train_set.intersection(val_set)) == 0, "Train-Val speaker overlap!"
    assert len(train_set.intersection(test_set)) == 0, "Train-Test speaker overlap!"
    assert len(val_set.intersection(test_set)) == 0, "Val-Test speaker overlap!"
    print("Verified: No speaker overlap between splits")
    
    # Show clip distribution
    print(f"\nCLIP DISTRIBUTION PER SPLIT:")
    split_counts = df['split'].value_counts()
    for split in ['train', 'val', 'test']:
        count = split_counts.get(split, 0)
        percentage = (count / len(df)) * 100
        print(f"   {split:8s}: {count:6d} clips ({percentage:5.1f}%)")
    
    # Show class distribution per split (brief)
    print(f"\nCLASS BALANCE CHECK:")
    for split in ['train', 'val', 'test']:
        split_df = df[df['split'] == split]
        if len(split_df) > 0:
            class_dist = split_df['class'].value_counts()
            min_class = class_dist.min()
            max_class = class_dist.max()
            imbalance_ratio = max_class / min_class if min_class > 0 else float('inf')
            
            print(f"   {split.upper():6s}: {len(split_df):5d} clips, "
                  f"Classes: {len(class_dist)}, Imbalance: {imbalance_ratio:.1f}x")
    
    return df

# Create splits
df_working = create_speaker_disjoint_splits(df_working, config)

## AUDIO PREPROCESSOR (OPTIMIZED FOR CPU)

In [ ]:
class AudioPreprocessorCPU:
    """
    Audio preprocessing optimized for CPU execution.
    Lightweight and memory-efficient.
    """
    
    def __init__(self, config):
        self.target_sr = config.SAMPLE_RATE
        self.target_length = config.MAX_AUDIO_LENGTH
        self.config = config
        
        print(f"Initialized AudioPreprocessorCPU")
        print(f"   Sample Rate: {self.target_sr} Hz")
        print(f"   Target Length: {self.target_length} samples ({config.CLIP_DURATION}s)")
    
    def load_and_preprocess(self, file_path):
        """
        Load and preprocess audio file efficiently.
        Returns: preprocessed audio array or None if error
        """
        try:
            # Load audio (mono, target sample rate)
            audio, sr = librosa.load(
                file_path, 
                sr=self.target_sr, 
                mono=True,
                duration=config.CLIP_DURATION + 0.5  # Load slightly more than needed
            )
            
            # Apply preprocessing
            audio = self._normalize_audio(audio)
            audio = self._fix_length(audio)
            
            return audio
            
        except Exception as e:
            print(f"   Error loading {os.path.basename(file_path)[:30]}...: {str(e)[:50]}")
            return None
    
    def _fix_length(self, audio):
        """Pad or trim audio to target length efficiently."""
        current_len = len(audio)
        
        if current_len > self.target_length:
            # Trim from center (preserve content)
            start = (current_len - self.target_length) // 2
            return audio[start:start + self.target_length]
        elif current_len < self.target_length:
            # Pad with zeros (efficient)
            return np.pad(audio, (0, self.target_length - current_len), 
                         mode='constant')
        else:
            return audio
    
    def _normalize_audio(self, audio):
        """Normalize audio to consistent volume."""
        # Simple peak normalization
        peak = np.max(np.abs(audio))
        if peak > 0:
            audio = audio / peak * 0.9  # Scale to 90% of peak
        
        # Optional: RMS normalization (more computationally expensive)
        # rms = np.sqrt(np.mean(audio**2))
        # if rms > 0:
        #     audio = audio * (0.1 / rms)  # Normalize to RMS = 0.1
        
        return audio

# Initialize preprocessor
preprocessor = AudioPreprocessorCPU(config)

## TEST PREPROCESSING ON SAMPLES (LIGHTWEIGHT)

In [ ]:
def test_preprocessing_samples_light(df, preprocessor, num_samples=2):
    """Lightweight preprocessing test for CPU environment."""
    
    print("\n" + "="*60)
    print("TESTING PREPROCESSING")
    print("="*60)
    
    # Select samples from each split
    test_samples = []
    for split in ['train', 'val', 'test']:
        split_samples = df[df['split'] == split].head(num_samples)
        if len(split_samples) > 0:
            test_samples.append((split, split_samples))
    
    fig, axes = plt.subplots(len(test_samples), num_samples, 
                            figsize=(4*num_samples, 3*len(test_samples)))
    
    if len(test_samples) == 1:
        axes = axes.reshape(1, -1)
    
    for row_idx, (split, split_df) in enumerate(test_samples):
        for col_idx, (_, sample) in enumerate(split_df.iterrows()):
            if col_idx >= num_samples:
                break
            
            ax = axes[row_idx, col_idx] if len(test_samples) > 1 else axes[col_idx]
            
            # Preprocess audio
            audio = preprocessor.load_and_preprocess(sample['file_path'])
            
            if audio is not None:
                # Plot
                time = np.arange(len(audio)) / preprocessor.target_sr
                ax.plot(time, audio, 'b-', linewidth=0.5, alpha=0.7)
                ax.set_xlabel('Time (s)', fontsize=9)
                if col_idx == 0:
                    ax.set_ylabel('Amplitude', fontsize=9)
                
                title = f"{split.upper()}: {sample['class']}"
                ax.set_title(title[:30] + ('...' if len(title) > 30 else ''), 
                           fontsize=10, pad=5)
                
                # Add stats
                stats = (f"SR: {preprocessor.target_sr/1000:.0f}kHz\n"
                        f"Len: {len(audio)}\n"
                        f"μ: {audio.mean():.3f}\n"
                        f"σ: {audio.std():.3f}")
                ax.text(0.02, 0.98, stats, transform=ax.transAxes,
                       fontsize=8, va='top',
                       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
                
                ax.grid(True, alpha=0.3)
                ax.set_xlim(0, time[-1])
    
    plt.tight_layout()
    
    # Save
    viz_path = os.path.join(config.OUTPUT_DIR, 'visualizations', 'preprocessing_test.png')
    plt.savefig(viz_path, dpi=120, bbox_inches='tight')
    print(f"Saved preprocessing test to: {viz_path}")
    plt.show()

# Run lightweight test
test_samples_count = 2  # Reduced for CPU
test_preprocessing_samples_light(df_working, preprocessor, test_samples_count)

## CREATE LABEL ENCODINGS

In [ ]:
print("\n" + "="*60)
print("CREATING LABEL ENCODINGS")
print("="*60)

# Create label mappings
unique_classes = sorted(df_working['class'].unique())
label2idx = {cls: idx for idx, cls in enumerate(unique_classes)}
idx2label = {idx: cls for cls, idx in label2idx.items()}

# Add encoded labels to dataframe
df_working['label_idx'] = df_working['class'].map(label2idx)

print("LABEL MAPPING:")
print("   " + "-" * 45)
for cls, idx in label2idx.items():
    count = len(df_working[df_working['class'] == cls])
    percentage = (count / len(df_working)) * 100
    print(f"   {idx:2d} | {cls:20s} | {count:5d} ({percentage:5.1f}%)")

# Save mappings
mappings = {
    'label2idx': label2idx,
    'idx2label': idx2label,
    'classes': unique_classes,
    'num_classes': len(unique_classes),
    'description': 'SEP-28k stutter event class mappings'
}

mapping_path = os.path.join(config.DATA_PROCESSED, 'label_mappings.json')
with open(mapping_path, 'w') as f:
    json.dump(mappings, f, indent=2)
print(f"Saved label mappings to: {mapping_path}")

## SAVE PROCESSED DATASET (ESSENTIAL FILES ONLY)

In [ ]:
print("\n" + "="*60)
print("SAVING PROCESSED DATASET")
print("="*60)

# Save essential files only
essential_files = []

# 1. Main processed CSV
main_csv = os.path.join(config.DATA_PROCESSED, 'dataset_processed.csv')
df_working.to_csv(main_csv, index=False)
essential_files.append(main_csv)
print(f"Main dataset: {main_csv} ({len(df_working):,} rows)")

# 2. Split files
for split in ['train', 'val', 'test']:
    split_df = df_working[df_working['split'] == split]
    if len(split_df) > 0:
        split_path = os.path.join(config.DATA_PROCESSED, f'{split}_split.csv')
        split_df.to_csv(split_path, index=False)
        essential_files.append(split_path)
        print(f"{split:6s} split: {split_path} ({len(split_df):,} rows)")

# 3. Dataset statistics
stats = {
    'total_samples': len(df_working),
    'num_classes': len(unique_classes),
    'classes': unique_classes,
    'sample_rate': config.SAMPLE_RATE,
    'clip_duration': config.CLIP_DURATION,
    'splits': {
        split: int(len(df_working[df_working['split'] == split]))
        for split in ['train', 'val', 'test']
    },
    'num_speakers': int(df_working['speaker_id'].nunique()),
    'preprocessing_date': datetime.now().isoformat()
}

stats_path = os.path.join(config.DATA_PROCESSED, 'dataset_stats.json')
with open(stats_path, 'w') as f:
    json.dump(stats, f, indent=2)
essential_files.append(stats_path)
print(f"Statistics: {stats_path}")

print(f"\nEssential files saved in: {config.DATA_PROCESSED}")

## QUICK VISUALIZATION

In [ ]:
def create_lightweight_visualizations(df, config):
    """Create essential visualizations only."""
    
    print("\n" + "="*60)
    print("CREATING ESSENTIAL VISUALIZATIONS")
    print("="*60)
    
    # 1. Class Distribution
    plt.figure(figsize=(10, 6))
    
    class_counts = df['class'].value_counts()
    colors = plt.cm.Set3(np.linspace(0, 1, len(class_counts)))
    
    bars = plt.bar(range(len(class_counts)), class_counts.values, color=colors)
    plt.xticks(range(len(class_counts)), class_counts.index, rotation=45, ha='right')
    plt.ylabel('Number of Clips')
    plt.title('Class Distribution', fontweight='bold')
    plt.grid(axis='y', alpha=0.3)
    
    # Add counts on bars
    for bar, count in zip(bars, class_counts.values):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                f'{count:,}', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    
    # Save
    dist_path = os.path.join(config.OUTPUT_DIR, 'visualizations', 'class_distribution.png')
    plt.savefig(dist_path, dpi=120, bbox_inches='tight')
    print(f"Class distribution: {dist_path}")
    plt.show()
    
    # 2. Split Distribution
    plt.figure(figsize=(8, 6))
    
    split_counts = df['split'].value_counts()
    colors = ['#2ecc71', '#3498db', '#e74c3c']  # Green, Blue, Red
    
    plt.pie(split_counts.values, labels=split_counts.index, colors=colors,
           autopct='%1.1f%%', startangle=90)
    plt.title('Train/Val/Test Split Distribution', fontweight='bold')
    
    plt.tight_layout()
    
    # Save
    split_path = os.path.join(config.OUTPUT_DIR, 'visualizations', 'split_distribution.png')
    plt.savefig(split_path, dpi=120, bbox_inches='tight')
    print(f"Split distribution: {split_path}")
    plt.show()

# Create visualizations
create_lightweight_visualizations(df_working, config)

## FINAL SUMMARY & NEXT STEPS

In [ ]:
print("\n" + "="*60)
print("PREPROCESSING COMPLETE - FINAL SUMMARY")
print("="*60)

# Calculate final statistics
total_clips = len(df_working)
train_clips = len(df_working[df_working['split'] == 'train'])
val_clips = len(df_working[df_working['split'] == 'val'])
test_clips = len(df_working[df_working['split'] == 'test'])

print(f"""
DATA PREPROCESSING COMPLETED SUCCESSFULLY!

DATASET SUMMARY:
   • Total clips: {total_clips:,}
   • Stutter classes: {config.NUM_CLASSES}
   • Unique speakers: {df_working['speaker_id'].nunique():,}
   
DATA SPLITS (SPEAKER-DISJOINT):
   • TRAIN: {train_clips:,} clips ({train_clips/total_clips:.1%})
   • VAL:   {val_clips:,} clips ({val_clips/total_clips:.1%})
   • TEST:  {test_clips:,} clips ({test_clips/total_clips:.1%})

CLASSES ({config.NUM_CLASSES} total):""")

for cls in config.STUTTER_TYPES:
    count = len(df_working[df_working['class'] == cls])
    percentage = (count / total_clips) * 100
    print(f"   • {cls:20s}: {count:5d} clips ({percentage:5.1f}%)")

print(f"""
OUTPUT FILES SAVED:
   • {config.DATA_PROCESSED}/dataset_processed.csv
   • {config.DATA_PROCESSED}/train_split.csv
   • {config.DATA_PROCESSED}/val_split.csv
   • {config.DATA_PROCESSED}/test_split.csv
   • {config.DATA_PROCESSED}/label_mappings.json
   • {config.DATA_PROCESSED}/dataset_stats.json

VISUALIZATIONS:
   • {config.OUTPUT_DIR}/visualizations/class_distribution.png
   • {config.OUTPUT_DIR}/visualizations/split_distribution.png
   • {config.OUTPUT_DIR}/visualizations/preprocessing_test.png

IMPORTANT NOTES FOR CPU TRAINING:
   1. Dataset size: {total_clips:,} clips
   2. Consider using a subset for initial testing
   3. Batch size set to {config.BATCH_SIZE} (adjust if memory issues)
   4. Training will be slower on CPU vs GPU
""")

print("="*60)
print("PHASE 3 COMPLETE - READY FOR MODEL DEVELOPMENT")
print("="*60)

# Update config and log completion
config.processed_data_path = os.path.join(config.DATA_PROCESSED, 'dataset_processed.csv')
config.save()

logger.log(f"Phase 3 completed. Processed {total_clips:,} clips. "
          f"Train: {train_clips:,}, Val: {val_clips:,}, Test: {test_clips:,}")

# Part 04

## NEURO-SYMBOLIC MODEL ARCHITECTURE

In [ ]:
# Build hybrid model combining Neural Networks + Differentiable Acoustic Rules

#IMPORT SOME LIBRARIES

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor
import math
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {config.DEVICE}")

## LOAD WAV2VEC2 (WITH LOCAL CACHE)

In [ ]:
print("\n" + "="*60)
print("LOADING WAV2VEC2 FEATURE EXTRACTOR")
print("="*60)

model_name = config.WAV2VEC2_MODEL
print(f"Loading model: {model_name}")

try:
    # Use local files to avoid re-downloading
    from transformers import Wav2Vec2Config
    
    # First, try to load from cache or local
    print("   Looking for cached model...")
    
    # Load with local_files_only=True first
    try:
        feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(
            model_name,
            local_files_only=True
        )
        wav2vec2_model = Wav2Vec2Model.from_pretrained(
            model_name,
            local_files_only=True
        )
        print("   Loaded from local cache")
        
    except:
        # Download if not cached (will cache for future)
        print("   Downloading model (first time only)...")
        feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(model_name)
        wav2vec2_model = Wav2Vec2Model.from_pretrained(model_name)
        print("   Downloaded and cached")
    
    print(f"Wav2Vec2 loaded successfully")
    print(f"   Hidden size: {wav2vec2_model.config.hidden_size}")
    print(f"   Num layers: {wav2vec2_model.config.num_hidden_layers}")
    print(f"   Model size: ~{sum(p.numel() for p in wav2vec2_model.parameters()) / 1e6:.1f}M params")
    
    # Test with small input
    print("\n   Testing feature extractor with dummy audio...")
    dummy_audio = torch.randn(1, 16000)  # 1 second at 16kHz
    inputs = feature_extractor(dummy_audio, sampling_rate=16000, return_tensors="pt")
    print(f"   Input shape: {dummy_audio.shape} -> Features shape: {inputs.input_values.shape}")
    
except Exception as e:
    print(f"Error loading Wav2Vec2: {e}")
    raise

## DIFFERENTIABLE SOFT-RULE MODULE

In [ ]:
class DifferentiableSoftRulesCPU(nn.Module):
    """
    Rules (interpretable):
    1. Voicing Continuity: Detects prolongations (sustained sounds)
    2. Spectral Similarity: Detects repetitions (similar patterns)
    3. Silence Detection: Detects blocks (unusual pauses)
    """
    
    def __init__(self, feature_dim=768, num_rules=3):
        super().__init__()
        
        self.num_rules = num_rules
        self.feature_dim = feature_dim
        
        # Learnable parameters (initialized based on acoustic theory)
        # Voicing continuity (prolongations)
        self.voicing_threshold = nn.Parameter(torch.tensor(0.3))
        self.voicing_weight = nn.Parameter(torch.tensor(2.0))
        
        # Spectral similarity (repetitions)
        self.similarity_threshold = nn.Parameter(torch.tensor(0.6))
        self.similarity_weight = nn.Parameter(torch.tensor(2.0))
        
        # Silence detection (blocks)
        self.silence_threshold = nn.Parameter(torch.tensor(0.2))
        self.silence_weight = nn.Parameter(torch.tensor(2.0))
        
        # Lightweight projections (smaller for CPU)
        self.voicing_proj = nn.Linear(feature_dim, 32)  # Reduced from 64
        self.spectral_proj = nn.Linear(feature_dim, 32)
        self.silence_proj = nn.Linear(feature_dim, 32)
        
        print(f"Soft-Rule Module initialized with {num_rules} rules")
        print(f"   Projection dims: 32")
        
    def voicing_continuity_rule(self, features):
        """Detect sustained sounds (prolongations)"""
        batch_size, seq_len, dim = features.shape
        
        # Lightweight projection
        features_proj = F.relu(self.voicing_proj(features))  # Simpler activation
        
        # Efficient temporal difference
        if seq_len > 1:
            # Simple difference (faster than norm)
            diff = features_proj[:, 1:, :] - features_proj[:, :-1, :]
            # Mean absolute difference (faster than L2)
            change_rate = torch.mean(torch.abs(diff), dim=-1)
            change_rate = F.pad(change_rate, (0, 1), value=0)
        else:
            change_rate = torch.zeros(batch_size, seq_len, device=features.device)
        
        # Smooth sigmoid for differentiability
        continuity = torch.sigmoid(
            self.voicing_weight * (self.voicing_threshold - change_rate)
        )
        
        return continuity
    
    def spectral_similarity_rule(self, features):
        """Detect repetitions (similar patterns)"""
        batch_size, seq_len, dim = features.shape
        
        features_proj = F.relu(self.spectral_proj(features))
        
        if seq_len > 1:
            # Efficient similarity: dot product of normalized features
            features_norm = F.normalize(features_proj, p=2, dim=-1, eps=1e-8)
            
            # Compare with previous frame (simplified)
            curr = features_norm[:, 1:, :]
            prev = features_norm[:, :-1, :]
            similarity = torch.sum(curr * prev, dim=-1)
            similarity = F.pad(similarity, (1, 0), value=0)
        else:
            similarity = torch.zeros(batch_size, seq_len, device=features.device)
        
        repetition = torch.sigmoid(
            self.similarity_weight * (similarity - self.similarity_threshold)
        )
        
        return repetition
    
    def silence_detection_rule(self, features):
        """Detect silence/blocks (low energy)"""
        batch_size, seq_len, dim = features.shape
        
        features_proj = F.relu(self.silence_proj(features))
        
        # Efficient energy calculation
        energy = torch.mean(torch.abs(features_proj), dim=-1)
        
        # Normalize per sequence
        energy_min = energy.min(dim=1, keepdim=True)[0]
        energy_max = energy.max(dim=1, keepdim=True)[0] + 1e-8
        energy_norm = (energy - energy_min) / (energy_max - energy_min)
        
        silence = torch.sigmoid(
            self.silence_weight * (self.silence_threshold - energy_norm)
        )
        
        return silence
    
    def forward(self, features):
        """Apply all rules"""
        voicing = self.voicing_continuity_rule(features)
        similarity = self.spectral_similarity_rule(features)
        silence = self.silence_detection_rule(features)
        
        # Stack outputs
        outputs = torch.stack([voicing, similarity, silence], dim=-1)
        
        return outputs

# Test with small input
print("\nTesting soft rules...")
soft_rules = DifferentiableSoftRulesCPU(feature_dim=config.HIDDEN_DIM)
test_features = torch.randn(1, 50, config.HIDDEN_DIM)  # Smaller for CPU test
with torch.no_grad():
    rule_outputs = soft_rules(test_features)
print(f"   Input: {test_features.shape}")
print(f"   Output: {rule_outputs.shape} ✓")
print(f"   Memory: {rule_outputs.element_size() * rule_outputs.nelement() / 1024:.1f} KB")

## ADAPTIVE GATING NETWORK

In [ ]:
class AdaptiveGatingNetworkCPU(nn.Module):
    """
    Lightweight gating network.
    Combines neural and rule-based predictions.
    """
    
    def __init__(self, feature_dim=256, num_rules=3, num_classes=6):
        super().__init__()
        
        self.num_rules = num_rules
        self.num_classes = num_classes
        
        # Smaller network for CPU
        gate_input_dim = feature_dim + num_rules
        
        self.gate_network = nn.Sequential(
            nn.Linear(gate_input_dim, 64),  # Reduced
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, num_rules + 1),
            nn.Softmax(dim=-1)
        )
        
        # Simple rule-to-class mappings
        self.rule_classifiers = nn.ModuleList([
            nn.Linear(1, num_classes, bias=False)  # No bias for simplicity
            for _ in range(num_rules)
        ])
        
        print(f"Adaptive Gating Network (CPU optimized)")
        print(f"   Gate network: 64 --> {num_rules + 1} units")
        
    def forward(self, neural_features, rule_scores, neural_logits):
        """
        Combine predictions with learned weights.
        """
        batch_size, seq_len, _ = neural_features.shape
        
        # Concatenate for gating decision
        gate_input = torch.cat([neural_features, rule_scores], dim=-1)
        gate_weights = self.gate_network(gate_input)
        
        # Split weights
        neural_gate = gate_weights[:, :, 0:1]
        rule_gates = gate_weights[:, :, 1:]
        
        # Apply rule classifiers
        rule_logits_list = []
        for i, classifier in enumerate(self.rule_classifiers):
            rule_score = rule_scores[:, :, i:i+1]
            rule_logit = classifier(rule_score)
            rule_logits_list.append(rule_logit)
        
        # Combine with gates
        combined = neural_gate * neural_logits
        
        for i, rule_logit in enumerate(rule_logits_list):
            rule_gate = rule_gates[:, :, i:i+1]
            combined = combined + rule_gate * rule_logit
        
        return combined, gate_weights

# Test gating network
print("\nTesting gating network...")
gating_net = AdaptiveGatingNetworkCPU(
    feature_dim=128,  # Smaller for test
    num_rules=3,
    num_classes=config.NUM_CLASSES
)

test_neural = torch.randn(1, 50, 128)
test_rules = torch.randn(1, 50, 3)
test_logits = torch.randn(1, 50, config.NUM_CLASSES)

with torch.no_grad():
    combined, gates = gating_net(test_neural, test_rules, test_logits)

print(f"   Combined shape: {combined.shape}")
print(f"   Gate weights shape: {gates.shape}")
print(f"   Gates sum to 1: {gates[0, 0].sum().item():.3f}")

## TEMPORAL DETECTION HEAD

In [ ]:
class TemporalDetectionHeadCPU(nn.Module):
    """
    Lightweight temporal model for frame-level predictions.
    Uses GRU instead of LSTM for CPU efficiency.
    """
    
    def __init__(self, input_dim=768, hidden_dim=128, num_layers=1,
                 num_classes=6, dropout=0.2):
        super().__init__()
        
        # Use GRU (faster than LSTM on CPU)
        self.gru = nn.GRU(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )
        
        # Lightweight classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )
        
        print(f"Temporal Detection Head (CPU optimized)")
        print(f"   GRU: {num_layers} layer(s), hidden_dim={hidden_dim}")
        print(f"   Output: {num_classes} classes")
        
    def forward(self, features):
        """Forward pass"""
        gru_out, _ = self.gru(features)
        logits = self.classifier(gru_out)
        return logits, gru_out

# Test temporal head
print("\nTesting temporal head...")
temporal_head = TemporalDetectionHeadCPU(
    input_dim=config.HIDDEN_DIM,
    hidden_dim=128,  # Reduced for CPU
    num_classes=config.NUM_CLASSES
)

test_features = torch.randn(1, 50, config.HIDDEN_DIM)
with torch.no_grad():
    logits, gru_features = temporal_head(test_features)

print(f"   Input: {test_features.shape}")
print(f"   Logits: {logits.shape}")
print(f"   GRU features: {gru_features.shape}")

## COMPLETE MODEL

In [ ]:
class NeuroSymbolicStutterDetectorCPU(nn.Module):
    """
    Complete model optimized for CPU execution.
    Memory-efficient and faster inference.
    """
    
    def __init__(self, config, freeze_encoder=True):
        super().__init__()
        
        self.config = config
        
        # 1. Wav2Vec2 Encoder (frozen by default)
        print("Loading Wav2Vec2 encoder...")
        self.encoder = wav2vec2_model
        
        if freeze_encoder:
            for param in self.encoder.parameters():
                param.requires_grad = False
            print("   Encoder frozen (not trainable)")
        else:
            print("   Encoder trainable")
        
        # 2. Temporal Head (CPU optimized)
        self.temporal_head = TemporalDetectionHeadCPU(
            input_dim=config.HIDDEN_DIM,
            hidden_dim=128,  # Reduced
            num_classes=config.NUM_CLASSES,
            dropout=0.2
        )
        
        # 3. Soft Rules
        self.soft_rules = DifferentiableSoftRulesCPU(
            feature_dim=config.HIDDEN_DIM,
            num_rules=3
        )
        
        # 4. Gating Network
        self.gating_network = AdaptiveGatingNetworkCPU(
            feature_dim=256,  # GRU hidden_dim * 2
            num_rules=3,
            num_classes=config.NUM_CLASSES
        )
        
        print("\nModel components assembled")
        
    def forward(self, audio_input, return_details=False):
        """
        Forward pass with memory optimization.
        """
        batch_size = audio_input.shape[0]
        
        # 1. Extract features (with gradient management)
        if not self.encoder.training:
            with torch.no_grad():
                encoder_output = self.encoder(audio_input)
                features = encoder_output.last_hidden_state
        else:
            encoder_output = self.encoder(audio_input)
            features = encoder_output.last_hidden_state
        
        # 2. Temporal predictions
        neural_logits, temporal_features = self.temporal_head(features)
        
        # 3. Rule-based predictions
        rule_scores = self.soft_rules(features)
        
        # 4. Adaptive gating
        final_logits, gate_weights = self.gating_network(
            temporal_features, rule_scores, neural_logits
        )
        
        if return_details:
            details = {
                'wav2vec_features': features.detach(),
                'neural_logits': neural_logits.detach(),
                'rule_scores': rule_scores.detach(),
                'gate_weights': gate_weights.detach(),
                'temporal_features': temporal_features.detach()
            }
            return final_logits, details
        
        return final_logits
    
    def get_num_parameters(self):
        #Count parameters
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total = sum(p.numel() for p in self.parameters())
        return trainable, total


## CREATE AND TEST MODEL

In [ ]:
# Create model instance
model = NeuroSymbolicStutterDetectorCPU(config, freeze_encoder=True)

# Move to device
model = model.to(config.DEVICE)

# Count parameters
trainable_params, total_params = model.get_num_parameters()
print(f"\nMODEL SIZE SUMMARY:")
print(f"   Device: {config.DEVICE}")
print(f"   Trainable parameters: {trainable_params:,}")
print(f"   Total parameters: {total_params:,}")
print(f"   Frozen parameters: {total_params - trainable_params:,}")
print(f"   Approx. memory: {(total_params * 4) / (1024**2):.1f} MB (float32)")

# Test with VERY small input
print("\nTesting with small input...")
test_audio = torch.randn(1, 16000 // 2).to(config.DEVICE)  # 0.5 second
print(f"   Test audio shape: {test_audio.shape}")

with torch.no_grad():
    try:
        outputs, details = model(test_audio, return_details=True)
        print(f"Forward pass successful.")
        
        print(f"\nOUTPUT SHAPES:")
        print(f"   Final logits: {outputs.shape}")
        print(f"   Wav2Vec features: {details['wav2vec_features'].shape}")
        print(f"   Neural logits: {details['neural_logits'].shape}")
        print(f"   Rule scores: {details['rule_scores'].shape}")
        print(f"   Gate weights: {details['gate_weights'].shape}")
        
        # Analyze gate weights
        gate_avg = details['gate_weights'].mean(dim=(0, 1))
        print(f"\nAVERAGE GATE WEIGHTS:")
        print(f"   Neural: {gate_avg[0].item():.3f}")
        print(f"   Voicing rule: {gate_avg[1].item():.3f}")
        print(f"   Similarity rule: {gate_avg[2].item():.3f}")
        print(f"   Silence rule: {gate_avg[3].item():.3f}")
        
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            print(f"OUT OF MEMORY ERROR")
        else:
            print(f"Error: {e}")

## SAVE MODEL ARCHITECTURE

In [ ]:
# Save architecture summary
model_summary = {
    'architecture': 'NeuroSymbolicStutterDetectorCPU',
    'purpose': 'Interpretable weakly-supervised stutter detection',
    'components': {
        'encoder': {
            'type': 'Wav2Vec2 (frozen)',
            'model': config.WAV2VEC2_MODEL,
            'hidden_dim': config.HIDDEN_DIM
        },
        'temporal_head': {
            'type': 'Bidirectional GRU',
            'hidden_dim': 128,
            'num_layers': 1,
            'dropout': 0.2
        },
        'soft_rules': {
            'type': 'DifferentiableSoftRulesCPU',
            'num_rules': 3,
            'rules': ['voicing_continuity', 'spectral_similarity', 'silence_detection']
        },
        'gating_network': {
            'type': 'AdaptiveGatingNetworkCPU',
            'hidden_layers': [64],
            'output': 'softmax_gates'
        }
    },
    'parameters': {
        'trainable': trainable_params,
        'total': total_params,
        'frozen': total_params - trainable_params,
        'memory_estimate_mb': (total_params * 4) / (1024**2)
    },
    'optimizations': {
        'cpu_optimized': True,
        'reduced_dimensions': True,
        'gru_instead_of_lstm': True,
        'lightweight_projections': True
    },
    'created': datetime.now().isoformat()
}

# Save to file
summary_path = os.path.join(config.CHECKPOINT_DIR, 'model_architecture_cpu.json')
with open(summary_path, 'w') as f:
    json.dump(model_summary, f, indent=2)

print(f"Model summary saved to: {summary_path}")

# Save initial model weights
checkpoint_path = os.path.join(config.CHECKPOINT_DIR, 'model_initial.pth')
torch.save({
    'model_state_dict': model.state_dict(),
    'config': {k: v for k, v in config.__dict__.items() 
               if not k.startswith('_') and not callable(v)},
    'model_summary': model_summary,
    'timestamp': datetime.now().isoformat()
}, checkpoint_path)

print(f"Initial weights saved to: {checkpoint_path}")

## VERIFICATION

In [ ]:
print(f"""
NEURO-SYMBOLIC MODEL SUCCESSFULLY BUILT

MODEL SPECIFICATIONS:
   • Architecture: Wav2Vec2 + GRU + Soft-Rules + Gating
   • Total parameters: {total_params:,}
   • Trainable parameters: {trainable_params:,}
   • Memory footprint: ~{(total_params * 4) / (1024**2):.1f} MB

KEY INNOVATIONS:
   1. Differentiable Soft-Rules: 3 interpretable acoustic rules
   2. Adaptive Gating: Learns to combine neural & rule predictions
   3. CPU Optimization: GRU, reduced dimensions, efficient ops

IMPORTANT FOR CPU TRAINING:
   1. Batch size: {config.BATCH_SIZE} (consider reducing if OOM)
   2. Sequence length: 3 seconds @ 16kHz = 48,000 samples
   3. Feature sequence: ~100 frames after Wav2Vec2

📁 FILES CREATED:
   • {summary_path}
   • {checkpoint_path}
""")

# Log completion
logger.log(f"Phase 4 completed. Model built with {trainable_params:,} trainable params.")

# Part 05

## STARTING TRAINING PIPELINE

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import json
import math
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.auto import tqdm
import copy
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {config.DEVICE}")

# Clear any cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()

## DATASET CLASS

In [ ]:
class CPUSutterDataset(Dataset):
    """
    Memory-efficient dataset for CPU training.
    Loads audio on-demand to conserve RAM.
    """
    
    def __init__(self, csv_path, preprocessor, label2idx, max_samples=None):
        print(f"Loading dataset from: {csv_path}")
        
        # Load only necessary columns to save memory
        self.df = pd.read_csv(csv_path, usecols=['file_path', 'class'])
        
        # Limit samples for CPU testing
        if max_samples and len(self.df) > max_samples:
            self.df = self.df.sample(max_samples, random_state=42)
            print(f"   Limited to {max_samples} samples for CPU testing")
        
        self.preprocessor = preprocessor
        self.label2idx = label2idx
        
        # Calculate class weights (simplified)
        self._calculate_class_weights()
        
        print(f"Dataset loaded: {len(self.df)} samples")
        
    def _calculate_class_weights(self):
        """Lightweight class weight calculation"""
        class_counts = self.df['class'].value_counts()
        total = len(self.df)
        num_classes = len(self.label2idx)
        
        self.class_weights = {}
        for cls, idx in self.label2idx.items():
            if cls in class_counts:
                count = class_counts[cls]
                # Inverse frequency
                weight = total / (num_classes * count)
                self.class_weights[idx] = min(weight, 5.0)  # Cap at 5
            else:
                self.class_weights[idx] = 1.0
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        file_path = row['file_path']
        class_name = row['class']
        
        try:
            # Load and preprocess
            audio = self.preprocessor.load_and_preprocess(file_path)
            if audio is None:
                # Fallback: silent audio
                audio = np.zeros(self.preprocessor.target_length, dtype=np.float32)
        except Exception as e:
            # On error, use silent audio
            print(f"   Warning loading {file_path}: {str(e)[:50]}")
            audio = np.zeros(self.preprocessor.target_length, dtype=np.float32)
        
        # Convert to tensor (single precision for CPU)
        audio_tensor = torch.from_numpy(audio).float()
        
        # Multi-hot label
        clip_label = torch.zeros(len(self.label2idx), dtype=torch.float32)
        if class_name in self.label2idx:
            idx = self.label2idx[class_name]
            clip_label[idx] = 1.0
        
        # Class weight
        class_weight = torch.tensor(
            self.class_weights.get(self.label2idx.get(class_name, 0), 1.0),
            dtype=torch.float32
        )
        
        return {
            'audio': audio_tensor,
            'clip_label': clip_label,
            'class_name': class_name,
            'class_weight': class_weight
        }

## LOAD LABEL MAPPINGS

In [ ]:
print("\nLoading label mappings...")
mapping_path = os.path.join(config.DATA_PROCESSED, 'label_mappings.json')

try:
    with open(mapping_path, 'r') as f:
        mappings = json.load(f)
    
    label2idx = mappings['label2idx']
    idx2label = mappings['idx2label']
    classes = mappings['classes']
    
    print(f"Loaded {len(classes)} classes")
    print(f"   Classes: {classes}")
    
except FileNotFoundError:
    print(f"Label mappings not found at: {mapping_path}")
    print("   Please run Part 3 (Data Preprocessing) first.")
    raise

## INITIALIZE PREPROCESSOR (FROM PART 3)

In [ ]:
print("\nInitializing audio preprocessor...")

# Reuse or recreate preprocessor from Part 3
class AudioPreprocessorCPU:
    """Lightweight preprocessor for CPU"""
    
    def __init__(self, target_sr=16000, target_length=48000):
        self.target_sr = target_sr
        self.target_length = target_length
        
    def load_and_preprocess(self, file_path):
        """Load and normalize audio"""
        try:
            audio, sr = librosa.load(
                file_path,
                sr=self.target_sr,
                mono=True,
                duration=3.5  # Load slightly more than needed
            )
            
            # Normalize
            peak = np.max(np.abs(audio))
            if peak > 0:
                audio = audio / peak * 0.9
            
            # Pad/trim to target length
            if len(audio) > self.target_length:
                start = (len(audio) - self.target_length) // 2
                audio = audio[start:start + self.target_length]
            elif len(audio) < self.target_length:
                audio = np.pad(audio, (0, self.target_length - len(audio)))
                
            return audio
            
        except Exception as e:
            print(f"   Error loading {os.path.basename(file_path)}: {str(e)[:50]}")
            return None

preprocessor = AudioPreprocessorCPU(
    target_sr=config.SAMPLE_RATE,
    target_length=config.MAX_AUDIO_LENGTH
)

## CREATE DATASETS WITH CPU LIMITS

In [ ]:
print("\nCreating datasets...")

# Set sample limits for CPU testing
# TRAIN_SAMPLES_LIMIT = 100  
# VAL_SAMPLES_LIMIT = 20
# TEST_SAMPLES_LIMIT = 20

# TRAIN_SAMPLES_LIMIT = 500   
# VAL_SAMPLES_LIMIT = 100      
# TEST_SAMPLES_LIMIT = 100     
# BATCH_SIZE = 4               
# NUM_EPOCHS = 5  

TRAIN_SAMPLES_LIMIT = 2000
VAL_SAMPLES_LIMIT = 400
TEST_SAMPLES_LIMIT = 400
BATCH_SIZE = 8
NUM_EPOCHS = 10

print(f"Using sample limits for CPU testing:")
print(f"   Train: {TRAIN_SAMPLES_LIMIT} samples")
print(f"   Val: {VAL_SAMPLES_LIMIT} samples")
print(f"   Test: {TEST_SAMPLES_LIMIT} samples")
print("\n   Increase these limits gradually as verify the pipeline.")

try:
    train_dataset = CPUSutterDataset(
        csv_path=os.path.join(config.DATA_PROCESSED, 'train_split.csv'),
        preprocessor=preprocessor,
        label2idx=label2idx,
        max_samples=TRAIN_SAMPLES_LIMIT
    )
    
    val_dataset = CPUSutterDataset(
        csv_path=os.path.join(config.DATA_PROCESSED, 'val_split.csv'),
        preprocessor=preprocessor,
        label2idx=label2idx,
        max_samples=VAL_SAMPLES_LIMIT
    )
    
    test_dataset = CPUSutterDataset(
        csv_path=os.path.join(config.DATA_PROCESSED, 'test_split.csv'),
        preprocessor=preprocessor,
        label2idx=label2idx,
        max_samples=TEST_SAMPLES_LIMIT
    )
    
    print(f"\nDatasets created:")
    print(f"   Train: {len(train_dataset)} samples")
    print(f"   Val:   {len(val_dataset)} samples")
    print(f"   Test:  {len(test_dataset)} samples")
    
except FileNotFoundError as e:
    print(f"Dataset file not found: {e}")
    print("   Please ensure Part 3 completed successfully.")
    raise

## CREATE DATALOADERS

In [ ]:
print("\nCreating CPU-optimized DataLoaders...")

BATCH_SIZE = 8  # Start with 2
print(f"Using batch size: {BATCH_SIZE} (critical for CPU memory)")
print("   Increase gradually if memory allows.")

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,  # 0 for CPU stability
    pin_memory=False,  # False for CPU
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

print(f"\nDataLoaders created:")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches:   {len(val_loader)}")
print(f"   Test batches:  {len(test_loader)}")

## SIMPLE LOSS FUNCTION

In [ ]:
class SimpleMILoss(nn.Module):
    """
    Simple Multi-Instance Loss.
    Max pooling over time, then BCE.
    """
    
    def forward(self, logits, targets):
        """
        logits: (batch, seq_len, num_classes)
        targets: (batch, num_classes) multi-hot
        """
        # Max pooling over time dimension
        clip_logits, _ = torch.max(logits, dim=1)  # (batch, num_classes)
        
        # Binary cross entropy
        loss = F.binary_cross_entropy_with_logits(clip_logits, targets)
        
        return loss

criterion = SimpleMILoss()
print(f"\nLoss function initialized: Simple MIL (max pooling + BCE)")

## METRICS CALCULATION

In [ ]:
def calculate_cpu_metrics(predictions, targets, threshold=0.5):
    """Lightweight metrics calculation"""
    # Binary predictions
    pred_binary = (predictions > threshold).float()
    
    # Simple accuracy
    correct = (pred_binary == targets).float()
    accuracy = correct.mean(dim=0).mean().item()  # Mean over batch and classes
    
    return {'accuracy': accuracy}

## VERIFY CONFIG HAS NUM_CLASSES

In [ ]:
print("\nVerifying configuration...")
print(f"   NUM_CLASSES in config: {config.NUM_CLASSES}")
print(f"   Actual classes: {classes}")

if config.NUM_CLASSES != len(classes):
    print(f"Mismatch! Setting NUM_CLASSES from {config.NUM_CLASSES} to {len(classes)}")
    config.NUM_CLASSES = len(classes)

# Updated the model class to use config.NUM_CLASSES
print("\nCreating model with fixed class count...")

# Updated the model creation to ensure NUM_CLASSES is passed
from models.neurosymbolic import NeuroSymbolicStutterDetectorCPU

# Create a wrapper that ensures proper class count
class FixedNeuroSymbolicModel(NeuroSymbolicStutterDetectorCPU):
    def __init__(self, config, wav2vec2_model=None, freeze_encoder=True):
        # Ensure NUM_CLASSES is set
        if not hasattr(config, 'NUM_CLASSES') or config.NUM_CLASSES <= 0:
            config.NUM_CLASSES = len(label2idx)
            print(f"   Set NUM_CLASSES to {config.NUM_CLASSES}")
        
        super().__init__(config, wav2vec2_model, freeze_encoder)

# Create model
model = FixedNeuroSymbolicModel(config, freeze_encoder=True)

## DEBUG: Test model with proper output

In [ ]:
# Test with a single sample
model.eval()
with torch.no_grad():
    sample = train_dataset[0]
    audio = sample['audio'].unsqueeze(0).to(config.DEVICE)
    target = sample['clip_label']
    
    print(f"Input shape: {audio.shape}")
    print(f"Target shape: {target.shape}")
    print(f"Target values: {target}")
    
    # Forward pass
    output = model(audio)
    print(f"\nModel output shape: {output.shape}")
    
    # Max pooling (as done in loss)
    clip_logits, _ = torch.max(output, dim=1)
    print(f"After max pooling shape: {clip_logits.shape}")
    print(f"Logits: {clip_logits}")
    
    # Check if shapes match
    if clip_logits.shape[1] == target.shape[0]:
        print("Shapes match. Model is producing correct number of classes.")
        
        # Test loss calculation
        loss_fn = SimpleMILoss()
        loss = loss_fn(output, target.unsqueeze(0))
        print(f"Loss calculation works: {loss.item():.4f}")
    else:
        print(f"Shape mismatch. Model output: {clip_logits.shape[1]}, Target: {target.shape[0]}")
        print("   The model is not producing the correct number of class outputs.")
        
        # Debug: Check model components
        print("\nDebugging model components...")
        
        # Test Wav2Vec2 output
        encoder_output = model.encoder(audio)
        features = encoder_output.last_hidden_state
        print(f"   Wav2Vec2 features shape: {features.shape}")
        
        # Test temporal head
        neural_logits, temporal_features = model.temporal_head(features)
        print(f"   Temporal head logits shape: {neural_logits.shape}")
        print(f"   Expected: (1, seq_len, {config.NUM_CLASSES})")
        
        # Check if temporal_head has correct num_classes
        print(f"   Temporal head classifier output dim: {model.temporal_head.classifier[-1].out_features}")

## MODEL INITIALIZATION 

In [ ]:
print("\nCreating model architecture...")

# First import necessary components
import sys
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'src'))

try:
    # Try to import from the module
    from models.neurosymbolic import NeuroSymbolicStutterDetectorCPU
    print("Successfully imported model from module")
except ImportError:
    print("Module not found. Creating model inline...")
    
    print("   Creating simplified fallback model...")
    
    class FallbackModel(nn.Module):
        """Simple fallback model for testing"""
        def __init__(self, config):
            super().__init__()
            self.config = config
            # Simple linear model for testing
            self.linear = nn.Linear(config.MAX_AUDIO_LENGTH, config.NUM_CLASSES)
            
        def forward(self, x):
            # Simple pooling
            x_pooled = x.mean(dim=1)  # (batch, features)
            return self.linear(x_pooled).unsqueeze(1)  # Add sequence dim
    
    model = FallbackModel(config)
    print("   Using fallback model (for testing only)")
else:
    # Use the proper model
    model = NeuroSymbolicStutterDetectorCPU(config, freeze_encoder=True)
    print("Neuro-symbolic model created")

# Move to CPU
model = model.to(config.DEVICE)
print(f"Model moved to device: {config.DEVICE}")

# Count parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print(f"\nModel Parameters:")
print(f"   Trainable: {trainable_params:,}")
print(f"   Total:     {total_params:,}")
print(f"   Memory: ~{(total_params * 4) / (1024**2):.1f} MB")

# Quick test of the model
print("\nQuick model test...")
try:
    with torch.no_grad():
        # Get a single sample
        sample = train_dataset[0]
        audio = sample['audio'].unsqueeze(0).to(config.DEVICE)
        
        print(f"   Input shape: {audio.shape}")
        print(f"   Input range: [{audio.min():.3f}, {audio.max():.3f}]")
        
        # Forward pass
        output = model(audio)
        
        print(f"Forward pass successful!")
        print(f"   Output shape: {output.shape}")
        
        # Check for issues
        if torch.isnan(output).any():
            print("WARNING: Output contains NaN values")
        else:
            print("No NaN values")
            
except Exception as e:
    print(f"Model test failed: {e}")
    raise

## TRAINING FUNCTIONS

In [ ]:
def train_epoch_cpu(model, dataloader, criterion, optimizer, device):
    """Train one epoch on CPU"""
    model.train()
    total_loss = 0
    batch_count = 0
    
    pbar = tqdm(dataloader, desc="Training", leave=False)
    for batch in pbar:
        audio = batch['audio'].to(device)
        targets = batch['clip_label'].to(device)
        
        # Forward pass
        output = model(audio)
        loss = criterion(output, targets)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        
        # Gradient clipping for stability
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        # Update
        optimizer.step()
        
        # Track loss
        total_loss += loss.item()
        batch_count += 1
        
        pbar.set_postfix({'loss': loss.item()})
    
    avg_loss = total_loss / batch_count if batch_count > 0 else 0
    return avg_loss

def validate_cpu(model, dataloader, criterion, device):
    """Validate on CPU"""
    model.eval()
    total_loss = 0
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Validation", leave=False):
            audio = batch['audio'].to(device)
            targets = batch['clip_label'].to(device)
            
            output = model(audio)
            loss = criterion(output, targets)
            total_loss += loss.item()
            
            # Get predictions
            clip_logits, _ = torch.max(output, dim=1)
            probs = torch.sigmoid(clip_logits)
            
            all_preds.append(probs.cpu())
            all_targets.append(targets.cpu())
    
    if all_preds:
        all_preds = torch.cat(all_preds)
        all_targets = torch.cat(all_targets)
        metrics = calculate_cpu_metrics(all_preds, all_targets)
        avg_loss = total_loss / len(dataloader)
        return avg_loss, metrics
    else:
        return 0, {'accuracy': 0}

## EARLY STOPPING

In [ ]:
class SimpleEarlyStopping:
    def __init__(self, patience=3, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = float('inf')
        
    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            return False
        else:
            self.counter += 1
            return self.counter >= self.patience

## MAIN TRAINING FUNCTION

In [ ]:
def train_model_cpu(model, train_loader, val_loader, config, num_epochs=3):
    """Main training function"""
    print("\n" + "="*60)
    print("STARTING CPU TRAINING")
    print("="*60)
    
    # Optimizer
    optimizer = AdamW(
        model.parameters(),
        lr=1e-4,  # Conservative learning rate for CPU
        weight_decay=0.01
    )
    
    # Simple scheduler
    scheduler = CosineAnnealingLR(
        optimizer,
        T_max=num_epochs
    )
    
    # Early stopping
    early_stopping = SimpleEarlyStopping(patience=2)
    
    # Training history
    history = {
        'train_loss': [],
        'val_loss': [],
        'val_accuracy': []
    }
    
    best_val_loss = float('inf')
    best_model_state = None
    
    print(f"\nTraining Configuration:")
    print(f"   Epochs: {num_epochs}")
    print(f"   Learning Rate: {1e-4}")
    print(f"   Batch Size: {BATCH_SIZE}")
    print(f"   Device: {config.DEVICE}")
    print(f"   Early Stopping: Patience=2")
    
    try:
        for epoch in range(num_epochs):
            print(f"\n{'='*50}")
            print(f"EPOCH {epoch+1}/{num_epochs}")
            print(f"{'='*50}")
            
            # Training
            train_loss = train_epoch_cpu(
                model, train_loader, criterion, optimizer, config.DEVICE
            )
            
            # Validation
            val_loss, val_metrics = validate_cpu(
                model, val_loader, criterion, config.DEVICE
            )
            
            # Update scheduler
            scheduler.step()
            
            # Update history
            history['train_loss'].append(train_loss)
            history['val_loss'].append(val_loss)
            history['val_accuracy'].append(val_metrics['accuracy'])
            
            # Print results
            print(f"\nEpoch {epoch+1} Results:")
            print(f"   Training Loss:   {train_loss:.4f}")
            print(f"   Validation Loss: {val_loss:.4f}")
            print(f"   Validation Acc:  {val_metrics['accuracy']:.4f}")
            print(f"   Learning Rate:   {optimizer.param_groups[0]['lr']:.6f}")
            
            # Save checkpoint
            checkpoint_path = os.path.join(config.CHECKPOINT_DIR, f'cpu_checkpoint_epoch_{epoch+1}.pth')
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'train_loss': train_loss,
                'val_loss': val_loss,
                'val_accuracy': val_metrics['accuracy']
            }, checkpoint_path)
            print(f"   Checkpoint saved")
            
            # Save best model
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_model_state = copy.deepcopy(model.state_dict())
                
                best_model_path = os.path.join(config.CHECKPOINT_DIR, 'cpu_best_model.pth')
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': best_model_state,
                    'val_loss': best_val_loss,
                    'val_accuracy': val_metrics['accuracy']
                }, best_model_path)
                print(f"   NEW BEST MODEL! Val Loss: {best_val_loss:.4f}")
            
            # Early stopping
            if early_stopping(val_loss):
                print(f"\nEarly stopping triggered at epoch {epoch+1}")
                break
    
    except KeyboardInterrupt:
        print("\nTraining interrupted by user")
    except Exception as e:
        print(f"\nTraining error: {e}")
        import traceback
        traceback.print_exc()
    
    # Load best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"\nLoaded best model with Val Loss: {best_val_loss:.4f}")
    
    return model, history

## START CPU TRAINING

In [ ]:
print("\n" + "="*60)
print("STARTING CPU TRAINING")
print("="*60)

print("\n   Starting in 3 seconds...")

import time
time.sleep(3)

# Train with very few epochs initially
NUM_EPOCHS = 10  # Start with just 2 epochs
print(f"\nStarting training with {NUM_EPOCHS} epochs...")

trained_model, training_history = train_model_cpu(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    config=config,
    num_epochs=NUM_EPOCHS
)

## QUICK EVALUATION

In [ ]:
print("\n" + "="*60)
print("QUICK EVALUATION")
print("="*60)

def evaluate_cpu(model, dataloader, device):
    """Quick evaluation"""
    model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        print(f"\nEvaluating on test set...")
        for batch in tqdm(dataloader, desc="Testing", leave=False):
            audio = batch['audio'].to(device)
            targets = batch['clip_label']
            
            output = model(audio)
            clip_logits, _ = torch.max(output, dim=1)
            probs = torch.sigmoid(clip_logits)
            
            all_preds.append(probs.cpu())
            all_targets.append(targets.cpu())
    
    if all_preds:
        all_preds = torch.cat(all_preds)
        all_targets = torch.cat(all_targets)
        metrics = calculate_cpu_metrics(all_preds, all_targets)
        return metrics, all_preds, all_targets
    else:
        return {'accuracy': 0}, None, None

# Run evaluation
test_metrics, test_preds, test_targets = evaluate_cpu(
    trained_model, test_loader, config.DEVICE
)

print(f"\nTEST RESULTS:")
print(f"   Accuracy: {test_metrics['accuracy']:.4f}")

## SAVE RESULTS

In [ ]:
print("\n" + "="*60)
print("SAVING RESULTS")
print("="*60)

# Save final model
final_model_path = os.path.join(config.CHECKPOINT_DIR, 'cpu_final_model.pth')
torch.save({
    'model_state_dict': trained_model.state_dict(),
    'history': training_history,
    'config': {k: v for k, v in config.__dict__.items() 
               if not k.startswith('_') and not callable(v)},
    'label2idx': label2idx,
    'test_metrics': test_metrics,
    'training_params': {
        'batch_size': BATCH_SIZE,
        'epochs': NUM_EPOCHS,
        'train_samples': TRAIN_SAMPLES_LIMIT
    }
}, final_model_path)
print(f"Final model saved: {final_model_path}")

# Save training history
history_path = os.path.join(config.OUTPUT_DIR, 'cpu_training_history.json')
with open(history_path, 'w') as f:
    json.dump(training_history, f, indent=2)
print(f"Training history saved: {history_path}")

# Save test results
if test_preds is not None:
    results_path = os.path.join(config.OUTPUT_DIR, 'cpu_test_results.npz')
    np.savez(
        results_path,
        predictions=test_preds.numpy(),
        targets=test_targets.numpy(),
        metrics=test_metrics
    )
    print(f"Test results saved: {results_path}")

## PLOT TRAINING CURVES

In [ ]:
print("\n" + "="*60)
print("PLOTTING TRAINING CURVES")
print("="*60)

def plot_cpu_curves(history):
    """Plot simple training curves"""
    if not history['train_loss']:
        print("   No training history to plot")
        return None
    
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    
    epochs = range(1, len(history['train_loss']) + 1)
    
    # Loss curve
    axes[0].plot(epochs, history['train_loss'], 'b-', label='Train', linewidth=2)
    axes[0].plot(epochs, history['val_loss'], 'r-', label='Validation', linewidth=2)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training Loss')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    # Accuracy curve
    if history['val_accuracy']:
        axes[1].plot(epochs, history['val_accuracy'], 'g-', linewidth=2)
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Accuracy')
        axes[1].set_title('Validation Accuracy')
        axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    
    # Save
    curves_path = os.path.join(config.OUTPUT_DIR, 'visualizations', 'cpu_training_curves.png')
    plt.savefig(curves_path, dpi=150, bbox_inches='tight')
    plt.show()
    
    return curves_path

curves_path = plot_cpu_curves(training_history)
if curves_path:
    print(f"Training curves saved: {curves_path}")

## SUMMARY

In [ ]:
print("\n" + "="*60)
print("TRAINING SUMMARY")
print("="*60)

print(f"""
TRAINING COMPLETED

RESULTS SUMMARY:
   • Training Epochs: {len(training_history['train_loss'])}
   • Best Val Loss: {min(training_history['val_loss']) if training_history['val_loss'] else 'N/A':.4f}
   • Test Accuracy: {test_metrics['accuracy']:.4f}

FILES SAVED:
   • Model: {final_model_path}
   • History: {history_path}
   • Test results: {results_path if 'results_path' in locals() else 'N/A'}
   • Training curves: {curves_path if curves_path else 'N/A'}

🔍 PERFORMANCE ANALYSIS:
""")

if test_metrics['accuracy'] > 0.6:
    print("   EXCELLENT: Model is learning well on CPU")
elif test_metrics['accuracy'] > 0.4:
    print("   GOOD: Model is learning, can increase dataset size")
elif test_metrics['accuracy'] > 0.2:
    print("   FAIR: Model shows some learning, needs tuning")
else:
    print("   POOR: Model needs significant improvement")

print("="*60)
print("PHASE 5 COMPLETE - PIPELINE VERIFIED.")
print("="*60)

# Final verification
print("\nFinal model verification...")
trained_model.eval()
with torch.no_grad():
    sample = train_dataset[0]
    audio = sample['audio'].unsqueeze(0).to(config.DEVICE)
    output = trained_model(audio)
    clip_logits, _ = torch.max(output, dim=1)
    probs = torch.sigmoid(clip_logits)
    
    print(f"Sample prediction:")
    for idx, cls in enumerate(classes):
        if idx < 5:  # Show first 5 classes
            print(f"  {cls}: {probs[0, idx].item():.3f}")

print("\nAll checks passed. Pipeline is working on CPU.")

# Part 06

## PSEUDO-LABEL GENERATION

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import json
import os
import warnings
warnings.filterwarnings('ignore')

print(f"Device: {config.DEVICE}")

## LOAD TRAINED CPU MODEL

In [ ]:
print("\n" + "="*60)
print("LOADING TRAINED CPU MODEL")
print("="*60)

# Find the CPU-trained model first
cpu_model_path = os.path.join(config.CHECKPOINT_DIR, 'cpu_final_model.pth')

if os.path.exists(cpu_model_path):
    print(f"Found CPU-trained model: {cpu_model_path}")
    stage1_model_path = cpu_model_path
else:
    # Fallback to any model
    print(f"CPU model not found, looking for alternatives...")
    checkpoints = [f for f in os.listdir(config.CHECKPOINT_DIR) 
                   if f.endswith('.pt') or f.endswith('.pth')]
    if checkpoints:
        stage1_model_path = os.path.join(config.CHECKPOINT_DIR, checkpoints[-1])
        print(f"Using: {checkpoints[-1]}")
    else:
        print("No model checkpoints found")
        raise FileNotFoundError("No model checkpoints found")

# Load checkpoint with CPU device
print(f"\nLoading checkpoint from: {stage1_model_path}")
try:
    checkpoint = torch.load(stage1_model_path, map_location=torch.device('cpu'))
    print("Checkpoint loaded successfully")
except Exception as e:
    print(f"Error loading checkpoint: {e}")
    raise

# Initialize model
print("\nInitializing model...")
try:
    from models.neurosymbolic import NeuroSymbolicStutterDetectorCPU
    pseudo_label_model = NeuroSymbolicStutterDetectorCPU(config, freeze_encoder=True)
    
    # Load state dict
    if 'model_state_dict' in checkpoint:
        pseudo_label_model.load_state_dict(checkpoint['model_state_dict'])
        print("Model weights loaded")
    else:
        # Try direct loading
        pseudo_label_model.load_state_dict(checkpoint)
        print("Model weights loaded (direct)")
    
    pseudo_label_model = pseudo_label_model.to(config.DEVICE)
    pseudo_label_model.eval()
    print("Model initialized and set to evaluation mode")
    
except Exception as e:
    print(f"Error initializing model: {e}")
    raise

# Display model info
print(f"\nModel Information:")
trainable_params = sum(p.numel() for p in pseudo_label_model.parameters() if p.requires_grad)
print(f"   Trainable parameters: {trainable_params:,}")
print(f"   Device: {config.DEVICE}")

## LOAD DATASET WITH FILE PATHS

In [ ]:
print("\n" + "="*60)
print("LOADING DATASET WITH FILE PATHS")
print("="*60)

# Load the training CSV to get file paths
train_csv_path = os.path.join(config.DATA_PROCESSED, 'train_split.csv')
print(f"Loading dataset from: {train_csv_path}")

try:
    train_df = pd.read_csv(train_csv_path)
    print(f"Loaded {len(train_df)} samples")
    
    # Limit samples for CPU testing
    MAX_SAMPLES = 2000  # Start with 50 samples
    if len(train_df) > MAX_SAMPLES:
        train_df = train_df.head(MAX_SAMPLES)
        print(f"Limited to {MAX_SAMPLES} samples for CPU testing")
    
    # Display sample info
    print(f"\nDataset Info:")
    print(f"   Columns: {train_df.columns.tolist()}")
    print(f"   Classes: {train_df['class'].unique().tolist()}")
    print(f"   First file: {train_df.iloc[0]['file_path']}")
    
except Exception as e:
    print(f"Error loading dataset: {e}")
    raise

## REINITIALIZE AUDIO PREPROCESSOR

In [ ]:
print("\n" + "="*60)
print("REINITIALIZING AUDIO PREPROCESSOR")
print("="*60)

class SimpleAudioPreprocessor:
    #Simple audio preprocessor for CPU
    
    def __init__(self, target_sr=16000, target_length=48000):
        self.target_sr = target_sr
        self.target_length = target_length
        
    def load_and_preprocess(self, file_path):
        #Load and normalize audio
        try:
            import librosa
            audio, sr = librosa.load(
                file_path,
                sr=self.target_sr,
                mono=True,
                duration=3.0
            )
            
            # Normalize
            peak = np.max(np.abs(audio))
            if peak > 0:
                audio = audio / peak * 0.9
            
            # Pad/trim to target length
            if len(audio) > self.target_length:
                start = (len(audio) - self.target_length) // 2
                audio = audio[start:start + self.target_length]
            elif len(audio) < self.target_length:
                audio = np.pad(audio, (0, self.target_length - len(audio)))
                
            return audio
            
        except Exception as e:
            print(f"   Error loading {os.path.basename(file_path)}: {str(e)[:50]}")
            return None

preprocessor = SimpleAudioPreprocessor(
    target_sr=config.SAMPLE_RATE,
    target_length=config.MAX_AUDIO_LENGTH
)

# Test preprocessor
test_file = train_df.iloc[0]['file_path']
test_audio = preprocessor.load_and_preprocess(test_file)
if test_audio is not None:
    print(f"Preprocessor test: {os.path.basename(test_file)} → {len(test_audio)} samples")
else:
    print(f"Preprocessor test failed")

## CPU-OPTIMIZED PSEUDO-LABEL GENERATOR

In [ ]:
print("\n" + "="*60)
print("SETTING UP PSEUDO-LABEL GENERATION")
print("="*60)

class CPUPseudoLabelGenerator:
    """
    Generates frame-level pseudo-labels with CPU optimization
    """
    
    def __init__(self, model, confidence_threshold=0.6, min_event_length=5):
        """
        Args:
            model: Trained Stage 1 model
            confidence_threshold: Minimum probability
            min_event_length: Minimum frames for event
        """
        self.model = model
        self.confidence_threshold = confidence_threshold
        self.min_event_length = min_event_length
        
        print(f"CPU Pseudo-Label Generator initialized")
        print(f"Confidence threshold: {confidence_threshold}")
        print(f"Min event length: {min_event_length} frames")
    
    @torch.no_grad()
    def generate_frame_predictions(self, audio_tensor):
        """
        Generate frame-level predictions (CPU optimized)
        """
        # Ensure batch dimension
        if audio_tensor.dim() == 1:
            audio_tensor = audio_tensor.unsqueeze(0)
        
        # Get frame-level logits
        try:
            frame_logits = self.model(audio_tensor)  # (1, seq_len, num_classes)
        except RuntimeError as e:
            if "out of memory" in str(e):
                print("   OOM - using smaller sequence")
                # Trim audio if too long
                audio_tensor = audio_tensor[:, :24000]  # 1.5 seconds
                frame_logits = self.model(audio_tensor)
            else:
                raise
        
        # Convert to probabilities
        frame_probs = torch.sigmoid(frame_logits).squeeze(0)  # (seq_len, num_classes)
        
        # Apply confidence threshold
        frame_labels = (frame_probs >= self.confidence_threshold).float()
        
        # Clip-level confidence (max pooling)
        clip_probs, _ = torch.max(frame_probs, dim=0)
        
        return frame_probs.cpu(), frame_labels.cpu(), clip_probs.cpu()
    
    def refine_pseudo_labels(self, frame_labels):
        """
        Refine pseudo-labels (CPU optimized with numpy)
        """
        if isinstance(frame_labels, torch.Tensor):
            frame_labels_np = frame_labels.numpy()
        else:
            frame_labels_np = frame_labels
        
        seq_len, num_classes = frame_labels_np.shape
        refined_labels = frame_labels_np.copy()
        
        for class_idx in range(num_classes):
            class_labels = frame_labels_np[:, class_idx]
            
            # Find runs of 1s (numpy optimized)
            diff = np.diff(np.concatenate([[0], class_labels, [0]]))
            run_starts = np.where(diff == 1)[0]
            run_ends = np.where(diff == -1)[0]
            
            # Remove short runs
            for start, end in zip(run_starts, run_ends):
                if (end - start) < self.min_event_length:
                    refined_labels[start:end, class_idx] = 0
        
        return torch.from_numpy(refined_labels) if isinstance(frame_labels, torch.Tensor) else refined_labels
    
    def generate_boundary_labels(self, frame_labels):
        """
        Generate onset/offset boundaries (CPU optimized)
        """
        if isinstance(frame_labels, torch.Tensor):
            frame_labels_np = frame_labels.numpy()
        else:
            frame_labels_np = frame_labels
        
        seq_len, num_classes = frame_labels_np.shape
        boundaries = []
        
        for class_idx in range(num_classes):
            class_labels = frame_labels_np[:, class_idx]
            
            # Find transitions
            diff = np.diff(np.concatenate([[0], class_labels, [0]]))
            onsets = np.where(diff == 1)[0]
            offsets = np.where(diff == -1)[0]
            
            # Create boundaries
            for onset, offset in zip(onsets, offsets):
                duration = offset - onset
                if duration >= self.min_event_length:
                    boundaries.append({
                        'class_idx': int(class_idx),
                        'onset_frame': int(onset),
                        'offset_frame': int(offset),
                        'duration_frames': int(duration)
                    })
        
        return boundaries

# Initialize generator
pseudo_gen = CPUPseudoLabelGenerator(
    model=pseudo_label_model,
    confidence_threshold=0.6,
    min_event_length=5
)

## GENERATE PSEUDO-LABELS

In [ ]:
print("\n" + "="*60)
print("GENERATING PSEUDO-LABELS")
print("="*60)

def generate_pseudo_labels_from_dataframe(df, preprocessor, generator, label2idx, idx2label):
    """
    Generate pseudo-labels directly from the dataframe
    """
    print(f"\nProcessing {len(df)} samples...")
    
    pseudo_labels_dict = {}
    statistics = {
        'total_samples': 0,
        'high_confidence_samples': 0,
        'frames_with_labels': 0,
        'total_frames': 0,
        'events_detected': 0,
        'class_distribution': {i: 0 for i in range(config.NUM_CLASSES)},
        'confidence_distribution': {'high': 0, 'medium': 0, 'low': 0}
    }
    
    pbar = tqdm(df.iterrows(), total=len(df), desc="Generating pseudo-labels")
    
    for idx, row in pbar:
        try:
            file_path = row['file_path']
            class_name = row['class']
            
            # Load and preprocess audio
            audio = preprocessor.load_and_preprocess(file_path)
            if audio is None:
                continue
            
            audio_tensor = torch.from_numpy(audio).float()
            
            # Generate predictions
            frame_probs, frame_labels, clip_probs = generator.generate_frame_predictions(audio_tensor)
            
            # Refine labels
            refined_labels = generator.refine_pseudo_labels(frame_labels)
            
            # Generate boundaries
            boundaries = generator.generate_boundary_labels(refined_labels)
            
            # Store data
            pseudo_labels_dict[file_path] = {
                'frame_labels': refined_labels.numpy().astype(np.uint8),
                'clip_probs': clip_probs.numpy().astype(np.float16),
                'boundaries': boundaries,
                'original_class': class_name,
                'original_class_idx': label2idx.get(class_name, -1)
            }
            
            # Update statistics
            statistics['total_samples'] += 1
            
            seq_len = refined_labels.shape[0]
            statistics['total_frames'] += seq_len
            
            labeled_frames = refined_labels.sum().item()
            statistics['frames_with_labels'] += labeled_frames
            
            num_events = len(boundaries)
            statistics['events_detected'] += num_events
            
            for boundary in boundaries:
                class_idx = boundary['class_idx']
                statistics['class_distribution'][class_idx] += 1
            
            # Confidence analysis
            max_prob = clip_probs.max().item()
            if max_prob >= 0.8:
                statistics['confidence_distribution']['high'] += 1
            elif max_prob >= 0.5:
                statistics['confidence_distribution']['medium'] += 1
            else:
                statistics['confidence_distribution']['low'] += 1
            
            # Update progress
            coverage = (statistics['frames_with_labels'] / max(statistics['total_frames'], 1)) * 100
            pbar.set_postfix({
                'samples': statistics['total_samples'],
                'coverage': f'{coverage:.1f}%',
                'events': statistics['events_detected']
            })
            
        except Exception as e:
            print(f"\nError processing {os.path.basename(file_path)}: {str(e)[:50]}")
            continue
    
    # Calculate final statistics
    if statistics['total_frames'] > 0:
        statistics['frame_coverage'] = (statistics['frames_with_labels'] / 
                                       statistics['total_frames'] * 100)
    else:
        statistics['frame_coverage'] = 0.0
    
    statistics['avg_events_per_sample'] = (statistics['events_detected'] / 
                                          max(statistics['total_samples'], 1))
    
    return pseudo_labels_dict, statistics

# Load label mappings
mapping_path = os.path.join(config.DATA_PROCESSED, 'label_mappings.json')
with open(mapping_path, 'r') as f:
    mappings = json.load(f)
    label2idx = mappings['label2idx']
    idx2label = mappings['idx2label']

# Generate pseudo-labels
train_pseudo_path = os.path.join(config.DATA_PROCESSED, 'cpu_train_pseudo_labels.npz')

train_pseudo_labels, train_stats = generate_pseudo_labels_from_dataframe(
    df=train_df,
    preprocessor=preprocessor,
    generator=pseudo_gen,
    label2idx=label2idx,
    idx2label=idx2label
)

print(f"\nGenerated pseudo-labels for {len(train_pseudo_labels)}/{len(train_df)} samples")

## SAVE PSEUDO-LABELS

In [ ]:
print("\n" + "="*60)
print("SAVING PSEUDO-LABELS")
print("="*60)

# Save in compressed format
print(f"\nSaving to: {train_pseudo_path}")

save_data = {}
for file_path, data in train_pseudo_labels.items():
    # Use short filename as key
    short_name = os.path.basename(file_path).replace('.wav', '')
    
    save_data[short_name] = {
        'labels': data['frame_labels'],
        'probs': data['clip_probs'],
        'class': data['original_class'],
        'class_idx': data['original_class_idx']
    }
    
    if data['boundaries']:
        boundaries_array = []
        for b in data['boundaries']:
            boundaries_array.append([
                b['class_idx'],
                b['onset_frame'],
                b['offset_frame'],
                b['duration_frames']
            ])
        save_data[short_name]['boundaries'] = np.array(boundaries_array, dtype=np.uint16)

np.savez_compressed(
    train_pseudo_path,
    pseudo_labels=save_data,
    statistics=train_stats,
    config={
        'confidence_threshold': pseudo_gen.confidence_threshold,
        'min_event_length': pseudo_gen.min_event_length,
        'num_samples': len(train_pseudo_labels),
        'num_classes': config.NUM_CLASSES
    }
)

print(f"Pseudo-labels saved ({os.path.getsize(train_pseudo_path) / 1024:.1f} KB)")

## ANALYZE RESULTS

In [ ]:
print("\n" + "="*60)
print("PSEUDO-LABEL QUALITY ANALYSIS")
print("="*60)

if train_stats['total_samples'] > 0:
    print(f"\nGeneration Statistics:")
    print(f"   Samples processed: {train_stats['total_samples']:,}")
    print(f"   Frame coverage: {train_stats['frame_coverage']:.2f}%")
    print(f"   Events detected: {train_stats['events_detected']:,}")
    print(f"   Avg events/sample: {train_stats['avg_events_per_sample']:.2f}")
    
    print(f"\nConfidence Distribution:")
    for level, count in train_stats['confidence_distribution'].items():
        percentage = count / train_stats['total_samples'] * 100
        print(f"   {level:6s}: {count:4d} samples ({percentage:5.1f}%)")
    
    print(f"\nEvents per Class:")
    for class_idx, count in train_stats['class_distribution'].items():
        class_name = idx2label[str(class_idx)]
        percentage = (count / max(train_stats['events_detected'], 1)) * 100
        print(f"   {class_name:20s}: {count:5d} ({percentage:5.1f}%)")
else:
    print(f"No samples were processed successfully!")
    print("   Check audio loading and model predictions.")

## QUICK QUALITY CHECK

In [ ]:
print("\n" + "="*60)
print("QUALITY CHECK")
print("="*60)

if train_stats['total_samples'] > 0:
    correct = 0
    total = 0
    
    for file_path, data in train_pseudo_labels.items():
        true_class = data['original_class']
        clip_probs = data['clip_probs']
        
        # Get predicted class
        pred_class_idx = np.argmax(clip_probs)
        pred_class = idx2label[str(pred_class_idx)]
        
        if pred_class == true_class:
            correct += 1
        total += 1
    
    accuracy = correct / max(total, 1)
    print(f"\nQuality Check Results:")
    print(f"   Accuracy: {accuracy:.4f} ({correct}/{total})")
    print(f"   This should match your training accuracy (~0.6667)")
else:
    print("No data for quality check")

## CREATE SIMPLE VISUALIZATION

In [ ]:
print("\n" + "="*60)
print("CREATING VISUALIZATION")
print("="*60)

if train_stats['total_samples'] > 0:
    # Take first 3 samples
    sample_files = list(train_pseudo_labels.keys())[:3]
    
    for i, file_path in enumerate(sample_files):
        try:
            data = train_pseudo_labels[file_path]
            frame_labels = data['frame_labels']
            clip_probs = data['clip_probs']
            original_class = data['original_class']
            
            # Simple plot
            fig, axes = plt.subplots(1, 2, figsize=(10, 4))
            
            # Frame labels heatmap
            im1 = axes[0].imshow(frame_labels.T, aspect='auto', cmap='RdYlGn',
                                vmin=0, vmax=1, interpolation='nearest')
            axes[0].set_title(f'Sample {i+1}: {original_class}')
            axes[0].set_xlabel('Frame')
            axes[0].set_ylabel('Class')
            axes[0].set_yticks(range(config.NUM_CLASSES))
            axes[0].set_yticklabels([idx2label[str(j)][:10] for j in range(config.NUM_CLASSES)])
            plt.colorbar(im1, ax=axes[0], label='Label')
            
            # Clip probabilities bar chart
            colors = ['red', 'green', 'blue', 'orange', 'purple']
            class_names = [idx2label[str(j)] for j in range(config.NUM_CLASSES)]
            bars = axes[1].bar(class_names, clip_probs, color=colors[:config.NUM_CLASSES])
            axes[1].axhline(y=0.5, color='red', linestyle='--', alpha=0.5)
            axes[1].set_ylim(0, 1.0)
            axes[1].set_title('Clip Probabilities')
            axes[1].set_ylabel('Probability')
            
            # Add values on bars
            for bar, prob in zip(bars, clip_probs):
                axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                           f'{prob:.2f}', ha='center', fontsize=9)
            
            plt.tight_layout()
            
            # Save
            viz_dir = os.path.join(config.OUTPUT_DIR, 'visualizations', 'pseudo_labels')
            os.makedirs(viz_dir, exist_ok=True)
            viz_path = os.path.join(viz_dir, f'pseudo_sample_{i}.png')
            plt.savefig(viz_path, dpi=120, bbox_inches='tight')
            plt.close()
            
            print(f"   Saved visualization: pseudo_sample_{i}.png")
            
        except Exception as e:
            print(f"   Error visualizing sample {i}: {str(e)[:50]}")

## SAVE CONFIGURATION

In [ ]:
print("\n" + "="*60)
print("SAVING CONFIGURATION")
print("="*60)

pseudo_config = {
    'generation_date': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
    'source_model': stage1_model_path,
    'confidence_threshold': float(pseudo_gen.confidence_threshold),
    'min_event_length': int(pseudo_gen.min_event_length),
    'samples_processed': int(train_stats.get('total_samples', 0)),
    'total_events': int(train_stats.get('events_detected', 0)),
    'frame_coverage': float(train_stats.get('frame_coverage', 0)),
    'num_classes': config.NUM_CLASSES,
    'classes': [idx2label[str(i)] for i in range(config.NUM_CLASSES)],
    'device': str(config.DEVICE)
}

config_save_path = os.path.join(config.DATA_PROCESSED, 'cpu_pseudo_config.json')
with open(config_save_path, 'w') as f:
    json.dump(pseudo_config, f, indent=2)

print(f"Configuration saved: {config_save_path}")

## FINAL SUMMARY

In [ ]:
print("\n" + "="*60)
print("PSEUDO-LABEL GENERATION SUMMARY")
print("="*60)

print(f"""
PSEUDO-LABEL GENERATION {'COMPLETED' if train_stats['total_samples'] > 0 else 'FAILED'}!

{'KEY RESULTS:' if train_stats['total_samples'] > 0 else 'ERROR:'}
""")

if train_stats['total_samples'] > 0:
    print(f"   • Samples processed: {train_stats['total_samples']:,}")
    print(f"   • Frame coverage: {train_stats['frame_coverage']:.2f}%")
    print(f"   • Events detected: {train_stats['events_detected']:,}")
    print(f"   • Avg events/sample: {train_stats['avg_events_per_sample']:.2f}")
else:
    print("   • No samples were processed successfully")
    print("   • Check audio file paths and model loading")

print(f"""
FILES CREATED:
   • Pseudo-labels: {train_pseudo_path}
   • Configuration: {config_save_path}
""")

print("="*60)
print("PART 6 COMPLETE")
print("="*60)

# Final verification
print("\nFinal verification...")
if os.path.exists(train_pseudo_path):
    with np.load(train_pseudo_path, allow_pickle=True) as data:
        num_samples = len(data['pseudo_labels'].item())
        print(f"   Pseudo-labels file contains: {num_samples} samples")
        print(f"   File size: {os.path.getsize(train_pseudo_path) / 1024:.1f} KB")
else:
    print("Pseudo-labels file not created")

# Part 07

## SELF-TRAINING WITH MEAN TEACHER FRAMEWORK

In [ ]:
print("\n" + "="*70)
print("SELF-TRAINING WITH MEAN TEACHER FRAMEWORK")
print("="*70)
print("Goal: Train student model using pseudo-labels while maintaining")
print("      a teacher model via Exponential Moving Average (EMA)")
print("="*70)

import copy
from torch.optim.lr_scheduler import CosineAnnealingLR

## LOADING PSEUDO-LABELS

In [ ]:
print("\nLOADING PSEUDO-LABELS")
print("-" * 40)

# Load pseudo-labels generated from Part 6
pseudo_labels_path = os.path.join(config.DATA_PROCESSED, 'cpu_train_pseudo_labels.npz')

if not os.path.exists(pseudo_labels_path):
    print(f"Pseudo-labels not found: {pseudo_labels_path}")
    print("Running pseudo-label generation first...")
    
    # Need to run Part 6 first
    raise FileNotFoundError("Please run Part 6 (Pseudo-label Generation) first")

print(f"Loading from: {pseudo_labels_path}")
with np.load(pseudo_labels_path, allow_pickle=True) as data:
    pseudo_data = data['pseudo_labels'].item()
    pseudo_stats = data['statistics'].item()

# Convert to file_path -> pseudo_label mapping
train_pseudo_labels = {}
for short_name, pseudo_info in pseudo_data.items():
    # Reconstruct file path (Need to store this mapping properly)
    # This is a simplification - you should save the mapping properly
    file_path = os.path.join(config.DATASET_PATH, short_name + '.wav')
    train_pseudo_labels[file_path] = {
        'frame_labels': pseudo_info['labels'],
        'clip_probs': pseudo_info['probs'],
        'original_class': pseudo_info['class']
    }

print(f"Loaded pseudo-labels for {len(train_pseudo_labels)} samples")
if 'frame_coverage' in pseudo_stats:
    print(f"   Frame coverage: {pseudo_stats['frame_coverage']:.2f}%")
if 'events_detected' in pseudo_stats:
    print(f"   Total events: {pseudo_stats['events_detected']:,}")

## CREATING ENHANCED DATASET WITH PSEUDO-LABELS

In [ ]:
print("\nCREATING ENHANCED DATASET WITH PSEUDO-LABELS")
print("-" * 40)

class FrameLevelStutterDataset(Dataset):
    """
    Dataset that returns both clip-level and frame-level labels
    Frame-level labels come from pseudo-labels
    """

    def __init__(self, csv_path, preprocessor, label2idx,
                 pseudo_labels_dict=None, use_frame_labels=False,
                 max_samples=None):
        """
        Args:
            csv_path: Path to split CSV
            preprocessor: AudioPreprocessor
            label2idx: Label to index mapping
            pseudo_labels_dict: Dictionary of pseudo-labels
            use_frame_labels: Whether to use frame-level supervision
            max_samples: Maximum samples to load (for CPU testing)
        """
        # Load only necessary columns to save memory
        self.df = pd.read_csv(csv_path)
        
        # Limit samples for CPU testing
        if max_samples and len(self.df) > max_samples:
            self.df = self.df.head(max_samples)
            print(f"   Limited to {max_samples} samples for CPU testing")
        
        self.preprocessor = preprocessor
        self.label2idx = label2idx
        self.pseudo_labels_dict = pseudo_labels_dict or {}
        self.use_frame_labels = use_frame_labels

        # Calculate how many samples have pseudo-labels
        self.samples_with_pseudo = 0
        if self.use_frame_labels:
            for idx in range(len(self.df)):
                file_path = self.df.iloc[idx]['file_path']
                if file_path in self.pseudo_labels_dict:
                    self.samples_with_pseudo += 1

        print(f"Dataset created: {len(self.df)} samples")
        if self.use_frame_labels:
            print(f"   Samples with pseudo-labels: {self.samples_with_pseudo}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        file_path = row['file_path']
        class_name = row['class']

        # Load audio
        try:
            audio = self.preprocessor.load_and_preprocess(file_path)
            if audio is None:
                audio = np.zeros(self.preprocessor.target_length)
        except Exception:
            audio = np.zeros(self.preprocessor.target_length)

        audio_tensor = torch.from_numpy(audio).float()

        # Clip-level label
        clip_label = torch.zeros(len(self.label2idx), dtype=torch.float32)
        if class_name in self.label2idx:
            clip_label[self.label2idx[class_name]] = 1.0

        # Frame-level label (from pseudo-labels)
        frame_label = None
        has_pseudo_label = False

        if self.use_frame_labels and file_path in self.pseudo_labels_dict:
            pseudo_data = self.pseudo_labels_dict[file_path]
            # Handle different formats
            if 'frame_labels' in pseudo_data:
                frame_label_np = pseudo_data['frame_labels']
                # Ensure proper shape
                if frame_label_np.ndim == 1:
                    frame_label_np = frame_label_np.reshape(-1, 1)
                frame_label = torch.from_numpy(frame_label_np.astype(np.float32))
                has_pseudo_label = True
            elif 'labels' in pseudo_data:
                frame_label_np = pseudo_data['labels']
                if frame_label_np.ndim == 1:
                    frame_label_np = frame_label_np.reshape(-1, 1)
                frame_label = torch.from_numpy(frame_label_np.astype(np.float32))
                has_pseudo_label = True

        # If frame_label is None, create empty one
        if frame_label is None and self.use_frame_labels:
            # Default sequence length after Wav2Vec2 (approx 100 frames for 3 seconds)
            seq_len = 100
            frame_label = torch.zeros(seq_len, len(self.label2idx), dtype=torch.float32)

        return {
            'audio': audio_tensor,
            'clip_label': clip_label,
            'frame_label': frame_label,
            'has_pseudo_label': has_pseudo_label,
            'file_path': file_path,
            'class_name': class_name
        }

# Create enhanced datasets (with sample limits for CPU)
print("\nCreating enhanced datasets...")
MAX_SAMPLES_TRAIN = 2000  # Increase as memory allows
MAX_SAMPLES_VAL = 100

train_dataset_enhanced = FrameLevelStutterDataset(
    csv_path=os.path.join(config.DATA_PROCESSED, 'train_split.csv'),
    preprocessor=preprocessor,
    label2idx=label2idx,
    pseudo_labels_dict=train_pseudo_labels,
    use_frame_labels=True,
    max_samples=MAX_SAMPLES_TRAIN
)

val_dataset_enhanced = FrameLevelStutterDataset(
    csv_path=os.path.join(config.DATA_PROCESSED, 'val_split.csv'),
    preprocessor=preprocessor,
    label2idx=label2idx,
    use_frame_labels=False,  # Validation uses clip-level only
    max_samples=MAX_SAMPLES_VAL
)

# Create DataLoaders (with smaller batch size for CPU)
BATCH_SIZE_STAGE2 = 4  # Even smaller for frame-level training
print(f"\nUsing batch size {BATCH_SIZE_STAGE2} for CPU memory constraints")

train_loader_stage2 = DataLoader(
    train_dataset_enhanced,
    batch_size=BATCH_SIZE_STAGE2,
    shuffle=True,
    num_workers=0,
    pin_memory=False,  # False for CPU
    drop_last=True
)

val_loader_stage2 = DataLoader(
    val_dataset_enhanced,
    batch_size=BATCH_SIZE_STAGE2,
    shuffle=False,
    num_workers=0,
    pin_memory=False
)

print(f"\nEnhanced DataLoaders created:")
print(f"   Train batches: {len(train_loader_stage2)}")
print(f"   Val batches: {len(val_loader_stage2)}")
print(f"   Samples with pseudo-labels: {train_dataset_enhanced.samples_with_pseudo}")

## IMPLEMENTING MEAN TEACHER FRAMEWORK

In [ ]:
print("\nIMPLEMENTING MEAN TEACHER FRAMEWORK (CPU)")
print("-" * 40)

class MeanTeacherCPU:
    """
    Mean Teacher framework
    """

    def __init__(self, student_model, ema_decay=0.99):  # Lower decay for faster adaptation
        """
        Args:
            student_model: The trainable model
            ema_decay: EMA decay rate (0.99 for CPU, faster updates)
        """
        self.student = student_model
        self.ema_decay = ema_decay

        # Create teacher as a copy of student
        self.teacher = copy.deepcopy(student_model)

        # Teacher is always in eval mode and doesn't require gradients
        self.teacher.eval()
        for param in self.teacher.parameters():
            param.requires_grad = False

        print(f"Mean Teacher initialized")
        print(f"   EMA decay: {ema_decay}")
        print(f"   Teacher parameters frozen: ✓")

    def update_teacher(self):
        """Update teacher model using EMA"""
        with torch.no_grad():
            for teacher_param, student_param in zip(
                self.teacher.parameters(),
                self.student.parameters()
            ):
                # EMA update: teacher = decay*teacher + (1-decay)*student
                teacher_param.data.mul_(self.ema_decay).add_(
                    student_param.data, alpha=1 - self.ema_decay
                )

    def get_teacher_predictions(self, audio, return_details=False):
        """Get predictions from teacher model"""
        self.teacher.eval()
        with torch.no_grad():
            if return_details:
                return self.teacher(audio, return_details=True)
            else:
                return self.teacher(audio)

    def get_student_predictions(self, audio, return_details=False):
        """Get predictions from student model"""
        if return_details:
            return self.student(audio, return_details=True)
        else:
            return self.student(audio)

# Initialize student model for Stage 2 (fresh model)
print("\nInitializing student model for Stage 2...")
from models.neurosymbolic import NeuroSymbolicStutterDetectorCPU

student_model = NeuroSymbolicStutterDetectorCPU(config, freeze_encoder=True)
student_model = student_model.to(config.DEVICE)

# Initialize Mean Teacher
mean_teacher = MeanTeacherCPU(
    student_model=student_model,
    ema_decay=0.99  # Faster decay for CPU training
)

print(f"Student and Teacher models ready")

## DEFINING BOUNDARY REGRESSION LOSS

In [ ]:
print("\nDEFINING BOUNDARY REGRESSION LOSS")
print("-" * 40)

class BoundaryRegressionLossCPU(nn.Module):
    """
    Combined loss for frame-level supervision
    1. Frame-level BCE: For presence/absence of stutter
    2. Boundary smoothness: For temporal consistency
    """

    def __init__(self, alpha=1.0, beta=0.1):  # Lower beta for CPU
        """
        Args:
            alpha: Weight for frame-level BCE
            beta: Weight for boundary smoothness
        """
        super().__init__()
        self.alpha = alpha
        self.beta = beta

        print(f"Boundary Regression Loss initialized")
        print(f"   Alpha (frame BCE): {alpha}")
        print(f"   Beta (boundary smooth): {beta}")

    def forward(self, frame_logits, frame_targets):
        """
        Args:
            frame_logits: (batch, seq_len, num_classes)
            frame_targets: (batch, seq_len, num_classes)

        Returns:
            loss: scalar
        """
        # 1. Frame-level BCE loss
        frame_loss = F.binary_cross_entropy_with_logits(
            frame_logits,
            frame_targets,
            reduction='mean'
        )

        # 2. Boundary smoothness
        # Encourage smooth transitions at boundaries
        if frame_logits.shape[1] > 1:
            # Simple temporal difference (L1 for CPU efficiency)
            temporal_diff = frame_logits[:, 1:, :] - frame_logits[:, :-1, :]
            smoothness_loss = torch.abs(temporal_diff).mean()
        else:
            smoothness_loss = torch.tensor(0.0, device=frame_logits.device)

        # Combined loss
        total_loss = self.alpha * frame_loss + self.beta * smoothness_loss

        return total_loss, {'frame': frame_loss.item(), 'smooth': smoothness_loss.item()}

## DEFINING CONSISTENCY LOSS (TEACHER-STUDENT)

In [ ]:
print("\nDEFINING CONSISTENCY LOSS")
print("-" * 40)

class ConsistencyLossCPU(nn.Module):
    """
    Consistency loss between teacher and student predictions
    Encourages student to match teacher's predictions
    """

    def __init__(self, temperature=1.0):
        super().__init__()
        self.temperature = temperature

        print(f"Consistency Loss initialized (temp={temperature})")

    def forward(self, student_logits, teacher_logits):
        """
        Args:
            student_logits: (batch, seq_len, num_classes)
            teacher_logits: (batch, seq_len, num_classes)
        """
        # Use L2 loss instead of MSE for simplicity
        # Soften predictions with temperature for CPU efficiency
        student_probs = torch.sigmoid(student_logits / self.temperature)
        teacher_probs = torch.sigmoid(teacher_logits / self.temperature)

        # L2 distance (simpler than MSE)
        consistency_loss = F.mse_loss(student_probs, teacher_probs)

        return consistency_loss

## DEFINING COMBINED LOSS FOR STAGE 2

In [ ]:
print("\nDEFINING COMBINED LOSS FOR STAGE 2")
print("-" * 40)

class Stage2CombinedLossCPU(nn.Module):
    """
    Combined loss for self-training
    1. Supervised loss (on pseudo-labels)
    2. Consistency loss (teacher-student)
    3. Weak supervision (clip-level) for samples without pseudo-labels
    """

    def __init__(self, lambda_supervised=1.0, lambda_consistency=0.3,
                 lambda_weak=0.2):
        super().__init__()

        self.lambda_supervised = lambda_supervised
        self.lambda_consistency = lambda_consistency
        self.lambda_weak = lambda_weak

        self.boundary_loss = BoundaryRegressionLossCPU(alpha=1.0, beta=0.1)
        self.consistency_loss = ConsistencyLossCPU(temperature=1.0)
        # Simple MIL loss for weak supervision
        self.weak_loss = nn.BCEWithLogitsLoss()

        print(f"Stage 2 Combined Loss initialized (CPU optimized)")
        print(f"   λ_supervised: {lambda_supervised}")
        print(f"   λ_consistency: {lambda_consistency}")
        print(f"   λ_weak: {lambda_weak}")

    def simple_mil_loss(self, logits, clip_targets):
        """Simple MIL: max pooling + BCE"""
        clip_logits, _ = torch.max(logits, dim=1)
        return F.binary_cross_entropy_with_logits(clip_logits, clip_targets)

    def forward(self, student_logits, teacher_logits, frame_targets,
                clip_targets, has_pseudo_mask):
        """
        Args:
            student_logits: (batch, seq_len, num_classes)
            teacher_logits: (batch, seq_len, num_classes)
            frame_targets: (batch, seq_len, num_classes) or None
            clip_targets: (batch, num_classes)
            has_pseudo_mask: (batch,) boolean mask
        """
        total_loss = 0.0
        loss_components = {}

        # 1. Supervised loss (only for samples with pseudo-labels)
        if has_pseudo_mask.any() and frame_targets is not None:
            supervised_loss, sup_components = self.boundary_loss(
                student_logits[has_pseudo_mask],
                frame_targets[has_pseudo_mask]
            )
            total_loss += self.lambda_supervised * supervised_loss
            loss_components['supervised'] = supervised_loss.item()
            loss_components.update(sup_components)
        else:
            loss_components['supervised'] = 0.0

        # 2. Consistency loss (all samples)
        consistency = self.consistency_loss(student_logits, teacher_logits)
        total_loss += self.lambda_consistency * consistency
        loss_components['consistency'] = consistency.item()

        # 3. Weak supervision (samples without pseudo-labels)
        if (~has_pseudo_mask).any():
            weak_loss = self.simple_mil_loss(
                student_logits[~has_pseudo_mask],
                clip_targets[~has_pseudo_mask]
            )
            total_loss += self.lambda_weak * weak_loss
            loss_components['weak'] = weak_loss.item()
        else:
            loss_components['weak'] = 0.0

        loss_components['total'] = total_loss.item()

        return total_loss, loss_components

# Initialize combined loss
stage2_criterion = Stage2CombinedLossCPU(
    lambda_supervised=1.0,
    lambda_consistency=0.3,  # Lower for CPU stability
    lambda_weak=0.2
)

## SETTING UP STAGE 2 TRAINING

In [ ]:
print("\nSETTING UP STAGE 2 TRAINING")
print("-" * 40)

def train_epoch_stage2_cpu(mean_teacher, dataloader, criterion, optimizer,
                           device, grad_accum_steps=4):
    """
    Train one epoch with Mean Teacher framework
    """
    mean_teacher.student.train()
    mean_teacher.teacher.eval()

    total_loss = 0
    loss_components_sum = {
        'supervised': 0,
        'consistency': 0,
        'weak': 0,
        'frame': 0,
        'smooth': 0
    }
    valid_batches = 0

    optimizer.zero_grad()

    pbar = tqdm(dataloader, desc="Training Stage 2", leave=False)
    for batch_idx, batch in enumerate(pbar):
        try:
            audio = batch['audio'].to(device)
            clip_labels = batch['clip_label'].to(device)
            frame_labels_list = batch['frame_label']
            has_pseudo = batch['has_pseudo_label']

            # Handle frame labels
            if frame_labels_list is not None:
                # Stack frame labels (some might be None or empty)
                frame_labels_batch = []
                for fl in frame_labels_list:
                    if fl is not None and fl.numel() > 0:
                        # Reshape if needed
                        if fl.dim() == 1:
                            fl = fl.unsqueeze(1)
                        frame_labels_batch.append(fl.to(device))
                    else:
                        # Create dummy frame labels
                        dummy_shape = (100, len(label2idx))  # Approx seq length
                        frame_labels_batch.append(
                            torch.zeros(dummy_shape, device=device)
                        )
                frame_labels = torch.stack(frame_labels_batch)
            else:
                frame_labels = torch.zeros(
                    audio.shape[0], 100, len(label2idx), device=device
                )

            has_pseudo_mask = torch.tensor(has_pseudo, device=device)

            # Student predictions
            student_logits = mean_teacher.get_student_predictions(audio)

            # Teacher predictions (no gradient)
            with torch.no_grad():
                teacher_logits = mean_teacher.get_teacher_predictions(audio)

            # Calculate loss
            loss, loss_components = criterion(
                student_logits, teacher_logits,
                frame_labels, clip_labels,
                has_pseudo_mask
            )

            # Scale loss for gradient accumulation
            loss = loss / grad_accum_steps

            # Check for NaN/Inf
            if torch.isnan(loss) or torch.isinf(loss):
                print(f"NaN/Inf loss at batch {batch_idx}, skipping")
                continue

            # Backward
            loss.backward()

            # Gradient accumulation
            if (batch_idx + 1) % grad_accum_steps == 0:
                # Gradient clipping for stability
                torch.nn.utils.clip_grad_norm_(
                    mean_teacher.student.parameters(),
                    max_norm=1.0
                )
                optimizer.step()
                optimizer.zero_grad()

                # Update teacher with EMA
                mean_teacher.update_teacher()

                valid_batches += 1

            # Track metrics
            total_loss += loss.item() * grad_accum_steps
            for key in loss_components_sum.keys():
                if key in loss_components:
                    loss_components_sum[key] += loss_components[key]

            # Update progress
            current_loss = loss.item() * grad_accum_steps
            current_sup = loss_components.get('supervised', 0)
            current_cons = loss_components.get('consistency', 0)
            
            pbar.set_postfix({
                'loss': f'{current_loss:.4f}',
                'sup': f'{current_sup:.4f}',
                'cons': f'{current_cons:.4f}'
            })

        except Exception as e:
            print(f"\nError at batch {batch_idx}: {str(e)[:100]}")
            continue

    # Calculate averages
    if valid_batches > 0:
        avg_loss = total_loss / valid_batches
        avg_components = {
            k: v / valid_batches for k, v in loss_components_sum.items()
        }
    else:
        avg_loss = 0
        avg_components = {k: 0 for k in loss_components_sum.keys()}

    return avg_loss, avg_components

## VALIDATION FUNCTION

In [ ]:
def validate_stage2_cpu(mean_teacher, dataloader, device, max_batches=10):
    """
    Validate using teacher model
    """
    mean_teacher.teacher.eval()

    all_preds = []
    all_targets = []

    with torch.no_grad():
        for batch_idx, batch in enumerate(dataloader):
            if batch_idx >= max_batches:
                break

            audio = batch['audio'].to(device)
            targets = batch['clip_label'].to(device)

            # Use teacher for validation
            logits = mean_teacher.get_teacher_predictions(audio)

            # Pool to clip level
            clip_logits, _ = torch.max(logits, dim=1)
            probs = torch.sigmoid(clip_logits)

            all_preds.append(probs.cpu())
            all_targets.append(targets.cpu())

    if all_preds:
        all_preds = torch.cat(all_preds)
        all_targets = torch.cat(all_targets)
        
        # Simple accuracy calculation
        preds_binary = (all_preds > 0.5).float()
        accuracy = (preds_binary == all_targets).float().mean().item()
        
        return {'accuracy': accuracy}
    else:
        return {'accuracy': 0}

## EARLY STOPPING

In [ ]:
class SimpleEarlyStopping:
    """Simple early stopping for training"""
    
    def __init__(self, patience=2, min_delta=0.001):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = float('inf')
    
    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            return False
        else:
            self.counter += 1
            return self.counter >= self.patience

## STARTING STAGE 2 TRAINING (MEAN TEACHER)

In [ ]:
print("\nSTARTING STAGE 2 TRAINING (MEAN TEACHER)")
print("-" * 40)

def train_stage2_mean_teacher_cpu(mean_teacher, train_loader, val_loader,
                                  config, num_epochs=10):
    """
    Main training function for Stage 2
    """
    print(f"\nStage 2 Training Configuration (CPU):")
    print(f"   Epochs: {num_epochs}")
    print(f"   Learning Rate: {config.LEARNING_RATE:.2e}")
    print(f"   EMA Decay: {mean_teacher.ema_decay}")
    print(f"   Batch Size: {BATCH_SIZE_STAGE2}")
    print(f"   Grad Accum Steps: 4")
    print(f"   Train Samples: {MAX_SAMPLES_TRAIN}")
    print(f"   Val Samples: {MAX_SAMPLES_VAL}")

    # Optimizer with conservative settings for CPU
    optimizer = AdamW(
        mean_teacher.student.parameters(),
        lr=config.LEARNING_RATE * 0.5,  # Lower LR for CPU stability
        weight_decay=0.01
    )

    # Simple scheduler
    scheduler = CosineAnnealingLR(
        optimizer,
        T_max=num_epochs
    )

    # Early stopping
    early_stopping = SimpleEarlyStopping(patience=2, min_delta=0.001)

    # History tracking
    history = {
        'train_loss': [],
        'train_supervised': [],
        'train_consistency': [],
        'train_weak': [],
        'val_accuracy': []
    }

    best_val_acc = 0
    best_student_state = None
    best_teacher_state = None

    print(f"\nStarting Stage 2 training...")

    for epoch in range(num_epochs):
        print(f"\n{'='*50}")
        print(f"EPOCH {epoch+1}/{num_epochs}")
        print(f"{'='*50}")

        # Training
        train_loss, loss_components = train_epoch_stage2_cpu(
            mean_teacher, train_loader, stage2_criterion,
            optimizer, config.DEVICE, grad_accum_steps=4
        )

        # Validation
        val_metrics = validate_stage2_cpu(
            mean_teacher, val_loader, config.DEVICE, max_batches=10
        )

        # Update scheduler
        scheduler.step()

        # Record history
        history['train_loss'].append(train_loss)
        history['train_supervised'].append(loss_components.get('supervised', 0))
        history['train_consistency'].append(loss_components.get('consistency', 0))
        history['train_weak'].append(loss_components.get('weak', 0))
        history['val_accuracy'].append(val_metrics['accuracy'])

        # Print results
        print(f"\nEpoch {epoch+1} Results:")
        print(f"   Train Loss:       {train_loss:.4f}")
        print(f"     - Supervised:  {loss_components.get('supervised', 0):.4f}")
        print(f"     - Consistency: {loss_components.get('consistency', 0):.4f}")
        print(f"     - Weak:        {loss_components.get('weak', 0):.4f}")
        if 'frame' in loss_components:
            print(f"     - Frame BCE:    {loss_components.get('frame', 0):.4f}")
        if 'smooth' in loss_components:
            print(f"     - Smoothness:   {loss_components.get('smooth', 0):.4f}")
        print(f"   Val Accuracy:     {val_metrics['accuracy']:.4f}")
        print(f"   Learning Rate:    {optimizer.param_groups[0]['lr']:.6f}")

        # Save checkpoint
        checkpoint_path = os.path.join(
            config.CHECKPOINT_DIR,
            f'stage2_epoch_{epoch+1}_cpu.pth'
        )
        torch.save({
            'epoch': epoch,
            'student_state_dict': mean_teacher.student.state_dict(),
            'teacher_state_dict': mean_teacher.teacher.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_accuracy': val_metrics['accuracy'],
            'train_loss': train_loss
        }, checkpoint_path)
        print(f"   Checkpoint saved: {checkpoint_path}")

        # Save best model
        if val_metrics['accuracy'] > best_val_acc:
            best_val_acc = val_metrics['accuracy']
            best_student_state = copy.deepcopy(mean_teacher.student.state_dict())
            best_teacher_state = copy.deepcopy(mean_teacher.teacher.state_dict())
            print(f"   NEW BEST MODEL! Accuracy: {best_val_acc:.4f}")

        # Early stopping
        if early_stopping(train_loss):
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

    # Load best models
    if best_student_state is not None:
        mean_teacher.student.load_state_dict(best_student_state)
        mean_teacher.teacher.load_state_dict(best_teacher_state)
        print(f"\nLoaded best models with Accuracy: {best_val_acc:.4f}")

    return mean_teacher, history

# Start Stage 2 training
print("\nInitiating Stage 2 Training...")

trained_mean_teacher, stage2_history = train_stage2_mean_teacher_cpu(
    mean_teacher=mean_teacher,
    train_loader=train_loader_stage2,
    val_loader=val_loader_stage2,
    config=config,
    num_epochs=config.NUM_EPOCHS_STAGE2  # Default: 3 epochs
)

## SAVING FINAL STAGE 2 MODELS

In [ ]:
print("\nSAVING FINAL STAGE 2 MODELS")
print("-" * 40)

# Save final models
final_stage2_path = os.path.join(config.CHECKPOINT_DIR, 'final_stage2_mean_teacher_cpu.pth')
torch.save({
    'student_state_dict': trained_mean_teacher.student.state_dict(),
    'teacher_state_dict': trained_mean_teacher.teacher.state_dict(),
    'history': stage2_history,
    'config': {k: v for k, v in config.__dict__.items() 
               if not k.startswith('_') and not callable(v)},
    'ema_decay': trained_mean_teacher.ema_decay
}, final_stage2_path)
print(f"Final models saved to: {final_stage2_path}")

# Save history
history_path = os.path.join(config.OUTPUT_DIR, 'stage2_training_history_cpu.json')
with open(history_path, 'w') as f:
    import json
    # Convert tensors to lists for JSON serialization
    serializable_history = {}
    for key, value in stage2_history.items():
        if isinstance(value, list):
            serializable_history[key] = [float(v) for v in value]
    json.dump(serializable_history, f, indent=2)
print(f"History saved to: {history_path}")

## CREATING TRAINING VISUALIZATIONS

In [ ]:
print("\nCREATING TRAINING VISUALIZATIONS")
print("-" * 40)

def plot_stage2_curves_cpu(history):
    """Plot Stage 2 training curves"""
    if not history['train_loss']:
        print("No training history to plot")
        return None
    
    try:
        fig, axes = plt.subplots(2, 2, figsize=(12, 8))
        epochs = range(1, len(history['train_loss']) + 1)

        # 1. Total loss
        ax = axes[0, 0]
        ax.plot(epochs, history['train_loss'], 'b-', label='Train Loss', linewidth=2)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Loss')
        ax.set_title('Total Training Loss', fontweight='bold')
        ax.legend()
        ax.grid(alpha=0.3)

        # 2. Loss components
        ax = axes[0, 1]
        if history['train_supervised']:
            ax.plot(epochs, history['train_supervised'], label='Supervised', linewidth=2)
        if history['train_consistency']:
            ax.plot(epochs, history['train_consistency'], label='Consistency', linewidth=2)
        if history['train_weak']:
            ax.plot(epochs, history['train_weak'], label='Weak', linewidth=2)
        ax.set_xlabel('Epoch')
        ax.set_ylabel('Loss')
        ax.set_title('Loss Components', fontweight='bold')
        ax.legend()
        ax.grid(alpha=0.3)

        # 3. Validation accuracy
        ax = axes[1, 0]
        if history['val_accuracy']:
            ax.plot(epochs, history['val_accuracy'], 'g-', label='Val Accuracy',
                    linewidth=2, marker='o')
            ax.set_xlabel('Epoch')
            ax.set_ylabel('Accuracy')
            ax.set_title('Validation Accuracy', fontweight='bold')
            ax.legend()
            ax.grid(alpha=0.3)

        # 4. Learning rate (placeholder)
        ax = axes[1, 1]
        ax.text(0.5, 0.5, 'Training Progress\nComplete', 
                horizontalalignment='center', verticalalignment='center',
                transform=ax.transAxes, fontsize=12)
        ax.axis('off')

        plt.tight_layout()

        curves_path = os.path.join(
            config.OUTPUT_DIR, 'visualizations', 'stage2_training_curves_cpu.png'
        )
        plt.savefig(curves_path, dpi=150, bbox_inches='tight')
        print(f"Training curves saved to: {curves_path}")
        plt.show()

        return curves_path
    except Exception as e:
        print(f"Could not create visualization: {e}")
        return None

curves_path = plot_stage2_curves_cpu(stage2_history)

## FINAL EVALUATION ON TEST SET

In [ ]:
print("\nFINAL EVALUATION ON TEST SET")
print("-" * 40)

def evaluate_final_stage2_cpu(mean_teacher, test_dataset, device, max_samples=50):
    """Final evaluation using teacher model"""
    mean_teacher.teacher.eval()
    
    # Create test DataLoader
    test_loader = DataLoader(
        test_dataset,
        batch_size=1,  # Small batch for CPU
        shuffle=False,
        num_workers=0
    )
    
    all_preds = []
    all_targets = []
    all_classes = []

    with torch.no_grad():
        for batch_idx, batch in enumerate(tqdm(test_loader, desc="Final Evaluation")):
            if batch_idx >= max_samples:
                break
                
            audio = batch['audio'].to(device)
            targets = batch['clip_label']
            classes = batch['class_name']

            # Teacher predictions
            logits = mean_teacher.get_teacher_predictions(audio)
            # Pool to clip level
            clip_logits, _ = torch.max(logits, dim=1)
            probs = torch.sigmoid(clip_logits)

            all_preds.append(probs.cpu())
            all_targets.append(targets)
            all_classes.extend(classes)

    if all_preds:
        # Concatenate results
        all_preds = torch.cat(all_preds)
        all_targets = torch.cat(all_targets)

        # Calculate simple accuracy
        preds_binary = (all_preds > 0.5).float()
        accuracy = (preds_binary == all_targets).float().mean().item()

        # Per-class accuracy
        per_class_metrics = {}
        for class_idx in range(config.NUM_CLASSES):
            class_name = idx2label[str(class_idx)]
            class_acc = (preds_binary[:, class_idx] == all_targets[:, class_idx]).float().mean()
            per_class_metrics[class_name] = class_acc.item()

        metrics = {
            'accuracy': accuracy,
            'samples_evaluated': len(all_preds)
        }

        return metrics, per_class_metrics, all_preds, all_targets
    else:
        print("No predictions generated")
        return {'accuracy': 0}, {}, None, None

# Load test dataset
print("\nLoading test dataset...")
test_dataset = FrameLevelStutterDataset(
    csv_path=os.path.join(config.DATA_PROCESSED, 'test_split.csv'),
    preprocessor=preprocessor,
    label2idx=label2idx,
    use_frame_labels=False,
    max_samples=50  # Small test for CPU
)

# Run final evaluation
print("\nRunning final evaluation...")
test_metrics, per_class_metrics, test_preds, test_targets = evaluate_final_stage2_cpu(
    trained_mean_teacher, test_dataset, config.DEVICE, max_samples=50
)

print(f"\nFINAL TEST RESULTS (CPU - {test_metrics['samples_evaluated']} samples):")
print(f"   Overall Accuracy: {test_metrics['accuracy']:.4f}")

if per_class_metrics:
    print(f"\nPER-CLASS ACCURACY:")
    for class_name, acc in per_class_metrics.items():
        print(f"   {class_name:20s}: {acc:.4f}")

## SAVING EVALUATION RESULTS

In [ ]:
print("\nSAVING EVALUATION RESULTS")
print("-" * 40)

# Save test results
test_results_path = os.path.join(config.OUTPUT_DIR, 'stage2_test_results_cpu.npz')
if test_preds is not None and test_targets is not None:
    np.savez(
        test_results_path,
        predictions=test_preds.numpy(),
        targets=test_targets.numpy(),
        metrics=test_metrics,
        per_class_metrics=per_class_metrics
    )
    print(f"Test results saved to: {test_results_path}")

# Save comprehensive report
report = {
    'phase': 'Phase 7 - Self-Training with Mean Teacher',
    'completion_time': datetime.now().isoformat(),
    'device': str(config.DEVICE),
    'training_config': {
        'epochs': config.NUM_EPOCHS_STAGE2,
        'batch_size': BATCH_SIZE_STAGE2,
        'learning_rate': config.LEARNING_RATE * 0.5,
        'ema_decay': trained_mean_teacher.ema_decay,
        'train_samples': MAX_SAMPLES_TRAIN,
        'val_samples': MAX_SAMPLES_VAL
    },
    'results': {
        'best_val_accuracy': max(stage2_history['val_accuracy']) if stage2_history['val_accuracy'] else 0,
        'final_test_accuracy': test_metrics.get('accuracy', 0),
        'test_samples': test_metrics.get('samples_evaluated', 0)
    },
    'files_saved': {
        'models': final_stage2_path,
        'history': history_path,
        'test_results': test_results_path if test_preds is not None else 'N/A',
        'visualization': curves_path if curves_path else 'N/A'
    }
}

report_path = os.path.join(config.OUTPUT_DIR, 'reports', 'phase7_report_cpu.json')
os.makedirs(os.path.dirname(report_path), exist_ok=True)
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)
print(f"Comprehensive report saved to: {report_path}")

## FINAL SUMMARY

In [ ]:
print("\n" + "="*70)
print("FINAL SUMMARY")
print("="*70)

best_val_acc = max(stage2_history['val_accuracy']) if stage2_history['val_accuracy'] else 0

summary_text = f"""
SELF-TRAINING WITH MEAN TEACHER COMPLETE

FINAL RESULTS:
• Best Validation Accuracy: {best_val_acc:.4f}
• Final Test Accuracy: {test_metrics.get('accuracy', 0):.4f}
• Training Epochs Completed: {len(stage2_history['train_loss'])}
• Samples with Pseudo-labels: {train_dataset_enhanced.samples_with_pseudo}/{len(train_dataset_enhanced)}

FILES SAVED:
• Final models: {os.path.basename(final_stage2_path)}
• Training history: {os.path.basename(history_path)}
• Test results: {os.path.basename(test_results_path) if test_preds is not None else 'N/A'}
• Report: {os.path.basename(report_path)}
• Visualization: {os.path.basename(curves_path) if curves_path else 'N/A'}

MODEL COMPARISON:
• Stage 1 (MIL): Weak supervision only (clip-level)
• Stage 2 (Mean Teacher): Refined with pseudo-labels + consistency training
• Key Innovation: Differentiable soft rules with adaptive gating
"""

print(summary_text)

# Log completion
logger.log(f"Phase 7 completed - Mean Teacher training finished on {config.DEVICE}")
logger.log(f"Best val accuracy: {best_val_acc:.4f}")
logger.log(f"Test accuracy: {test_metrics.get('accuracy', 0):.4f}")
logger.log(f"Training epochs: {len(stage2_history['train_loss'])}")

# Part 08

## FIX FOR PSEUDO-LABEL LOADING ISSUE

In [ ]:
print("\n" + "="*70)
print("FIXING PSEUDO-LABEL LOADING ISSUE")
print("="*70)

# First, let's debug what's in the pseudo-label file
print("\nDEBUG: Inspecting pseudo-label file structure...")
with np.load(pseudo_labels_path, allow_pickle=True) as data:
    print(f"Keys in npz file: {list(data.keys())}")
    
    # Check the structure of pseudo_labels
    pseudo_data = data['pseudo_labels'].item()
    print(f"\nType of pseudo_data: {type(pseudo_data)}")
    
    # Look at first few items
    sample_keys = list(pseudo_data.keys())[:3]
    print(f"\nFirst 3 sample keys:")
    for i, key in enumerate(sample_keys):
        print(f"  {i+1}. {key}")
        sample_info = pseudo_data[key]
        print(f"     Keys in sample: {list(sample_info.keys())}")
        if 'labels' in sample_info:
            print(f"     labels shape: {sample_info['labels'].shape}")
        if 'probs' in sample_info:
            print(f"     probs shape: {sample_info['probs'].shape}")
        if 'class' in sample_info:
            print(f"     class: {sample_info['class']}")

# The issue is that we need to map short filenames back to full file paths
# Let's create a proper mapping
print("\nCreating proper file path mapping...")

# Load the train split CSV to get all file paths
train_csv_path = os.path.join(config.DATA_PROCESSED, 'train_split.csv')
train_df = pd.read_csv(train_csv_path)
print(f"Total train samples: {len(train_df)}")

# Create mapping from short name to full path
short_to_full = {}
for idx, row in train_df.iterrows():
    file_path = row['file_path']
    filename = os.path.basename(file_path)
    short_name = filename.replace('.wav', '')
    short_to_full[short_name] = file_path

print(f"Created mapping for {len(short_to_full)} files")

# Now reload pseudo-labels with proper mapping
print("\nReloading pseudo-labels with proper mapping...")
with np.load(pseudo_labels_path, allow_pickle=True) as data:
    pseudo_data = data['pseudo_labels'].item()
    pseudo_stats = data['statistics'].item()

# Create new mapping with full file paths
train_pseudo_labels = {}
valid_samples = 0

for short_name, pseudo_info in pseudo_data.items():
    # Try to find full file path
    if short_name in short_to_full:
        file_path = short_to_full[short_name]
        train_pseudo_labels[file_path] = {
            'frame_labels': pseudo_info.get('labels'),
            'clip_probs': pseudo_info.get('probs'),
            'original_class': pseudo_info.get('class'),
            'boundaries': pseudo_info.get('boundaries')
        }
        valid_samples += 1
    else:
        print(f"Could not find full path for: {short_name}")

print(f"\nSuccessfully mapped {valid_samples}/{len(pseudo_data)} pseudo-labels")
print(f"   Frame coverage: {pseudo_stats.get('frame_coverage', 0):.2f}%")
print(f"   Events detected: {pseudo_stats.get('events_detected', 0)}")

## FIXED DATASET CLASS

In [ ]:
print("\nUPDATING DATASET CLASS WITH FIXES")
print("-" * 40)

class FixedFrameLevelStutterDataset(Dataset):
    """
    Fixed dataset class with proper handling of pseudo-labels
    """

    def __init__(self, csv_path, preprocessor, label2idx,
                 pseudo_labels_dict=None, use_frame_labels=False,
                 max_samples=None, default_seq_len=100):
        """
        Args:
            csv_path: Path to split CSV
            preprocessor: AudioPreprocessor
            label2idx: Label to index mapping
            pseudo_labels_dict: Dictionary of pseudo-labels
            use_frame_labels: Whether to use frame-level supervision
            max_samples: Maximum samples to load
            default_seq_len: Default sequence length for frame labels
        """
        # Load the CSV
        self.df = pd.read_csv(csv_path)
        
        # Limit samples for CPU testing
        if max_samples and len(self.df) > max_samples:
            self.df = self.df.head(max_samples)
            print(f"   Limited to {max_samples} samples for CPU testing")
        
        self.preprocessor = preprocessor
        self.label2idx = label2idx
        self.pseudo_labels_dict = pseudo_labels_dict or {}
        self.use_frame_labels = use_frame_labels
        self.default_seq_len = default_seq_len
        self.num_classes = len(label2idx)

        # Calculate how many samples have pseudo-labels
        self.samples_with_pseudo = 0
        if self.use_frame_labels:
            for idx in range(len(self.df)):
                file_path = self.df.iloc[idx]['file_path']
                if file_path in self.pseudo_labels_dict:
                    pseudo_data = self.pseudo_labels_dict[file_path]
                    if pseudo_data.get('frame_labels') is not None:
                        self.samples_with_pseudo += 1

        print(f"Dataset created: {len(self.df)} samples")
        if self.use_frame_labels:
            print(f"   Samples with valid pseudo-labels: {self.samples_with_pseudo}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        file_path = row['file_path']
        class_name = row['class']

        # Load audio
        try:
            audio = self.preprocessor.load_and_preprocess(file_path)
            if audio is None:
                audio = np.zeros(self.preprocessor.target_length)
        except Exception as e:
            print(f"Error loading audio {file_path}: {e}")
            audio = np.zeros(self.preprocessor.target_length)

        audio_tensor = torch.from_numpy(audio).float()

        # Clip-level label
        clip_label = torch.zeros(self.num_classes, dtype=torch.float32)
        if class_name in self.label2idx:
            clip_label[self.label2idx[class_name]] = 1.0

        # Frame-level label (from pseudo-labels)
        frame_label = None
        has_pseudo_label = False

        if self.use_frame_labels and file_path in self.pseudo_labels_dict:
            pseudo_data = self.pseudo_labels_dict[file_path]
            frame_labels_np = pseudo_data.get('frame_labels')
            
            if frame_labels_np is not None and frame_labels_np.size > 0:
                # Convert to tensor
                frame_label_tensor = torch.from_numpy(frame_labels_np.astype(np.float32))
                
                # Ensure proper shape
                if frame_label_tensor.dim() == 1:
                    # If 1D, reshape to (seq_len, 1)
                    frame_label_tensor = frame_label_tensor.unsqueeze(1)
                
                # If it has wrong number of classes, adjust
                if frame_label_tensor.shape[1] != self.num_classes:
                    if frame_label_tensor.shape[1] == 1 and self.num_classes > 1:
                        # Expand single class to multi-class
                        frame_label_tensor = frame_label_tensor.repeat(1, self.num_classes)
                    else:
                        # Create proper shape
                        frame_label_tensor = torch.zeros(
                            frame_label_tensor.shape[0], self.num_classes,
                            dtype=torch.float32
                        )
                
                frame_label = frame_label_tensor
                has_pseudo_label = True
        
        # If no frame label but need one, create an empty tensor
        if frame_label is None and self.use_frame_labels:
            frame_label = torch.zeros(self.default_seq_len, self.num_classes, dtype=torch.float32)
            has_pseudo_label = False

        # Ensure all returns are tensors (not None)
        if frame_label is None:
            frame_label = torch.zeros(self.default_seq_len, self.num_classes, dtype=torch.float32)

        return {
            'audio': audio_tensor,
            'clip_label': clip_label,
            'frame_label': frame_label,
            'has_pseudo_label': has_pseudo_label,
            'file_path': file_path,
            'class_name': class_name
        }

## RE-CREATE DATASETS WITH FIXED CLASS

In [ ]:
print("\nRE-CREATING DATASETS WITH FIXES")
print("-" * 40)

# Create enhanced datasets with fixed class
train_dataset_enhanced = FixedFrameLevelStutterDataset(
    csv_path=os.path.join(config.DATA_PROCESSED, 'train_split.csv'),
    preprocessor=preprocessor,
    label2idx=label2idx,
    pseudo_labels_dict=train_pseudo_labels,
    use_frame_labels=True,
    max_samples=MAX_SAMPLES_TRAIN,
    default_seq_len=100  # Approx sequence length after Wav2Vec2
)

val_dataset_enhanced = FixedFrameLevelStutterDataset(
    csv_path=os.path.join(config.DATA_PROCESSED, 'val_split.csv'),
    preprocessor=preprocessor,
    label2idx=label2idx,
    pseudo_labels_dict=None,  # No pseudo-labels for validation
    use_frame_labels=False,
    max_samples=MAX_SAMPLES_VAL,
    default_seq_len=100
)

# Create DataLoaders with a custom collate function
print("\nCREATING DATALOADERS WITH CUSTOM COLLATE")
print("-" * 40)

def custom_collate_fn(batch):
    """
    Custom collate function to handle variable-length frame labels
    """
    batch_dict = {}
    
    # Get all keys from the first sample
    keys = batch[0].keys()
    
    for key in keys:
        values = [sample[key] for sample in batch]
        
        if key == 'file_path' or key == 'class_name':
            # Keep as a list for strings
            batch_dict[key] = values
        elif key == 'has_pseudo_label':
            # Convert to tensor
            batch_dict[key] = torch.tensor(values, dtype=torch.bool)
        elif key == 'frame_label':
            # Handle frame labels - they might have different sequence lengths
            # Find max sequence length
            max_seq_len = max([v.shape[0] for v in values])
            num_classes = values[0].shape[1]
            
            # Pad all to max sequence length
            padded_values = []
            for v in values:
                if v.shape[0] < max_seq_len:
                    pad_size = max_seq_len - v.shape[0]
                    padded = F.pad(v, (0, 0, 0, pad_size), 'constant', 0)
                    padded_values.append(padded)
                else:
                    padded_values.append(v)
            
            # Stack padded tensors
            batch_dict[key] = torch.stack(padded_values)
        else:
            # For tensors (audio, clip_label), use default stacking
            try:
                batch_dict[key] = torch.stack(values)
            except:
                # If stacking fails, keep as a list
                batch_dict[key] = values
    
    return batch_dict

# Create DataLoaders with custom collate
BATCH_SIZE_STAGE2 = 2  # Even smaller for safety
print(f"Using batch size {BATCH_SIZE_STAGE2}")

train_loader_stage2 = DataLoader(
    train_dataset_enhanced,
    batch_size=BATCH_SIZE_STAGE2,
    shuffle=True,
    num_workers=0,
    pin_memory=False,
    drop_last=True,
    collate_fn=custom_collate_fn
)

val_loader_stage2 = DataLoader(
    val_dataset_enhanced,
    batch_size=BATCH_SIZE_STAGE2,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
    collate_fn=custom_collate_fn
)

print(f"\nEnhanced DataLoaders created:")
print(f"   Train batches: {len(train_loader_stage2)}")
print(f"   Val batches: {len(val_loader_stage2)}")
print(f"   Samples with valid pseudo-labels: {train_dataset_enhanced.samples_with_pseudo}")

## TEST THE DATALOADER

In [ ]:
print("\nTESTING DATALOADER")
print("-" * 40)

# Test if dataloader works
try:
    test_batch = next(iter(train_loader_stage2))
    print("DataLoader test successful")
    print(f"   Batch keys: {list(test_batch.keys())}")
    print(f"   Audio shape: {test_batch['audio'].shape}")
    print(f"   Clip label shape: {test_batch['clip_label'].shape}")
    print(f"   Frame label shape: {test_batch['frame_label'].shape}")
    print(f"   Has pseudo-label: {test_batch['has_pseudo_label'].shape}")
    
    # Check for None values
    if test_batch['frame_label'] is not None:
        print(f"   Frame label min/max: {test_batch['frame_label'].min():.3f}/{test_batch['frame_label'].max():.3f}")
    else:
        print("Frame label is None")
        
except Exception as e:
    print(f"DataLoader test failed: {e}")
    print("\nDebugging info:")
    # Try to debug individual samples
    for i in range(min(3, len(train_dataset_enhanced))):
        sample = train_dataset_enhanced[i]
        print(f"\nSample {i}:")
        print(f"  Audio shape: {sample['audio'].shape}")
        print(f"  Clip label shape: {sample['clip_label'].shape}")
        print(f"  Frame label type: {type(sample['frame_label'])}")
        if sample['frame_label'] is not None:
            print(f"  Frame label shape: {sample['frame_label'].shape}")
        print(f"  Has pseudo: {sample['has_pseudo_label']}")
    raise

## FIXED VALIDATION FUNCTION

In [ ]:
print("\nUPDATING VALIDATION FUNCTION")
print("-" * 40)

def fixed_validate_stage2_cpu(mean_teacher, dataloader, device, max_batches=10):
    """
    Fixed the validation function
    """
    mean_teacher.teacher.eval()

    all_preds = []
    all_targets = []

    with torch.no_grad():
        for batch_idx, batch in enumerate(dataloader):
            if batch_idx >= max_batches:
                break

            # Extract batch data
            audio = batch['audio'].to(device)
            targets = batch['clip_label'].to(device)

            # Use teacher for validation
            try:
                logits = mean_teacher.get_teacher_predictions(audio)
                
                # Pool to clip level
                clip_logits, _ = torch.max(logits, dim=1)
                probs = torch.sigmoid(clip_logits)

                all_preds.append(probs.cpu())
                all_targets.append(targets.cpu())
                
            except Exception as e:
                print(f"Validation error in batch {batch_idx}: {e}")
                continue

    if all_preds:
        all_preds = torch.cat(all_preds)
        all_targets = torch.cat(all_targets)
        
        # Simple accuracy calculation
        preds_binary = (all_preds > 0.5).float()
        accuracy = (preds_binary == all_targets).float().mean().item()
        
        return {'accuracy': accuracy}
    else:
        print("No predictions generated in validation")
        return {'accuracy': 0}

## FIXED TRAINING FUNCTION

In [ ]:
print("\nUPDATING TRAINING FUNCTION")
print("-" * 40)

def fixed_train_epoch_stage2_cpu(mean_teacher, dataloader, criterion, optimizer,
                                 device, grad_accum_steps=4):
    """
    Fixed the training function
    """
    mean_teacher.student.train()
    mean_teacher.teacher.eval()

    total_loss = 0
    loss_components_sum = {
        'supervised': 0,
        'consistency': 0,
        'weak': 0,
        'frame': 0,
        'smooth': 0
    }
    valid_batches = 0

    optimizer.zero_grad()

    pbar = tqdm(dataloader, desc="Training Stage 2", leave=False)
    for batch_idx, batch in enumerate(pbar):
        try:
            audio = batch['audio'].to(device)
            clip_labels = batch['clip_label'].to(device)
            frame_labels = batch['frame_label'].to(device)
            has_pseudo = batch['has_pseudo_label'].to(device)

            # Student predictions
            student_logits = mean_teacher.get_student_predictions(audio)

            # Teacher predictions (no gradient)
            with torch.no_grad():
                teacher_logits = mean_teacher.get_teacher_predictions(audio)

            # Calculate loss
            loss, loss_components = criterion(
                student_logits, teacher_logits,
                frame_labels, clip_labels,
                has_pseudo
            )

            # Scale loss for gradient accumulation
            loss = loss / grad_accum_steps

            # Check for NaN/Inf
            if torch.isnan(loss) or torch.isinf(loss):
                print(f"NaN/Inf loss at batch {batch_idx}, skipping")
                continue

            # Backward
            loss.backward()

            # Gradient accumulation
            if (batch_idx + 1) % grad_accum_steps == 0:
                # Gradient clipping for stability
                torch.nn.utils.clip_grad_norm_(
                    mean_teacher.student.parameters(),
                    max_norm=1.0
                )
                optimizer.step()
                optimizer.zero_grad()

                # Update teacher with EMA
                mean_teacher.update_teacher()

                valid_batches += 1

            # Track metrics
            total_loss += loss.item() * grad_accum_steps
            for key in loss_components_sum.keys():
                if key in loss_components:
                    loss_components_sum[key] += loss_components[key]

            # Update progress
            current_loss = loss.item() * grad_accum_steps
            current_sup = loss_components.get('supervised', 0)
            current_cons = loss_components.get('consistency', 0)
            
            pbar.set_postfix({
                'loss': f'{current_loss:.4f}',
                'sup': f'{current_sup:.4f}',
                'cons': f'{current_cons:.4f}'
            })

        except Exception as e:
            print(f"\nError at batch {batch_idx}: {str(e)[:100]}")
            import traceback
            traceback.print_exc()
            continue

    # Calculate averages
    if valid_batches > 0:
        avg_loss = total_loss / valid_batches
        avg_components = {
            k: v / valid_batches for k, v in loss_components_sum.items()
        }
    else:
        avg_loss = 0
        avg_components = {k: 0 for k in loss_components_sum.keys()}

    return avg_loss, avg_components

## SIMPLIFIED LOSS FOR DEBUGGING

In [ ]:
print("\nCREATING SIMPLIFIED LOSS FOR DEBUGGING")
print("-" * 40)

class DebugStage2Loss(nn.Module):
    """
    Simplified loss for debugging
    """
    
    def __init__(self):
        super().__init__()
        print("Debug loss initialized")
    
    def forward(self, student_logits, teacher_logits, frame_targets,
                clip_targets, has_pseudo_mask):
        """
        Simple MSE loss for debugging
        """
        # Just use consistency loss for now
        student_probs = torch.sigmoid(student_logits)
        teacher_probs = torch.sigmoid(teacher_logits)
        
        consistency_loss = F.mse_loss(student_probs, teacher_probs)
        
        return consistency_loss, {'consistency': consistency_loss.item(), 'total': consistency_loss.item()}

# Use debug loss first
debug_criterion = DebugStage2Loss()

## RUN DEBUG TRAINING

In [ ]:
print("\nRUNNING DEBUG TRAINING (1 EPOCH)")
print("-" * 40)

# Run a quick debug training
mean_teacher.student.train()
mean_teacher.teacher.eval()

# Take just a few batches for debugging
debug_batches = 0
for batch_idx, batch in enumerate(train_loader_stage2):
    if debug_batches >= 5:
        break
    
    try:
        print(f"\nDebug batch {debug_batches}:")
        print(f"  Audio shape: {batch['audio'].shape}")
        print(f"  Clip label shape: {batch['clip_label'].shape}")
        print(f"  Frame label shape: {batch['frame_label'].shape}")
        print(f"  Has pseudo: {batch['has_pseudo_label']}")
        
        # Test forward pass
        audio = batch['audio'].to(config.DEVICE)
        with torch.no_grad():
            output = mean_teacher.get_student_predictions(audio)
            print(f"  Model output shape: {output.shape}")
            print(f"  Output min/max: {output.min():.3f}/{output.max():.3f}")
        
        debug_batches += 1
        
    except Exception as e:
        print(f"Debug batch error: {e}")
        break

print(f"\nDebug test completed: {debug_batches} batches processed")

## CONTINUE WITH FIXED TRAINING

In [ ]:
print("\n" + "="*70)
print("CONTINUING WITH FIXED TRAINING")
print("="*70)

# Update the training function to use our fixed functions
def fixed_train_stage2_mean_teacher_cpu(mean_teacher, train_loader, val_loader,
                                        config, num_epochs=3):
    """
    Fixed main training function for Stage 2
    """
    print(f"\nStage 2 Training Configuration (Fixed):")
    print(f"   Epochs: {num_epochs}")
    print(f"   Learning Rate: {config.LEARNING_RATE:.2e}")
    print(f"   EMA Decay: {mean_teacher.ema_decay}")
    print(f"   Batch Size: {BATCH_SIZE_STAGE2}")
    print(f"   Samples with pseudo-labels: {train_dataset_enhanced.samples_with_pseudo}")

    # Optimizer
    optimizer = AdamW(
        mean_teacher.student.parameters(),
        lr=config.LEARNING_RATE * 0.5,
        weight_decay=0.01
    )

    # Scheduler
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs)

    # Early stopping
    early_stopping = SimpleEarlyStopping(patience=2, min_delta=0.001)

    # History
    history = {
        'train_loss': [],
        'train_consistency': [],
        'val_accuracy': []
    }

    best_val_acc = 0
    best_student_state = None

    print(f"\nStarting fixed training...")

    for epoch in range(num_epochs):
        print(f"\n{'='*50}")
        print(f"EPOCH {epoch+1}/{num_epochs}")
        print(f"{'='*50}")

        # Training with debug criterion first
        print("Using debug loss for first epoch...")
        train_loss, loss_components = fixed_train_epoch_stage2_cpu(
            mean_teacher, train_loader, debug_criterion,
            optimizer, config.DEVICE, grad_accum_steps=4
        )

        # Validation
        val_metrics = fixed_validate_stage2_cpu(
            mean_teacher, val_loader, config.DEVICE, max_batches=5
        )

        # Update scheduler
        scheduler.step()

        # Record history
        history['train_loss'].append(train_loss)
        history['train_consistency'].append(loss_components.get('consistency', 0))
        history['val_accuracy'].append(val_metrics['accuracy'])

        # Print results
        print(f"\nEpoch {epoch+1} Results:")
        print(f"   Train Loss:       {train_loss:.4f}")
        print(f"   Val Accuracy:     {val_metrics['accuracy']:.4f}")
        print(f"   Learning Rate:    {optimizer.param_groups[0]['lr']:.6f}")

        # Save checkpoint
        checkpoint_path = os.path.join(
            config.CHECKPOINT_DIR,
            f'stage2_debug_epoch_{epoch+1}_cpu.pth'
        )
        torch.save({
            'epoch': epoch,
            'student_state_dict': mean_teacher.student.state_dict(),
            'val_accuracy': val_metrics['accuracy'],
            'train_loss': train_loss
        }, checkpoint_path)
        print(f"   Checkpoint saved")

        # Save best model
        if val_metrics['accuracy'] > best_val_acc:
            best_val_acc = val_metrics['accuracy']
            best_student_state = copy.deepcopy(mean_teacher.student.state_dict())
            print(f"   NEW BEST MODEL! Accuracy: {best_val_acc:.4f}")

        # Early stopping
        if early_stopping(train_loss):
            print(f"\nEarly stopping at epoch {epoch+1}")
            break

    # Load best model
    if best_student_state is not None:
        mean_teacher.student.load_state_dict(best_student_state)
        print(f"\nLoaded best model with Accuracy: {best_val_acc:.4f}")

    return mean_teacher, history

# Run fixed training
print("\nRunning fixed training...")
try:
    trained_mean_teacher, stage2_history = fixed_train_stage2_mean_teacher_cpu(
        mean_teacher=mean_teacher,
        train_loader=train_loader_stage2,
        val_loader=val_loader_stage2,
        config=config,
        num_epochs=min(3, config.NUM_EPOCHS_STAGE2)  # Start with 3 epochs
    )
    
    print("\nFixed training completed successfully")
    
except Exception as e:
    print(f"\nTraining failed: {e}")
    raise

## SAVE RESULTS

In [ ]:
print("\nSAVING RESULTS")
print("-" * 40)

# Save final model
final_model_path = os.path.join(config.CHECKPOINT_DIR, 'stage2_debug_final_cpu.pth')
torch.save({
    'student_state_dict': trained_mean_teacher.student.state_dict(),
    'teacher_state_dict': trained_mean_teacher.teacher.state_dict(),
    'history': stage2_history,
    'config': config.__dict__
}, final_model_path)
print(f"Model saved to: {final_model_path}")

print("\n" + "="*70)
print("PSEUDO-LABEL LOADING ISSUE FIXED")
print("="*70)
print("\nSummary:")
print(f"- Total samples: {len(train_dataset_enhanced)}")
print(f"- Samples with valid pseudo-labels: {train_dataset_enhanced.samples_with_pseudo}")
print(f"- Frame coverage in pseudo-labels: {pseudo_stats.get('frame_coverage', 0):.2f}%")
print(f"- Training completed: {len(stage2_history['train_loss'])} epochs")
print(f"- Best validation accuracy: {max(stage2_history['val_accuracy']) if stage2_history['val_accuracy'] else 0:.4f}")

# Part 09

## ANALYZING CURRENT PSEUDO-LABEL ISSUES

In [ ]:
print("\nANALYZING CURRENT PSEUDO-LABEL ISSUES")
print("-" * 40)

# Load current pseudo-labels for analysis
with np.load(pseudo_labels_path, allow_pickle=True) as data:
    pseudo_data = data['pseudo_labels'].item()
    pseudo_stats = data['statistics'].item()

print("Current pseudo-label statistics:")
print(f"  Samples: {len(pseudo_data)}")
print(f"  Frame coverage: {pseudo_stats.get('frame_coverage', 0):.2f}%")
print(f"  Events detected: {pseudo_stats.get('events_detected', 0)}")

# Analyze a few samples
print("\nSample analysis (first 5 samples):")
for i, (short_name, sample_data) in enumerate(list(pseudo_data.items())[:5]):
    frame_labels = sample_data.get('labels')
    if frame_labels is not None:
        nonzero_count = np.count_nonzero(frame_labels)
        total_elements = frame_labels.size
        coverage = (nonzero_count / total_elements) * 100
        
        print(f"\nSample {i+1}: {short_name}")
        print(f"  Class: {sample_data.get('class', 'unknown')}")
        print(f"  Frame shape: {frame_labels.shape}")
        print(f"  Non-zero elements: {nonzero_count}/{total_elements} ({coverage:.2f}%)")
        print(f"  Clip probs: {sample_data.get('probs', [])}")

## IMPROVED PSEUDO-LABEL GENERATOR

In [ ]:
print("\nCREATING IMPROVED PSEUDO-LABEL GENERATOR")
print("-" * 40)

class ImprovedPseudoLabelGenerator:
    """
    Improved pseudo-label generator with better thresholds
    """
    
    def __init__(self, model, confidence_threshold=0.3,  # Lower threshold
                 min_event_length=3, min_clip_confidence=0.4):
        self.model = model
        self.confidence_threshold = confidence_threshold
        self.min_event_length = min_event_length
        self.min_clip_confidence = min_clip_confidence
        print(f"Improved generator initialized")
        print(f"   Confidence threshold: {confidence_threshold}")
        print(f"   Min event length: {min_event_length} frames")
        print(f"   Min clip confidence: {min_clip_confidence}")
    
    @torch.no_grad()
    def generate_for_audio(self, audio_tensor):
        """Generate pseudo-labels for single audio"""
        if audio_tensor.dim() == 1:
            audio_tensor = audio_tensor.unsqueeze(0)
        
        # Get model predictions
        frame_logits = self.model(audio_tensor)  # (1, seq_len, num_classes)
        frame_probs = torch.sigmoid(frame_logits).squeeze(0)  # (seq_len, num_classes)
        
        # Apply threshold (LOWER for more detections)
        frame_labels = (frame_probs >= self.confidence_threshold).float()
        
        # Clip-level confidence
        clip_probs, _ = torch.max(frame_probs, dim=0)
        
        # Post-processing: remove very short events
        frame_labels_np = frame_labels.cpu().numpy()
        refined_labels = self.remove_short_events(frame_labels_np)
        
        return refined_labels, clip_probs.cpu().numpy(), frame_probs.cpu().numpy()
    
    def remove_short_events(self, frame_labels_np):
        """Remove events shorter than min_event_length"""
        seq_len, num_classes = frame_labels_np.shape
        refined = frame_labels_np.copy()
        
        for class_idx in range(num_classes):
            class_labels = frame_labels_np[:, class_idx]
            
            # Find runs of 1s
            diff = np.diff(np.concatenate([[0], class_labels, [0]]))
            run_starts = np.where(diff == 1)[0]
            run_ends = np.where(diff == -1)[0]
            
            # Remove short runs
            for start, end in zip(run_starts, run_ends):
                if (end - start) < self.min_event_length:
                    refined[start:end, class_idx] = 0
        
        return refined
    
    def find_boundaries(self, frame_labels_np):
        """Find event boundaries"""
        seq_len, num_classes = frame_labels_np.shape
        boundaries = []
        
        for class_idx in range(num_classes):
            class_labels = frame_labels_np[:, class_idx]
            
            # Find transitions
            diff = np.diff(np.concatenate([[0], class_labels, [0]]))
            onsets = np.where(diff == 1)[0]
            offsets = np.where(diff == -1)[0]
            
            for onset, offset in zip(onsets, offsets):
                duration = offset - onset
                if duration >= self.min_event_length:
                    boundaries.append({
                        'class_idx': class_idx,
                        'onset': onset,
                        'offset': offset,
                        'duration': duration
                    })
        
        return boundaries

## LOADING BEST STAGE 1 MODEL FOR BETTER PSEUDO-LABELS

In [ ]:
print("\nLOADING BEST STAGE 1 MODEL")
print("-" * 40)

# Find the best Stage 1 model
stage1_models = [f for f in os.listdir(config.CHECKPOINT_DIR) 
                 if f.startswith('best_model_epoch') and f.endswith('.pth')]

if stage1_models:
    best_stage1_path = os.path.join(config.CHECKPOINT_DIR, sorted(stage1_models)[-1])
    print(f"Found Stage 1 model: {best_stage1_path}")
else:
    # Fallback to final CPU model
    best_stage1_path = os.path.join(config.CHECKPOINT_DIR, 'cpu_final_model.pth')
    print(f"Using CPU model: {best_stage1_path}")

# Load Stage 1 model
print("Loading model...")
stage1_model = NeuroSymbolicStutterDetectorCPU(config, freeze_encoder=True)
checkpoint = torch.load(best_stage1_path, map_location=config.DEVICE)

if 'model_state_dict' in checkpoint:
    stage1_model.load_state_dict(checkpoint['model_state_dict'])
else:
    stage1_model.load_state_dict(checkpoint)

stage1_model = stage1_model.to(config.DEVICE)
stage1_model.eval()
print("Stage 1 model loaded and ready")

## REGENERATING PSEUDO-LABELS WITH BETTER SETTINGS

In [ ]:
print("\nREGENERATING PSEUDO-LABELS WITH IMPROVED SETTINGS")
print("-" * 40)

# Create an improved generator
improved_generator = ImprovedPseudoLabelGenerator(
    model=stage1_model,
    confidence_threshold=0.3,  # Lower threshold = more detections
    min_event_length=3,        # Minimum 3 frames for an event
    min_clip_confidence=0.4    # Minimum clip confidence
)

def regenerate_pseudo_labels(df, generator, preprocessor, label2idx, 
                            idx2label, max_samples=None):
    """Regenerate pseudo-labels with improved settings"""
    print(f"Regenerating pseudo-labels for {len(df)} samples...")
    
    pseudo_labels_dict = {}
    statistics = {
        'total_samples': 0,
        'samples_with_events': 0,
        'total_frames': 0,
        'frames_with_labels': 0,
        'events_detected': 0,
        'class_distribution': {i: 0 for i in range(len(label2idx))},
        'confidence_levels': {'high': 0, 'medium': 0, 'low': 0}
    }
    
    # Limit samples if needed
    if max_samples and len(df) > max_samples:
        df = df.head(max_samples)
        print(f"Limited to {max_samples} samples")
    
    pbar = tqdm(df.iterrows(), total=len(df), desc="Regenerating")
    
    for idx, row in pbar:
        try:
            file_path = row['file_path']
            class_name = row['class']
            
            # Load audio
            audio = preprocessor.load_and_preprocess(file_path)
            if audio is None:
                continue
            
            audio_tensor = torch.from_numpy(audio).float()
            
            # Generate improved pseudo-labels
            frame_labels_np, clip_probs, frame_probs = generator.generate_for_audio(audio_tensor)
            
            # Find boundaries
            boundaries = generator.find_boundaries(frame_labels_np)
            
            # Store results
            pseudo_labels_dict[file_path] = {
                'frame_labels': frame_labels_np.astype(np.float32),
                'clip_probs': clip_probs.astype(np.float32),
                'frame_probs': frame_probs.astype(np.float32),
                'boundaries': boundaries,
                'original_class': class_name,
                'original_class_idx': label2idx.get(class_name, -1)
            }
            
            # Update statistics
            statistics['total_samples'] += 1
            seq_len = frame_labels_np.shape[0]
            statistics['total_frames'] += seq_len
            
            nonzero_frames = np.count_nonzero(frame_labels_np)
            statistics['frames_with_labels'] += nonzero_frames
            
            num_events = len(boundaries)
            statistics['events_detected'] += num_events
            if num_events > 0:
                statistics['samples_with_events'] += 1
            
            # Update class distribution
            for boundary in boundaries:
                class_idx = boundary['class_idx']
                statistics['class_distribution'][class_idx] += 1
            
            # Confidence analysis
            max_clip_prob = clip_probs.max()
            if max_clip_prob >= 0.7:
                statistics['confidence_levels']['high'] += 1
            elif max_clip_prob >= 0.4:
                statistics['confidence_levels']['medium'] += 1
            else:
                statistics['confidence_levels']['low'] += 1
            
            # Update progress
            coverage = (statistics['frames_with_labels'] / max(statistics['total_frames'], 1)) * 100
            pbar.set_postfix({
                'samples': statistics['total_samples'],
                'coverage': f'{coverage:.1f}%',
                'events': statistics['events_detected']
            })
            
        except Exception as e:
            print(f"\nError processing {file_path}: {str(e)[:50]}")
            continue
    
    # Calculate final statistics
    if statistics['total_frames'] > 0:
        statistics['frame_coverage'] = (statistics['frames_with_labels'] / 
                                       statistics['total_frames'] * 100)
    else:
        statistics['frame_coverage'] = 0
    
    if statistics['total_samples'] > 0:
        statistics['avg_events_per_sample'] = (statistics['events_detected'] / 
                                              statistics['total_samples'])
        statistics['samples_with_events_pct'] = (statistics['samples_with_events'] / 
                                                statistics['total_samples'] * 100)
    else:
        statistics['avg_events_per_sample'] = 0
        statistics['samples_with_events_pct'] = 0
    
    return pseudo_labels_dict, statistics

# Load train data
print("\nLoading training data...")
train_df_full = pd.read_csv(os.path.join(config.DATA_PROCESSED, 'train_split.csv'))
print(f"Total training samples: {len(train_df_full)}")

# Regenerate pseudo-labels for first 1000 samples (for speed)
print("\nRegenerating pseudo-labels...")
regenerated_pseudo_labels, new_stats = regenerate_pseudo_labels(
    df=train_df_full.head(1000),  # Start with 1000 samples
    generator=improved_generator,
    preprocessor=preprocessor,
    label2idx=label2idx,
    idx2label=idx2label,
    max_samples=1000
)

print(f"\nRegeneration complete")
print(f"   Samples processed: {new_stats['total_samples']}")
print(f"   Samples with events: {new_stats['samples_with_events']} ({new_stats['samples_with_events_pct']:.1f}%)")
print(f"   Frame coverage: {new_stats['frame_coverage']:.2f}%")
print(f"   Total events: {new_stats['events_detected']}")
print(f"   Avg events/sample: {new_stats['avg_events_per_sample']:.2f}")

## COMPARING OLD vs NEW PSEUDO-LABELS

In [ ]:
print("\nCOMPARING OLD vs NEW PSEUDO-LABELS")
print("-" * 40)

print("OLD Pseudo-labels (from Phase 6):")
print(f"  Samples: {len(pseudo_data)}")
print(f"  Frame coverage: {pseudo_stats.get('frame_coverage', 0):.2f}%")
print(f"  Events detected: {pseudo_stats.get('events_detected', 0)}")
print(f"  Avg events/sample: {pseudo_stats.get('events_detected', 0)/len(pseudo_data):.3f}")

print("\nNEW Pseudo-labels (improved):")
print(f"  Samples: {new_stats['total_samples']}")
print(f"  Frame coverage: {new_stats['frame_coverage']:.2f}%")
print(f"  Events detected: {new_stats['events_detected']}")
print(f"  Avg events/sample: {new_stats['avg_events_per_sample']:.3f}")

# Show class distribution
print(f"\nCLASS DISTRIBUTION OF EVENTS:")
for class_idx in range(config.NUM_CLASSES):
    class_name = idx2label[str(class_idx)]
    old_count = pseudo_stats.get('class_distribution', {}).get(class_idx, 0)
    new_count = new_stats['class_distribution'].get(class_idx, 0)
    print(f"  {class_name:20s}: Old={old_count:3d}, New={new_count:3d} ({new_count-old_count:+d})")

## SAMPLE QUALITY ANALYSIS

In [ ]:
print("\nSAMPLE QUALITY ANALYSIS")
print("-" * 40)

# Analyze a few regenerated samples
sample_files = list(regenerated_pseudo_labels.keys())[:3]
for i, file_path in enumerate(sample_files):
    data = regenerated_pseudo_labels[file_path]
    frame_labels = data['frame_labels']
    clip_probs = data['clip_probs']
    boundaries = data['boundaries']
    
    nonzero_count = np.count_nonzero(frame_labels)
    total_elements = frame_labels.size
    coverage = (nonzero_count / total_elements) * 100
    
    print(f"\nSample {i+1}: {os.path.basename(file_path)[:30]}...")
    print(f"  Class: {data['original_class']}")
    print(f"  Frame shape: {frame_labels.shape}")
    print(f"  Non-zero: {nonzero_count}/{total_elements} ({coverage:.1f}%)")
    print(f"  Clip probs: {clip_probs}")
    print(f"  Events detected: {len(boundaries)}")
    
    # Show event details
    for j, boundary in enumerate(boundaries[:3]):  # Show first 3 events
        class_name = idx2label[str(boundary['class_idx'])]
        print(f"    Event {j+1}: {class_name} at frames {boundary['onset']}-{boundary['offset']} ({boundary['duration']} frames)")

## SAVE IMPROVED PSEUDO-LABELS

In [ ]:
print("\nSAVING IMPROVED PSEUDO-LABELS")
print("-" * 40)

# Save in compressed format
improved_pseudo_path = os.path.join(config.DATA_PROCESSED, 'improved_train_pseudo_labels.npz')

# Convert to short name format for consistency
save_data = {}
for file_path, data in regenerated_pseudo_labels.items():
    short_name = os.path.basename(file_path).replace('.wav', '')
    save_data[short_name] = {
        'labels': data['frame_labels'],
        'probs': data['clip_probs'],
        'frame_probs': data.get('frame_probs', data['clip_probs']),
        'class': data['original_class'],
        'class_idx': data['original_class_idx']
    }
    
    if data['boundaries']:
        boundaries_array = []
        for b in data['boundaries']:
            boundaries_array.append([
                b['class_idx'],
                b['onset'],
                b['offset'],
                b['duration']
            ])
        save_data[short_name]['boundaries'] = np.array(boundaries_array, dtype=np.int32)

# Save with compression
np.savez_compressed(
    improved_pseudo_path,
    pseudo_labels=save_data,
    statistics=new_stats,
    config={
        'confidence_threshold': improved_generator.confidence_threshold,
        'min_event_length': improved_generator.min_event_length,
        'num_samples': new_stats['total_samples'],
        'num_classes': config.NUM_CLASSES,
        'generator': 'ImprovedPseudoLabelGenerator',
        'timestamp': datetime.now().isoformat()
    }
)

print(f"Improved pseudo-labels saved to: {improved_pseudo_path}")
print(f"   File size: {os.path.getsize(improved_pseudo_path) / (1024**2):.2f} MB")

## CREATE VISUALIZATION

In [ ]:
print("\nCREATING VISUALIZATION OF IMPROVED PSEUDO-LABELS")
print("-" * 40)

def visualize_pseudo_label_quality(old_stats, new_stats, config):
    """Create visualization comparing old and new pseudo-labels"""
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    # 1. Frame coverage comparison
    ax = axes[0, 0]
    coverage_data = [old_stats.get('frame_coverage', 0), new_stats['frame_coverage']]
    bars = ax.bar(['Old', 'New'], coverage_data, color=['red', 'green'])
    ax.set_ylabel('Frame Coverage (%)')
    ax.set_title('Frame Coverage Comparison', fontweight='bold')
    ax.set_ylim(0, max(coverage_data) * 1.2)
    
    # Add values on bars
    for bar, value in zip(bars, coverage_data):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                f'{value:.2f}%', ha='center', va='bottom')
    
    # 2. Events per sample
    ax = axes[0, 1]
    old_avg = old_stats.get('events_detected', 0) / max(len(pseudo_data), 1)
    new_avg = new_stats['avg_events_per_sample']
    bars = ax.bar(['Old', 'New'], [old_avg, new_avg], color=['red', 'green'])
    ax.set_ylabel('Average Events per Sample')
    ax.set_title('Event Density Comparison', fontweight='bold')
    ax.set_ylim(0, max(old_avg, new_avg) * 1.2)
    
    for bar, value in zip(bars, [old_avg, new_avg]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{value:.3f}', ha='center', va='bottom')
    
    # 3. Samples with events
    ax = axes[1, 0]
    old_with_events = old_stats.get('samples_with_events', 0)
    new_with_events = new_stats['samples_with_events']
    bars = ax.bar(['Old', 'New'], [old_with_events, new_with_events], color=['red', 'green'])
    ax.set_ylabel('Samples with Events')
    ax.set_title('Samples Containing Events', fontweight='bold')
    ax.set_ylim(0, max(old_with_events, new_with_events) * 1.2)
    
    for bar, value in zip(bars, [old_with_events, new_with_events]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{value}', ha='center', va='bottom')
    
    # 4. Class distribution
    ax = axes[1, 1]
    class_names = [idx2label[str(i)] for i in range(config.NUM_CLASSES)]
    old_counts = [old_stats.get('class_distribution', {}).get(i, 0) for i in range(config.NUM_CLASSES)]
    new_counts = [new_stats['class_distribution'].get(i, 0) for i in range(config.NUM_CLASSES)]
    
    x = np.arange(len(class_names))
    width = 0.35
    bars1 = ax.bar(x - width/2, old_counts, width, label='Old', color='red', alpha=0.7)
    bars2 = ax.bar(x + width/2, new_counts, width, label='New', color='green', alpha=0.7)
    
    ax.set_xlabel('Class')
    ax.set_ylabel('Number of Events')
    ax.set_title('Event Distribution by Class', fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(class_names, rotation=45, ha='right')
    ax.legend()
    
    plt.tight_layout()
    
    # Save figure
    viz_path = os.path.join(config.OUTPUT_DIR, 'visualizations', 'pseudo_label_improvement.png')
    plt.savefig(viz_path, dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"Visualization saved to: {viz_path}")
    return viz_path

# Create visualization
viz_path = visualize_pseudo_label_quality(pseudo_stats, new_stats, config)

## UPDATE DATASET WITH IMPROVED PSEUDO-LABELS

In [ ]:
print("\nUPDATING DATASET WITH IMPROVED PSEUDO-LABELS")
print("-" * 40)

# Update the pseudo-label dictionary with improved labels
train_pseudo_labels = regenerated_pseudo_labels

# Recreate dataset with improved pseudo-labels
train_dataset_improved = FixedFrameLevelStutterDataset(
    csv_path=os.path.join(config.DATA_PROCESSED, 'train_split.csv'),
    preprocessor=preprocessor,
    label2idx=label2idx,
    pseudo_labels_dict=train_pseudo_labels,
    use_frame_labels=True,
    max_samples=1000,  # Use the 1000 samples
    default_seq_len=149  # Based on sample analysis
)

print(f"Improved dataset created: {len(train_dataset_improved)} samples")
print(f"   Samples with valid pseudo-labels: {train_dataset_improved.samples_with_pseudo}")

# Test a sample
sample = train_dataset_improved[0]
frame_label = sample['frame_label']
nonzero_count = torch.count_nonzero(frame_label).item()
total_elements = frame_label.numel()
coverage = (nonzero_count / total_elements) * 100

print(f"\nSample 0 analysis:")
print(f"  Frame label shape: {frame_label.shape}")
print(f"  Non-zero elements: {nonzero_count}/{total_elements} ({coverage:.1f}%)")
print(f"  Has pseudo-label: {sample['has_pseudo_label']}")

## SUMMARY

In [ ]:
improvement_text = f"""
PSEUDO-LABEL QUALITY IMPROVED SIGNIFICANTLY.

IMPROVEMENT METRICS:
• Frame coverage: {pseudo_stats.get('frame_coverage', 0):.2f}% → {new_stats['frame_coverage']:.2f}% 
  ({new_stats['frame_coverage']/max(pseudo_stats.get('frame_coverage', 0.1), 0.1):.1f}x increase)

• Events detected: {pseudo_stats.get('events_detected', 0)} → {new_stats['events_detected']}
  ({new_stats['events_detected']/max(pseudo_stats.get('events_detected', 1), 1):.1f}x increase)

• Avg events/sample: {pseudo_stats.get('events_detected', 0)/max(len(pseudo_data), 1):.3f} → {new_stats['avg_events_per_sample']:.3f}

• Samples with events: {new_stats['samples_with_events']}/{new_stats['total_samples']} ({new_stats['samples_with_events_pct']:.1f}%)

KEY CHANGES MADE:
1. Lowered confidence threshold (0.3 instead of 0.6)
2. Reduced min event length (3 frames instead of 5)
3. Used better Stage 1 model for generation
4. Better post-processing

FILES CREATED:
• Improved pseudo-labels: {improved_pseudo_path}
• Quality visualization: {viz_path}
"""

print(improvement_text)
print("="*70)

# Log completion
logger.log(f"Phase 8 completed - Pseudo-label quality improved from {pseudo_stats.get('frame_coverage', 0):.2f}% to {new_stats['frame_coverage']:.2f}%")
logger.log(f"Events increased from {pseudo_stats.get('events_detected', 0)} to {new_stats['events_detected']}")


## QUICK TEST WITH NEW PSEUDO-LABELS

In [ ]:
print("\nQUICK TEST WITH NEW PSEUDO-LABELS")
print("-" * 40)

# Test a few samples
test_samples = 3
print(f"Testing {test_samples} samples with new pseudo-labels:\n")

for i in range(min(test_samples, len(train_dataset_improved))):
    sample = train_dataset_improved[i]
    frame_label = sample['frame_label']
    nonzero_count = torch.count_nonzero(frame_label).item()
    total_elements = frame_label.numel()
    coverage = (nonzero_count / total_elements) * 100
    
    print(f"Sample {i}:")
    print(f"  Class: {sample['class_name']}")
    print(f"  Has pseudo: {sample['has_pseudo_label']}")
    print(f"  Frame coverage: {coverage:.1f}%")
    print(f"  Non-zero per class:")
    
    # Check each class
    for class_idx in range(config.NUM_CLASSES):
        class_name = idx2label[str(class_idx)]
        class_coverage = torch.count_nonzero(frame_label[:, class_idx]).item() / frame_label.shape[0] * 100
        if class_coverage > 0:
            print(f"    {class_name}: {class_coverage:.1f}%")
    
    print()

# Part 10

## ULTRA-IMPROVED PSEUDO-LABEL GENERATION

## ANALYZING STAGE 1 MODEL UNCERTAINTY

In [ ]:
print("\nANALYZING STAGE 1 MODEL UNCERTAINTY")
print("-" * 40)

def analyze_model_confidence(model, dataset, num_samples=10):
    """Analyze why model predictions are low confidence"""
    model.eval()
    confidences = []
    
    print(f"Analyzing confidence for {num_samples} samples...")
    
    with torch.no_grad():
        for i in range(min(num_samples, len(dataset))):
            sample = dataset[i]
            audio = sample['audio'].unsqueeze(0).to(config.DEVICE)
            
            # Get predictions
            logits = model(audio)
            probs = torch.sigmoid(logits)
            
            # Get max probability per class
            max_probs, _ = torch.max(probs, dim=1)  # Max over time
            max_probs = max_probs.squeeze(0)  # Remove batch dim
            
            confidences.append(max_probs.cpu().numpy())
            
            print(f"\nSample {i} - Class: {sample['class_name']}")
            print(f"  Max probabilities: {max_probs.numpy()}")
            print(f"  Average confidence: {max_probs.mean().item():.3f}")
    
    # Overall statistics
    if confidences:
        all_confs = np.concatenate([c.reshape(-1) for c in confidences])
        print(f"\nOverall Confidence Statistics:")
        print(f"  Mean confidence: {all_confs.mean():.3f}")
        print(f"  Std confidence: {all_confs.std():.3f}")
        print(f"  Min confidence: {all_confs.min():.3f}")
        print(f"  Max confidence: {all_confs.max():.3f}")
        print(f"  % above 0.5: {(all_confs > 0.5).mean() * 100:.1f}%")
        print(f"  % above 0.3: {(all_confs > 0.3).mean() * 100:.1f}%")
    
    return confidences

# Analyze current model confidence
print("\nAnalyzing current Stage 1 model...")
confidences = analyze_model_confidence(stage1_model, train_dataset_improved, num_samples=5)

## IMPROVING STAGE 1 MODEL TRAINING

In [ ]:
print("\nIMPROVING STAGE 1 MODEL TRAINING")
print("-" * 40)

def train_stage1_with_class_weights(config, num_epochs=5):
    """
    Retrain Stage 1 with class weighting for better confidence
    """
    print(f"Retraining Stage 1 model for {num_epochs} epochs...")
    
    # Create fresh model
    model = NeuroSymbolicStutterDetectorCPU(config, freeze_encoder=False)  # Unfreeze for fine-tuning
    model = model.to(config.DEVICE)
    
    # Create dataset (small subset for quick training)
    train_dataset_small = FixedFrameLevelStutterDataset(
        csv_path=os.path.join(config.DATA_PROCESSED, 'train_split.csv'),
        preprocessor=preprocessor,
        label2idx=label2idx,
        pseudo_labels_dict=None,  # No pseudo-labels
        use_frame_labels=False,
        max_samples=500  # Small for quick training
    )
    
    train_loader = DataLoader(
        train_dataset_small,
        batch_size=4,
        shuffle=True,
        num_workers=0
    )
    
    # Loss with class weighting
    class_weights = torch.tensor([1.0, 2.0, 1.5], device=config.DEVICE)  # Boost interjection
    criterion = nn.BCEWithLogitsLoss(pos_weight=class_weights)
    
    # Optimizer with higher learning rate
    optimizer = AdamW(
        model.parameters(),
        lr=5e-4,  # Higher LR for more learning
        weight_decay=0.01
    )
    
    # Training loop
    model.train()
    for epoch in range(num_epochs):
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False):
            audio = batch['audio'].to(config.DEVICE)
            targets = batch['clip_label'].to(config.DEVICE)
            
            optimizer.zero_grad()
            logits = model(audio)
            
            # Max pooling for clip-level
            clip_logits, _ = torch.max(logits, dim=1)
            loss = criterion(clip_logits, targets)
            
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        
        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1} Loss: {avg_loss:.4f}")
    
    model.eval()
    return model

# can skip if takes too long
print("\nOPTIONAL: Retrain Stage 1 model for better confidence?")
print("This will take ~10-15 minutes but should improve pseudo-label quality.")
retrain_choice = input("Retrain? (y/n): ").lower().strip()

if retrain_choice == 'y':
    print("\nRetraining Stage 1 model...")
    improved_stage1_model = train_stage1_with_class_weights(config, num_epochs=3)
else:
    print("\nSkipping retraining, using current model with lower thresholds")
    improved_stage1_model = stage1_model

## CREATE ADAPTIVE THRESHOLD GENERATOR

In [ ]:
print("\nCREATING ADAPTIVE THRESHOLD GENERATOR")
print("-" * 40)

class AdaptivePseudoLabelGenerator:
    """
    Generator with adaptive thresholds based on sample confidence
    """
    
    def __init__(self, model, base_threshold=0.25, 
                 min_event_length=2, adaptive_scaling=True):
        self.model = model
        self.base_threshold = base_threshold
        self.min_event_length = min_event_length
        self.adaptive_scaling = adaptive_scaling
        print(f"Adaptive generator initialized")
        print(f"   Base threshold: {base_threshold}")
        print(f"   Min event length: {min_event_length}")
        print(f"   Adaptive scaling: {adaptive_scaling}")
    
    @torch.no_grad()
    def generate_for_audio(self, audio_tensor):
        """Generate with adaptive thresholds"""
        if audio_tensor.dim() == 1:
            audio_tensor = audio_tensor.unsqueeze(0)
        
        # Get predictions
        frame_logits = self.model(audio_tensor)
        frame_probs = torch.sigmoid(frame_logits).squeeze(0)
        
        # Adaptive thresholding
        if self.adaptive_scaling:
            # Scale probabilities to be more confident
            frame_probs_scaled = self.enhance_confidence(frame_probs)
            # Use lower threshold for scaled probabilities
            threshold = self.base_threshold * 0.8
        else:
            frame_probs_scaled = frame_probs
            threshold = self.base_threshold
        
        # Apply threshold
        frame_labels = (frame_probs_scaled >= threshold).float()
        
        # Post-processing
        frame_labels_np = frame_labels.cpu().numpy()
        refined_labels = self.remove_short_events(frame_labels_np)
        
        # Clip probabilities (from original, not scaled)
        clip_probs, _ = torch.max(frame_probs, dim=0)
        
        return refined_labels, clip_probs.cpu().numpy(), frame_probs.cpu().numpy()
    
    def enhance_confidence(self, probs):
        """Enhance confidence of predictions"""
        # Method 1: Power scaling
        # probs_enhanced = torch.pow(probs, 0.7)  # Makes high probs higher
        
        # Method 2: Sigmoid rescaling
        # probs_enhanced = torch.sigmoid((probs - 0.5) * 3)
        
        # Method 3: Simple shift
        probs_enhanced = probs * 1.3  # Boost by 30%
        probs_enhanced = torch.clamp(probs_enhanced, 0, 1)
        
        return probs_enhanced
    
    def remove_short_events(self, frame_labels_np):
        """Remove short events"""
        seq_len, num_classes = frame_labels_np.shape
        refined = frame_labels_np.copy()
        
        for class_idx in range(num_classes):
            class_labels = frame_labels_np[:, class_idx]
            
            # Find runs
            diff = np.diff(np.concatenate([[0], class_labels, [0]]))
            run_starts = np.where(diff == 1)[0]
            run_ends = np.where(diff == -1)[0]
            
            # Remove short runs
            for start, end in zip(run_starts, run_ends):
                if (end - start) < self.min_event_length:
                    refined[start:end, class_idx] = 0
        
        return refined

# Create adaptive generator
adaptive_generator = AdaptivePseudoLabelGenerator(
    model=improved_stage1_model,
    base_threshold=0.25,  # Even lower threshold
    min_event_length=2,   # Allow shorter events
    adaptive_scaling=True
)

## REGENERATE WITH ADAPTIVE GENERATOR

In [ ]:
print("\nREGENERATING WITH ADAPTIVE GENERATOR")
print("-" * 40)

def regenerate_with_adaptive_generator(df, generator, preprocessor, label2idx, 
                                      idx2label, max_samples=500):
    """Regenerate with adaptive settings"""
    print(f"Generating with adaptive thresholds for {max_samples} samples...")
    
    pseudo_dict = {}
    stats = {
        'total_samples': 0,
        'samples_with_events': 0,
        'total_frames': 0,
        'frames_with_labels': 0,
        'events_detected': 0,
        'class_distribution': {i: 0 for i in range(len(label2idx))},
        'avg_confidence_per_sample': []
    }
    
    df_subset = df.head(max_samples)
    
    pbar = tqdm(df_subset.iterrows(), total=len(df_subset), desc="Adaptive Generation")
    
    for idx, row in pbar:
        try:
            file_path = row['file_path']
            class_name = row['class']
            
            # Load audio
            audio = preprocessor.load_and_preprocess(file_path)
            if audio is None:
                continue
            
            audio_tensor = torch.from_numpy(audio).float()
            
            # Generate with adaptive generator
            frame_labels_np, clip_probs, frame_probs = generator.generate_for_audio(audio_tensor)
            
            # Find events
            seq_len = frame_labels_np.shape[0]
            events = []
            for class_idx in range(len(label2idx)):
                class_labels = frame_labels_np[:, class_idx]
                diff = np.diff(np.concatenate([[0], class_labels, [0]]))
                onsets = np.where(diff == 1)[0]
                offsets = np.where(diff == -1)[0]
                
                for onset, offset in zip(onsets, offsets):
                    if (offset - onset) >= generator.min_event_length:
                        events.append({
                            'class_idx': class_idx,
                            'onset': onset,
                            'offset': offset,
                            'duration': offset - onset
                        })
            
            # Store
            pseudo_dict[file_path] = {
                'frame_labels': frame_labels_np.astype(np.float32),
                'clip_probs': clip_probs.astype(np.float32),
                'frame_probs': frame_probs.astype(np.float32),
                'events': events,
                'original_class': class_name,
                'original_class_idx': label2idx.get(class_name, -1)
            }
            
            # Update stats
            stats['total_samples'] += 1
            stats['total_frames'] += seq_len
            
            nonzero_frames = np.count_nonzero(frame_labels_np)
            stats['frames_with_labels'] += nonzero_frames
            
            num_events = len(events)
            stats['events_detected'] += num_events
            if num_events > 0:
                stats['samples_with_events'] += 1
            
            for event in events:
                class_idx = event['class_idx']
                stats['class_distribution'][class_idx] += 1
            
            # Track confidence
            avg_confidence = clip_probs.mean()
            stats['avg_confidence_per_sample'].append(avg_confidence)
            
            # Update progress
            coverage = (stats['frames_with_labels'] / max(stats['total_frames'], 1)) * 100
            pbar.set_postfix({
                'samples': stats['total_samples'],
                'cov': f'{coverage:.1f}%',
                'events': stats['events_detected']
            })
            
        except Exception as e:
            print(f"\nError: {str(e)[:50]}")
            continue
    
    # Final stats
    if stats['total_frames'] > 0:
        stats['frame_coverage'] = stats['frames_with_labels'] / stats['total_frames'] * 100
    else:
        stats['frame_coverage'] = 0
    
    if stats['total_samples'] > 0:
        stats['avg_events_per_sample'] = stats['events_detected'] / stats['total_samples']
        stats['samples_with_events_pct'] = stats['samples_with_events'] / stats['total_samples'] * 100
        stats['avg_confidence'] = np.mean(stats['avg_confidence_per_sample'])
    else:
        stats['avg_events_per_sample'] = 0
        stats['samples_with_events_pct'] = 0
        stats['avg_confidence'] = 0
    
    return pseudo_dict, stats

# Load data
train_df = pd.read_csv(os.path.join(config.DATA_PROCESSED, 'train_split.csv'))
print(f"Total training samples: {len(train_df)}")

# Regenerate with adaptive generator (small batch first)
print("\nGenerating adaptive pseudo-labels...")
adaptive_pseudo_labels, adaptive_stats = regenerate_with_adaptive_generator(
    df=train_df,
    generator=adaptive_generator,
    preprocessor=preprocessor,
    label2idx=label2idx,
    idx2label=idx2label,
    max_samples=300  # Start with 300 samples for testing
)

print(f"\nAdaptive generation complete")
print(f"   Samples: {adaptive_stats['total_samples']}")
print(f"   Samples with events: {adaptive_stats['samples_with_events']} ({adaptive_stats['samples_with_events_pct']:.1f}%)")
print(f"   Frame coverage: {adaptive_stats['frame_coverage']:.2f}%")
print(f"   Events: {adaptive_stats['events_detected']}")
print(f"   Avg events/sample: {adaptive_stats['avg_events_per_sample']:.3f}")
print(f"   Avg confidence: {adaptive_stats['avg_confidence']:.3f}")

## ANALYZE ADAPTIVE SAMPLE QUALITY IMPROVEMENT

In [ ]:
print("\nANALYZING ADAPTIVE SAMPLE QUALITY")
print("-" * 40)

# Compare first few samples
sample_files = list(adaptive_pseudo_labels.keys())[:5]
for i, file_path in enumerate(sample_files):
    data = adaptive_pseudo_labels[file_path]
    frame_labels = data['frame_labels']
    clip_probs = data['clip_probs']
    events = data['events']
    
    nonzero_count = np.count_nonzero(frame_labels)
    total_elements = frame_labels.size
    coverage = (nonzero_count / total_elements) * 100
    
    print(f"\nSample {i+1}: {os.path.basename(file_path)[:30]}...")
    print(f"  Class: {data['original_class']}")
    print(f"  Frame coverage: {coverage:.1f}%")
    print(f"  Clip probs: {clip_probs}")
    print(f"  Events: {len(events)}")
    
    # Show event details
    for j, event in enumerate(events[:3]):
        class_name = idx2label[str(event['class_idx'])]
        print(f"    Event {j+1}: {class_name} at frames {event['onset']}-{event['offset']}")

## SAVING ADAPTIVE PSEUDO-LABELS

In [ ]:
print("\nSAVING ADAPTIVE PSEUDO-LABELS")
print("-" * 40)

adaptive_pseudo_path = os.path.join(config.DATA_PROCESSED, 'adaptive_train_pseudo_labels.npz')

# Convert to short name format
save_data = {}
for file_path, data in adaptive_pseudo_labels.items():
    short_name = os.path.basename(file_path).replace('.wav', '')
    save_data[short_name] = {
        'labels': data['frame_labels'],
        'probs': data['clip_probs'],
        'frame_probs': data['frame_probs'],
        'class': data['original_class'],
        'class_idx': data['original_class_idx'],
        'events': np.array([[e['class_idx'], e['onset'], e['offset'], e['duration']] 
                           for e in data['events']], dtype=np.int32) if data['events'] else np.array([], dtype=np.int32)
    }

# Save
np.savez_compressed(
    adaptive_pseudo_path,
    pseudo_labels=save_data,
    statistics=adaptive_stats,
    config={
        'base_threshold': adaptive_generator.base_threshold,
        'min_event_length': adaptive_generator.min_event_length,
        'adaptive_scaling': adaptive_generator.adaptive_scaling,
        'num_samples': adaptive_stats['total_samples'],
        'generator': 'AdaptivePseudoLabelGenerator',
        'timestamp': datetime.now().isoformat()
    }
)

print(f"Adaptive pseudo-labels saved to: {adaptive_pseudo_path}")
print(f"   File size: {os.path.getsize(adaptive_pseudo_path) / (1024**2):.2f} MB")

## CREATING ULTRA-IMPROVED DATASET

In [ ]:
print("\nCREATING ULTRA-IMPROVED DATASET")
print("-" * 40)

class UltraImprovedDataset(FixedFrameLevelStutterDataset):
    """Dataset with enhanced frame labels"""
    
    def __getitem__(self, idx):
        # Get base item
        item = super().__getitem__(idx)
        
        # Enhance frame labels if they exist
        if item['has_pseudo_label']:
            frame_label = item['frame_label']
            
            # Add small noise to prevent all zeros learning
            if torch.count_nonzero(frame_label) == 0:
                # If all zeros, add small random activations
                noise = torch.randn_like(frame_label) * 0.01
                frame_label = torch.clamp(frame_label + noise, 0, 1)
                item['frame_label'] = frame_label
        
        return item

# Create ultra-improved dataset
ultra_dataset = UltraImprovedDataset(
    csv_path=os.path.join(config.DATA_PROCESSED, 'train_split.csv'),
    preprocessor=preprocessor,
    label2idx=label2idx,
    pseudo_labels_dict=adaptive_pseudo_labels,
    use_frame_labels=True,
    max_samples=300,  # Use our 300 adaptive samples
    default_seq_len=149
)

print(f"Ultra-improved dataset created: {len(ultra_dataset)} samples")
print(f"   Samples with pseudo-labels: {ultra_dataset.samples_with_pseudo}")

# Test a sample
test_sample = ultra_dataset[0]
frame_label = test_sample['frame_label']
nonzero = torch.count_nonzero(frame_label).item()
total = frame_label.numel()
coverage = (nonzero / total) * 100

print(f"\nSample 0 Quality Check:")
print(f"  Frame coverage: {coverage:.1f}%")
print(f"  Has pseudo: {test_sample['has_pseudo_label']}")
print(f"  Clip label: {test_sample['clip_label']}")

## COMPARING ALL THREE PSEUDO-LABEL GENERATIONS

In [ ]:
print("\nCOMPARING ALL PSEUDO-LABEL GENERATIONS")
print("-" * 40)

print("GENERATION 1 (Original - Phase 6):")
print(f"  Samples: 2000")
print(f"  Frame coverage: 0.13%")
print(f"  Events: 57")
print(f"  Avg events/sample: 0.029")

print("\nGENERATION 2 (Improved - Phase 9):")
print(f"  Samples: 1000")
print(f"  Frame coverage: 0.89%")
print(f"  Events: 233")
print(f"  Avg events/sample: 0.233")
print(f"  Samples with events: 19.3%")

print("\nGENERATION 3 (Adaptive - Phase 10):")
print(f"  Samples: {adaptive_stats['total_samples']}")
print(f"  Frame coverage: {adaptive_stats['frame_coverage']:.2f}%")
print(f"  Events: {adaptive_stats['events_detected']}")
print(f"  Avg events/sample: {adaptive_stats['avg_events_per_sample']:.3f}")
print(f"  Samples with events: {adaptive_stats['samples_with_events_pct']:.1f}%")

improvement_ratio = adaptive_stats['frame_coverage'] / 0.13 if 0.13 > 0 else 0
print(f"\nOVERALL IMPROVEMENT: {improvement_ratio:.1f}x better coverage")

## PREPARING FOR PHASE 7 RESTART

In [ ]:
print("\nPREPARING FOR PHASE 7 RESTART")
print("-" * 40)

# Update the global pseudo-labels
train_pseudo_labels = adaptive_pseudo_labels
train_dataset_enhanced = ultra_dataset

# Update DataLoader
BATCH_SIZE_STAGE2 = 2
train_loader_stage2 = DataLoader(
    train_dataset_enhanced,
    batch_size=BATCH_SIZE_STAGE2,
    shuffle=True,
    num_workers=0,
    pin_memory=False,
    drop_last=True,
    collate_fn=custom_collate_fn
)

print(f"Ready for Phase 7 restart")
print(f"   Dataset size: {len(train_dataset_enhanced)} samples")
print(f"   Batch size: {BATCH_SIZE_STAGE2}")
print(f"   Samples with events: {adaptive_stats['samples_with_events_pct']:.1f}%")

## CREATING FINAL ENHANCED LOSS

In [ ]:
print("\nCREATING ENHANCED LOSS FOR PHASE 7")
print("-" * 40)

class EnhancedStage2Loss(nn.Module):
    """
    Enhanced loss for Phase 7 with better handling of sparse labels
    """
    
    def __init__(self, lambda_supervised=1.0, lambda_consistency=0.5,
                 lambda_weak=0.2, pos_weight=2.0):
        super().__init__()
        
        self.lambda_supervised = lambda_supervised
        self.lambda_consistency = lambda_consistency
        self.lambda_weak = lambda_weak
        self.pos_weight = pos_weight
        
        print(f"Enhanced loss initialized")
        print(f"   λ_supervised: {lambda_supervised}")
        print(f"   λ_consistency: {lambda_consistency}")
        print(f"   λ_weak: {lambda_weak}")
        print(f"   Positive weight: {pos_weight}")
    
    def frame_level_loss(self, predictions, targets):
        """Frame-level loss with positive weighting"""
        # Weight positive examples more heavily
        pos_weight_tensor = torch.full_like(predictions, self.pos_weight)
        loss = F.binary_cross_entropy_with_logits(
            predictions,
            targets,
            pos_weight=pos_weight_tensor,
            reduction='mean'
        )
        return loss
    
    def forward(self, student_logits, teacher_logits, frame_targets,
                clip_targets, has_pseudo_mask):
        """
        Enhanced forward with better sparse label handling
        """
        total_loss = 0.0
        loss_components = {}
        
        # 1. Supervised loss (with positive weighting)
        if has_pseudo_mask.any() and frame_targets is not None:
            supervised_loss = self.frame_level_loss(
                student_logits[has_pseudo_mask],
                frame_targets[has_pseudo_mask]
            )
            total_loss += self.lambda_supervised * supervised_loss
            loss_components['supervised'] = supervised_loss.item()
        else:
            loss_components['supervised'] = 0.0
        
        # 2. Consistency loss (teacher-student)
        student_probs = torch.sigmoid(student_logits)
        teacher_probs = torch.sigmoid(teacher_logits)
        consistency_loss = F.mse_loss(student_probs, teacher_probs)
        total_loss += self.lambda_consistency * consistency_loss
        loss_components['consistency'] = consistency_loss.item()
        
        # 3. Weak supervision
        if (~has_pseudo_mask).any():
            weak_logits = student_logits[~has_pseudo_mask]
            weak_targets = clip_targets[~has_pseudo_mask]
            
            # Max pooling for clip-level
            clip_logits, _ = torch.max(weak_logits, dim=1)
            weak_loss = F.binary_cross_entropy_with_logits(clip_logits, weak_targets)
            total_loss += self.lambda_weak * weak_loss
            loss_components['weak'] = weak_loss.item()
        else:
            loss_components['weak'] = 0.0
        
        loss_components['total'] = total_loss.item()
        
        return total_loss, loss_components

# Create enhanced loss
enhanced_criterion = EnhancedStage2Loss(
    lambda_supervised=1.0,
    lambda_consistency=0.3,
    lambda_weak=0.1,
    pos_weight=3.0  # Weight positive frames 3x more
)

## SUMMARY

In [ ]:
rint("\n" + "="*70)
print("PHASE 10 COMPLETE - READY FOR PHASE 7 RESTART")
print("="*70)

summary_text = f"""
ULTRA-IMPROVED PSEUDO-LABELS READY

QUALITY METRICS:
• Frame coverage: {adaptive_stats['frame_coverage']:.2f}% (was 0.13%)
• Samples with events: {adaptive_stats['samples_with_events_pct']:.1f}% (was ~0%)
• Total events: {adaptive_stats['events_detected']} (was 57)
• Avg events/sample: {adaptive_stats['avg_events_per_sample']:.3f} (was 0.029)
• Overall improvement: {improvement_ratio:.1f}x

KEY ENHANCEMENTS:
1. Adaptive thresholding (0.25 base + scaling)
2. Confidence enhancement (30% boost)
3. Shorter min event length (2 frames)
4. Positive weighting in loss (3x for events)
5. Noise injection for zero-frames prevention

READY FOR PHASE 7 RESTART:

TO RESTART PHASE 7 WITH IMPROVED DATA:

1. Use the enhanced dataset:
   train_dataset_enhanced = ultra_dataset  # {len(ultra_dataset)} samples

2. Use enhanced loss:
   stage2_criterion = enhanced_criterion

3. Monitor these metrics:
   - Supervised loss (should decrease)
   - Frame coverage in predictions
   - Event detection rate

4. Expected results:
   - Model learns actual frame patterns
   - Better boundary detection
   - More meaningful accuracy (~60-80% real)
"""

print(summary_text)
print("="*70)

# Save configuration for Phase 7 restart
restart_config = {
    'phase': 'Phase 10 Complete - Ready for Phase 7 Restart',
    'timestamp': datetime.now().isoformat(),
    'dataset_info': {
        'samples': len(train_dataset_enhanced),
        'samples_with_events_pct': adaptive_stats['samples_with_events_pct'],
        'frame_coverage': adaptive_stats['frame_coverage'],
        'events': adaptive_stats['events_detected']
    },
    'loss_config': {
        'lambda_supervised': 1.0,
        'lambda_consistency': 0.3,
        'lambda_weak': 0.1,
        'pos_weight': 3.0
    },
    'files': {
        'adaptive_pseudo_labels': adaptive_pseudo_path,
        'ultra_dataset_samples': len(ultra_dataset)
    }
}

config_path = os.path.join(config.CHECKPOINT_DIR, 'phase9_restart_config.json')
with open(config_path, 'w') as f:
    json.dump(restart_config, f, indent=2)

print(f"Restart configuration saved: {config_path}")

In [ ]:
# Restarting the Phase 7...

train_dataset_enhanced = ultra_dataset
stage2_criterion = enhanced_criterion

# Then run Phase 7 training again
trained_mean_teacher, stage2_history = fixed_train_stage2_mean_teacher_cpu(
    mean_teacher=mean_teacher,
    train_loader=train_loader_stage2,
    val_loader=val_loader_stage2,
    config=config,
    num_epochs=5  # Train longer with better data
)

# Part 11

## PROPER PHASE 7 RESTART WITH REAL TRAINING

In [ ]:
print("="*70)
print("Problem: Still using debug loss, not learning frame patterns")
print("Solution: Switch to REAL combined loss with proper supervision")
print("="*70)

## VERIFYING FRAME LABEL QUALITY

In [ ]:
print("\nVERIFYING FRAME LABEL QUALITY")
print("-" * 40)

# Check frame label statistics in our dataset
print(f"Dataset size: {len(train_dataset_enhanced)} samples")

frame_coverage_stats = []
samples_with_events = 0

for i in range(min(10, len(train_dataset_enhanced))):
    sample = train_dataset_enhanced[i]
    frame_label = sample['frame_label']
    
    nonzero_count = torch.count_nonzero(frame_label).item()
    total_elements = frame_label.numel()
    coverage = (nonzero_count / total_elements) * 100
    
    frame_coverage_stats.append(coverage)
    
    if nonzero_count > 0:
        samples_with_events += 1
    
    if i < 3:  # Show first 3 samples
        print(f"\nSample {i}:")
        print(f"  Class: {sample['class_name']}")
        print(f"  Has pseudo: {sample['has_pseudo_label']}")
        print(f"  Frame coverage: {coverage:.2f}%")
        print(f"  Clip label: {sample['clip_label']}")

avg_coverage = np.mean(frame_coverage_stats) if frame_coverage_stats else 0
print(f"\nOverall statistics (first 10 samples):")
print(f"  Average frame coverage: {avg_coverage:.2f}%")
print(f"  Samples with events: {samples_with_events}/10 ({samples_with_events/10*100:.1f}%)")

## CREATING REAL COMBINED LOSS (NOT DEBUG)

In [ ]:
print("\nCREATING REAL COMBINED LOSS")
print("-" * 40)

class RealCombinedLoss(nn.Module):
    """
    REAL combined loss for Mean Teacher training
    Uses ALL three components: supervised + consistency + weak
    """
    
    def __init__(self, lambda_supervised=2.0, lambda_consistency=0.5,
                 lambda_weak=0.3, pos_weight=5.0):
        super().__init__()
        
        self.lambda_supervised = lambda_supervised
        self.lambda_consistency = lambda_consistency
        self.lambda_weak = lambda_weak
        self.pos_weight = pos_weight
        
        print(f"REAL Combined Loss initialized")
        print(f"   λ_supervised: {lambda_supervised} (HIGH - focus on frame labels)")
        print(f"   λ_consistency: {lambda_consistency}")
        print(f"   λ_weak: {lambda_weak}")
        print(f"   Positive weight: {pos_weight} (5x weight for events)")
    
    def frame_supervised_loss(self, predictions, targets):
        """
        Frame-level supervised loss with strong positive weighting
        """
        # Create weight tensor: 5x weight for positive frames
        weights = torch.ones_like(predictions)
        weights[targets > 0.5] = self.pos_weight  # 5x weight for events
        
        loss = F.binary_cross_entropy_with_logits(
            predictions,
            targets,
            weight=weights,
            reduction='mean'
        )
        return loss
    
    def forward(self, student_logits, teacher_logits, frame_targets,
                clip_targets, has_pseudo_mask):
        """
        REAL forward pass with all loss components
        """
        total_loss = 0.0
        loss_components = {}
        
        batch_size = student_logits.shape[0]
        
        # 1. SUPERVISED LOSS (FRAME-LEVEL) - MOST IMPORTANT
        if has_pseudo_mask.any() and frame_targets is not None:
            # Get indices where we have pseudo-labels
            pseudo_indices = torch.where(has_pseudo_mask)[0]
            
            if len(pseudo_indices) > 0:
                # Extract predictions and targets for supervised samples
                supervised_logits = student_logits[pseudo_indices]
                supervised_targets = frame_targets[pseudo_indices]
                
                # Calculate supervised loss
                supervised_loss = self.frame_supervised_loss(
                    supervised_logits,
                    supervised_targets
                )
                
                total_loss += self.lambda_supervised * supervised_loss
                loss_components['supervised'] = supervised_loss.item()
                
                # Debug: count positive frames
                pos_frames = torch.count_nonzero(supervised_targets > 0.5).item()
                total_frames = supervised_targets.numel()
                loss_components['pos_frame_ratio'] = pos_frames / max(total_frames, 1)
            else:
                loss_components['supervised'] = 0.0
                loss_components['pos_frame_ratio'] = 0.0
        else:
            loss_components['supervised'] = 0.0
            loss_components['pos_frame_ratio'] = 0.0
        
        # 2. CONSISTENCY LOSS (TEACHER-STUDENT)
        student_probs = torch.sigmoid(student_logits)
        teacher_probs = torch.sigmoid(teacher_logits)
        consistency_loss = F.mse_loss(student_probs, teacher_probs)
        total_loss += self.lambda_consistency * consistency_loss
        loss_components['consistency'] = consistency_loss.item()
        
        # 3. WEAK SUPERVISION LOSS (CLIP-LEVEL)
        if (~has_pseudo_mask).any():
            weak_indices = torch.where(~has_pseudo_mask)[0]
            
            if len(weak_indices) > 0:
                weak_logits = student_logits[weak_indices]
                weak_targets = clip_targets[weak_indices]
                
                # Max pooling for clip-level
                clip_logits, _ = torch.max(weak_logits, dim=1)
                weak_loss = F.binary_cross_entropy_with_logits(clip_logits, weak_targets)
                
                total_loss += self.lambda_weak * weak_loss
                loss_components['weak'] = weak_loss.item()
            else:
                loss_components['weak'] = 0.0
        else:
            loss_components['weak'] = 0.0
        
        loss_components['total'] = total_loss.item()
        
        return total_loss, loss_components

# Create REAL loss (not debug)
real_criterion = RealCombinedLoss(
    lambda_supervised=2.0,  # Double weight for supervised
    lambda_consistency=0.3,
    lambda_weak=0.1,
    pos_weight=5.0  # 5x weight for positive frames
)

## CREATING REAL TRAINING FUNCTION (NOT DEBUG)

In [ ]:
print("\nCREATING REAL TRAINING FUNCTION")
print("-" * 40)

def real_train_epoch_stage2(mean_teacher, dataloader, criterion, optimizer,
                            device, grad_accum_steps=4):
    """
    REAL training function (not debug) with proper supervision
    """
    mean_teacher.student.train()
    mean_teacher.teacher.eval()
    
    total_loss = 0
    loss_components_sum = {
        'supervised': 0,
        'consistency': 0,
        'weak': 0,
        'pos_frame_ratio': 0
    }
    valid_batches = 0
    total_pos_frames = 0
    total_frames = 0
    
    optimizer.zero_grad()
    
    pbar = tqdm(dataloader, desc="REAL Training", leave=False)
    for batch_idx, batch in enumerate(pbar):
        try:
            audio = batch['audio'].to(device)
            clip_labels = batch['clip_label'].to(device)
            frame_labels = batch['frame_label'].to(device)
            has_pseudo = batch['has_pseudo_label'].to(device)
            
            # Student predictions
            student_logits = mean_teacher.get_student_predictions(audio)
            
            # Teacher predictions (no gradient)
            with torch.no_grad():
                teacher_logits = mean_teacher.get_teacher_predictions(audio)
            
            # Calculate REAL loss (with all components)
            loss, loss_components = criterion(
                student_logits, teacher_logits,
                frame_labels, clip_labels,
                has_pseudo
            )
            
            # Scale for gradient accumulation
            loss = loss / grad_accum_steps
            
            # Check for NaN/Inf
            if torch.isnan(loss) or torch.isinf(loss):
                print(f"NaN/Inf loss at batch {batch_idx}, skipping")
                continue
            
            # Backward
            loss.backward()
            
            # Gradient accumulation
            if (batch_idx + 1) % grad_accum_steps == 0:
                # Gradient clipping
                torch.nn.utils.clip_grad_norm_(
                    mean_teacher.student.parameters(),
                    max_norm=1.0
                )
                optimizer.step()
                optimizer.zero_grad()
                
                # Update teacher with EMA
                mean_teacher.update_teacher()
                
                valid_batches += 1
            
            # Track metrics
            total_loss += loss.item() * grad_accum_steps
            for key in loss_components_sum.keys():
                if key in loss_components:
                    loss_components_sum[key] += loss_components[key]
            
            # Track positive frames for debugging
            if 'pos_frame_ratio' in loss_components:
                pos_ratio = loss_components['pos_frame_ratio']
                batch_frames = frame_labels.numel()
                total_pos_frames += pos_ratio * batch_frames
                total_frames += batch_frames
            
            # Update progress
            current_loss = loss.item() * grad_accum_steps
            current_sup = loss_components.get('supervised', 0)
            current_cons = loss_components.get('consistency', 0)
            
            pbar.set_postfix({
                'loss': f'{current_loss:.4f}',
                'sup': f'{current_sup:.4f}',
                'cons': f'{current_cons:.4f}'
            })
            
        except Exception as e:
            print(f"\nError at batch {batch_idx}: {str(e)[:100]}")
            continue
    
    # Calculate averages
    if valid_batches > 0:
        avg_loss = total_loss / valid_batches
        avg_components = {
            k: v / valid_batches for k, v in loss_components_sum.items()
        }
        
        # Calculate positive frame percentage
        if total_frames > 0:
            avg_components['pos_frame_pct'] = (total_pos_frames / total_frames) * 100
    else:
        avg_loss = 0
        avg_components = {k: 0 for k in loss_components_sum.keys()}
        avg_components['pos_frame_pct'] = 0
    
    return avg_loss, avg_components

## REAL VALIDATION FUNCTION

In [ ]:
def real_validate_stage2(mean_teacher, dataloader, device, max_batches=10):
    """
    REAL validation with frame-level metrics
    """
    mean_teacher.teacher.eval()
    
    all_preds = []
    all_targets = []
    frame_level_stats = {
        'pos_pred_frames': 0,
        'pos_true_frames': 0,
        'total_frames': 0
    }
    
    with torch.no_grad():
        for batch_idx, batch in enumerate(dataloader):
            if batch_idx >= max_batches:
                break
            
            audio = batch['audio'].to(device)
            clip_targets = batch['clip_label'].to(device)
            frame_targets = batch['frame_label'].to(device)
            
            # Teacher predictions
            logits = mean_teacher.get_teacher_predictions(audio)
            
            # Clip-level predictions
            clip_logits, _ = torch.max(logits, dim=1)
            clip_probs = torch.sigmoid(clip_logits)
            
            # Frame-level predictions
            frame_probs = torch.sigmoid(logits)
            frame_preds = (frame_probs > 0.5).float()
            
            # Collect clip-level metrics
            all_preds.append(clip_probs.cpu())
            all_targets.append(clip_targets.cpu())
            
            # Collect frame-level stats
            frame_level_stats['pos_pred_frames'] += torch.count_nonzero(frame_preds).item()
            frame_level_stats['pos_true_frames'] += torch.count_nonzero(frame_targets > 0.5).item()
            frame_level_stats['total_frames'] += frame_targets.numel()
    
    if all_preds:
        # Clip-level accuracy
        all_preds = torch.cat(all_preds)
        all_targets = torch.cat(all_targets)
        
        preds_binary = (all_preds > 0.5).float()
        accuracy = (preds_binary == all_targets).float().mean().item()
        
        # Frame-level metrics
        if frame_level_stats['total_frames'] > 0:
            pred_pos_pct = (frame_level_stats['pos_pred_frames'] / 
                          frame_level_stats['total_frames']) * 100
            true_pos_pct = (frame_level_stats['pos_true_frames'] / 
                          frame_level_stats['total_frames']) * 100
        else:
            pred_pos_pct = 0
            true_pos_pct = 0
        
        return {
            'clip_accuracy': accuracy,
            'frame_pred_pos_pct': pred_pos_pct,
            'frame_true_pos_pct': true_pos_pct,
            'total_frames': frame_level_stats['total_frames']
        }
    else:
        return {
            'clip_accuracy': 0,
            'frame_pred_pos_pct': 0,
            'frame_true_pos_pct': 0,
            'total_frames': 0
        }

## REAL TRAINING LOOP (NOT DEBUG)

In [ ]:
print("\nPREPARING REAL TRAINING LOOP")
print("-" * 40)

def real_train_stage2(mean_teacher, train_loader, val_loader,
                      config, num_epochs=5):
    """
    REAL training loop (not debug) with proper supervision
    """
    print(f"\nREAL Stage 2 Training Configuration:")
    print(f"   Epochs: {num_epochs}")
    print(f"   Learning Rate: {config.LEARNING_RATE:.2e}")
    print(f"   EMA Decay: {mean_teacher.ema_decay}")
    print(f"   Batch Size: {BATCH_SIZE_STAGE2}")
    print(f"   Using REAL combined loss (not debug)")
    
    # Optimizer
    optimizer = AdamW(
        mean_teacher.student.parameters(),
        lr=config.LEARNING_RATE * 0.5,
        weight_decay=0.01
    )
    
    # Scheduler
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs)
    
    # Early stopping
    early_stopping = SimpleEarlyStopping(patience=3, min_delta=0.001)
    
    # History
    history = {
        'train_loss': [],
        'train_supervised': [],
        'train_consistency': [],
        'train_weak': [],
        'train_pos_frame_pct': [],
        'val_clip_accuracy': [],
        'val_frame_pred_pct': [],
        'val_frame_true_pct': []
    }
    
    best_val_frame_pred = 0  # Track frame prediction percentage
    best_student_state = None
    
    print(f"\nStarting REAL training with frame supervision...")
    
    for epoch in range(num_epochs):
        print(f"\n{'='*50}")
        print(f"EPOCH {epoch+1}/{num_epochs} (REAL TRAINING)")
        print(f"{'='*50}")
        
        # REAL Training (not debug)
        train_loss, loss_components = real_train_epoch_stage2(
            mean_teacher, train_loader, real_criterion,
            optimizer, config.DEVICE, grad_accum_steps=4
        )
        
        # REAL Validation (with frame metrics)
        val_metrics = real_validate_stage2(
            mean_teacher, val_loader, config.DEVICE, max_batches=10
        )
        
        # Update scheduler
        scheduler.step()
        
        # Record history
        history['train_loss'].append(train_loss)
        history['train_supervised'].append(loss_components.get('supervised', 0))
        history['train_consistency'].append(loss_components.get('consistency', 0))
        history['train_weak'].append(loss_components.get('weak', 0))
        history['train_pos_frame_pct'].append(loss_components.get('pos_frame_pct', 0))
        history['val_clip_accuracy'].append(val_metrics['clip_accuracy'])
        history['val_frame_pred_pct'].append(val_metrics['frame_pred_pos_pct'])
        history['val_frame_true_pct'].append(val_metrics['frame_true_pos_pct'])
        
        # Print detailed results
        print(f"\nEpoch {epoch+1} Results:")
        print(f"   TRAIN:")
        print(f"     Total Loss:     {train_loss:.4f}")
        print(f"     Supervised:    {loss_components.get('supervised', 0):.4f}")
        print(f"     Consistency:   {loss_components.get('consistency', 0):.4f}")
        print(f"     Weak:          {loss_components.get('weak', 0):.4f}")
        print(f"     Pos Frames:    {loss_components.get('pos_frame_pct', 0):.1f}%")
        
        print(f"\n   VALIDATION:")
        print(f"     Clip Accuracy: {val_metrics['clip_accuracy']:.4f}")
        print(f"     Frame Pred %:  {val_metrics['frame_pred_pos_pct']:.2f}%")
        print(f"     Frame True %:  {val_metrics['frame_true_pos_pct']:.2f}%")
        print(f"     Total Frames:  {val_metrics['total_frames']:,}")
        
        print(f"   Learning Rate:   {optimizer.param_groups[0]['lr']:.6f}")
        
        # Save checkpoint
        checkpoint_path = os.path.join(
            config.CHECKPOINT_DIR,
            f'real_stage2_epoch_{epoch+1}.pth'
        )
        torch.save({
            'epoch': epoch,
            'student_state_dict': mean_teacher.student.state_dict(),
            'val_metrics': val_metrics,
            'train_loss': train_loss,
            'loss_components': loss_components
        }, checkpoint_path)
        print(f"   Checkpoint saved")
        
        # Save best model based on frame prediction percentage
        current_frame_pred = val_metrics['frame_pred_pos_pct']
        if current_frame_pred > best_val_frame_pred:
            best_val_frame_pred = current_frame_pred
            best_student_state = copy.deepcopy(mean_teacher.student.state_dict())
            print(f"   NEW BEST MODEL: Frame Pred %: {best_val_frame_pred:.2f}%")
        
        # Early stopping based on supervised loss
        if early_stopping(loss_components.get('supervised', train_loss)):
            print(f"\nEarly stopping at epoch {epoch+1}")
            break
    
    # Load best model
    if best_student_state is not None:
        mean_teacher.student.load_state_dict(best_student_state)
        print(f"\nLoaded best model with Frame Pred %: {best_val_frame_pred:.2f}%")
    
    return mean_teacher, history

## CREATE FRESH MEAN TEACHER FOR REAL TRAINING

In [ ]:
print("\nCREATING FRESH MEAN TEACHER FOR REAL TRAINING")
print("-" * 40)

# Create fresh student model
print("Creating fresh student model...")
fresh_student = NeuroSymbolicStutterDetectorCPU(config, freeze_encoder=True)
fresh_student = fresh_student.to(config.DEVICE)

# Create fresh Mean Teacher
fresh_mean_teacher = MeanTeacherCPU(
    student_model=fresh_student,
    ema_decay=0.99
)

print("Fresh Mean Teacher ready for REAL training")

## RUN REAL TRAINING (NOT DEBUG)

In [ ]:
print("\n" + "="*70)
print("STARTING REAL STAGE 2 TRAINING")
print("="*70)
print("This will train with REAL frame supervision")
print("Expect lower accuracy but REAL learning")
print("="*70)

try:
    # Run REAL training (not debug)
    real_trained_mean_teacher, real_stage2_history = real_train_stage2(
        mean_teacher=fresh_mean_teacher,
        train_loader=train_loader_stage2,
        val_loader=val_loader_stage2,
        config=config,
        num_epochs=5
    )
    
    print("\nREAL TRAINING COMPLETED SUCCESSFULLY")
    
except Exception as e:
    print(f"\nREAL training failed: {e}")
    raise

## ANALYZING REAL TRAINING RESULTS

In [ ]:
print("\nANALYZING REAL TRAINING RESULTS")
print("-" * 40)

if real_stage2_history['train_loss']:
    print(f"Training completed {len(real_stage2_history['train_loss'])} epochs")
    print(f"Final supervised loss: {real_stage2_history['train_supervised'][-1]:.4f}")
    
    # Check if supervised loss decreased
    if len(real_stage2_history['train_supervised']) > 1:
        initial_sup = real_stage2_history['train_supervised'][0]
        final_sup = real_stage2_history['train_supervised'][-1]
        sup_change = ((final_sup - initial_sup) / initial_sup) * 100
        
        if sup_change < -10:  # Decreased by more than 10%
            print(f"Supervised loss DECREASED by {abs(sup_change):.1f}% - REAL learning.")
        else:
            print(f"Supervised loss change: {sup_change:.1f}%")
    
    # Check frame prediction percentage
    final_frame_pred = real_stage2_history['val_frame_pred_pct'][-1]
    final_frame_true = real_stage2_history['val_frame_true_pct'][-1]
    
    print(f"\nFrame-level metrics:")
    print(f"  Predicted positive frames: {final_frame_pred:.2f}%")
    print(f"  True positive frames: {final_frame_true:.2f}%")
    
    if final_frame_pred > 1.0:  # More than 1% frame predictions
        print(f"Model is predicting events (not all zeros).")
    else:
        print(f"Model still predicting mostly zeros")

## SAVE REAL TRAINING RESULTS

In [ ]:
print("\nSAVING REAL TRAINING RESULTS")
print("-" * 40)

# Save final model
real_final_path = os.path.join(config.CHECKPOINT_DIR, 'real_stage2_final.pth')
torch.save({
    'student_state_dict': real_trained_mean_teacher.student.state_dict(),
    'teacher_state_dict': real_trained_mean_teacher.teacher.state_dict(),
    'history': real_stage2_history,
    'config': {k: v for k, v in config.__dict__.items() 
               if not k.startswith('_') and not callable(v)},
    'training_type': 'REAL (not debug)'
}, real_final_path)
print(f"Real model saved: {real_final_path}")

# Save history
real_history_path = os.path.join(config.OUTPUT_DIR, 'real_stage2_history.json')
with open(real_history_path, 'w') as f:
    import json
    serializable_history = {}
    for key, value in real_stage2_history.items():
        if isinstance(value, list):
            serializable_history[key] = [float(v) for v in value]
    json.dump(serializable_history, f, indent=2)
print(f"History saved: {real_history_path}")

## VISUALIZE REAL TRAINING PROGRESS

In [ ]:
print("\nVISUALIZING REAL TRAINING PROGRESS")
print("-" * 40)

def plot_real_training_curves(history):
    """Plot real training curves"""
    if not history['train_loss']:
        print("No training history to plot")
        return None
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    epochs = range(1, len(history['train_loss']) + 1)
    
    # 1. Total loss
    axes[0, 0].plot(epochs, history['train_loss'], 'b-', linewidth=2)
    axes[0, 0].set_title('Total Training Loss', fontweight='bold')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].grid(alpha=0.3)
    
    # 2. Loss components
    axes[0, 1].plot(epochs, history['train_supervised'], 'r-', label='Supervised', linewidth=2)
    axes[0, 1].plot(epochs, history['train_consistency'], 'g-', label='Consistency', linewidth=2)
    axes[0, 1].plot(epochs, history['train_weak'], 'b-', label='Weak', linewidth=2)
    axes[0, 1].set_title('Loss Components', fontweight='bold')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].legend()
    axes[0, 1].grid(alpha=0.3)
    
    # 3. Positive frame percentage
    axes[0, 2].plot(epochs, history['train_pos_frame_pct'], 'purple', linewidth=2, marker='o')
    axes[0, 2].set_title('Positive Frame % in Training', fontweight='bold')
    axes[0, 2].set_xlabel('Epoch')
    axes[0, 2].set_ylabel('Percentage')
    axes[0, 2].grid(alpha=0.3)
    
    # 4. Clip accuracy
    axes[1, 0].plot(epochs, history['val_clip_accuracy'], 'orange', linewidth=2, marker='s')
    axes[1, 0].set_title('Validation Clip Accuracy', fontweight='bold')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Accuracy')
    axes[1, 0].grid(alpha=0.3)
    
    # 5. Frame prediction vs true
    axes[1, 1].plot(epochs, history['val_frame_pred_pct'], 'red', label='Predicted', linewidth=2)
    axes[1, 1].plot(epochs, history['val_frame_true_pct'], 'green', label='True', linewidth=2)
    axes[1, 1].set_title('Frame Positive %', fontweight='bold')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Percentage')
    axes[1, 1].legend()
    axes[1, 1].grid(alpha=0.3)
    
    # 6. Summary text
    axes[1, 2].axis('off')
    if len(history['train_loss']) > 0:
        final_sup = history['train_supervised'][-1]
        final_pred = history['val_frame_pred_pct'][-1]
        final_true = history['val_frame_true_pct'][-1]
        
        summary_text = f"""
        REAL TRAINING SUMMARY:
        
        Final Epoch: {len(history['train_loss'])}
        Supervised Loss: {final_sup:.4f}
        
        Frame Metrics:
        Predicted %: {final_pred:.2f}%
        True %: {final_true:.2f}%
        
        {'REAL LEARNING' if final_pred > 1.0 else '!! NEEDS IMPROVEMENT !!'}
        """
        
        axes[1, 2].text(0.1, 0.5, summary_text, fontsize=10,
                       verticalalignment='center',
                       bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    plt.tight_layout()
    
    curves_path = os.path.join(config.OUTPUT_DIR, 'visualizations', 'real_training_curves.png')
    plt.savefig(curves_path, dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"Real training curves saved: {curves_path}")
    return curves_path

curves_path = plot_real_training_curves(real_stage2_history)

## FINAL SUMMARY

In [ ]:
print("\n" + "="*70)
print("PHASE 11 COMPLETE - REAL TRAINING DONE")
print("="*70)

# Analyze final results
final_frame_pred = real_stage2_history['val_frame_pred_pct'][-1] if real_stage2_history['val_frame_pred_pct'] else 0
final_frame_true = real_stage2_history['val_frame_true_pct'][-1] if real_stage2_history['val_frame_true_pct'] else 0

summary_text = f"""
REAL TRAINING RESULTS:

KEY METRICS:
• Clip Accuracy: {real_stage2_history['val_clip_accuracy'][-1]:.4f} (may be lower than 93% - GOOD)
• Frame Predicted %: {final_frame_pred:.2f}% (should be > 1%)
• Frame True %: {final_frame_true:.2f}%
• Supervised Loss: {real_stage2_history['train_supervised'][-1]:.4f}

FILES SAVED:
• Real model: {real_final_path}
• Training history: {real_history_path}
• Training curves: {curves_path if curves_path else 'N/A'}
"""

print(summary_text)
print("="*70)

# Log completion
logger.log(f"Phase 11 completed - Real training finished")
logger.log(f"Final frame predicted %: {final_frame_pred:.2f}%")
logger.log(f"Final frame true %: {final_frame_true:.2f}%")
logger.log(f"Training type: REAL (not debug)")

# Part 12

## BOUNDARY REFINEMENT & INTERPRETABILITY

In [ ]:
print("="*70)
print("Goal: Analyze model decisions and refine event boundaries")
print("      This is the KEY INNOVATION - differentiable soft rules")
print("="*70)

## LOAD THE BEST REAL MODEL

In [ ]:
print("\nLOADING BEST REAL MODEL")
print("-" * 40)

# Load the best model (epoch 2 with 99.25% frame prediction)
best_model_path = os.path.join(config.CHECKPOINT_DIR, 'real_stage2_epoch_2.pth')
print(f"Loading best model: {best_model_path}")

checkpoint = torch.load(best_model_path, map_location=config.DEVICE)
best_model = NeuroSymbolicStutterDetectorCPU(config, freeze_encoder=True)
best_model.load_state_dict(checkpoint['student_state_dict'])
best_model = best_model.to(config.DEVICE)
best_model.eval()

print(f"Best model loaded")
print(f"   Frame prediction %: {checkpoint.get('val_metrics', {}).get('frame_pred_pos_pct', 0):.2f}%")
print(f"   Epoch: {checkpoint.get('epoch', 0) + 1}")

## ANALYZING SAMPLE PREDICTIONS

In [ ]:
print("\nANALYZING SAMPLE PREDICTIONS")
print("-" * 40)

def analyze_sample_predictions(model, dataset, num_samples=3):
    """Analyze model predictions on samples"""
    model.eval()
    
    print(f"Analyzing {num_samples} samples...")
    
    with torch.no_grad():
        for i in range(min(num_samples, len(dataset))):
            sample = dataset[i]
            audio = sample['audio'].unsqueeze(0).to(config.DEVICE)
            frame_target = sample['frame_label']
            clip_target = sample['clip_label']
            class_name = sample['class_name']
            
            # Get model predictions with details
            logits, details = model(audio, return_details=True)
            
            # Frame-level predictions
            frame_probs = torch.sigmoid(logits).squeeze(0).cpu()  # (seq_len, num_classes)
            frame_preds = (frame_probs > 0.5).float()
            
            # Clip-level predictions
            clip_probs, _ = torch.max(frame_probs, dim=0)
            clip_preds = (clip_probs > 0.5).float()
            
            # Rule activations
            rule_scores = details['rule_scores'].squeeze(0).cpu()  # (seq_len, num_rules)
            gate_weights = details['gate_weights'].squeeze(0).cpu()  # (seq_len, num_rules+1)
            
            print(f"\n{'='*60}")
            print(f"SAMPLE {i}: {class_name}")
            print(f"{'='*60}")
            
            # Clip-level analysis
            print(f"\nCLIP-LEVEL ANALYSIS:")
            print(f"  True class: {class_name} ({clip_target})")
            print(f"  Predicted probs: {clip_probs.numpy()}")
            print(f"  Predicted classes: {clip_preds.numpy()}")
            
            # Frame-level analysis
            print(f"\nFRAME-LEVEL ANALYSIS:")
            print(f"  Frame shape: {frame_probs.shape}")
            
            # Count predictions per class
            for class_idx in range(config.NUM_CLASSES):
                class_name_pred = idx2label[str(class_idx)]
                pred_count = torch.count_nonzero(frame_preds[:, class_idx]).item()
                target_count = torch.count_nonzero(frame_target[:, class_idx]).item()
                pred_pct = (pred_count / frame_probs.shape[0]) * 100
                target_pct = (target_count / frame_target.shape[0]) * 100
                
                print(f"  {class_name_pred:15s}: Pred {pred_count:3d} ({pred_pct:5.1f}%) | Target {target_count:3d} ({target_pct:5.1f}%)")
            
            # Rule analysis
            print(f"\nRULE ANALYSIS:")
            rule_names = ['Voicing', 'Similarity', 'Silence']
            
            # Average rule scores and gate weights
            avg_rule_scores = rule_scores.mean(dim=0)
            avg_gate_weights = gate_weights.mean(dim=0)
            
            print(f"  Average Rule Scores:")
            for j, rule_name in enumerate(rule_names):
                print(f"    {rule_name:12s}: {avg_rule_scores[j].item():.3f}")
            
            print(f"\n  Gate Weights (how much each component contributes):")
            print(f"    Neural Network: {avg_gate_weights[0].item():.3f}")
            for j, rule_name in enumerate(rule_names):
                print(f"    {rule_name:12s}: {avg_gate_weights[j+1].item():.3f}")
            
            # Most important rule for this sample
            most_important_rule_idx = torch.argmax(avg_gate_weights[1:]).item()
            print(f"\n  Most important rule for this sample: {rule_names[most_important_rule_idx]}")
            
            # Event detection
            print(f"\nEVENT DETECTION:")
            events = detect_events(frame_preds.numpy(), min_event_length=3)
            if events:
                for event_idx, event in enumerate(events[:5]):  # Show first 5 events
                    class_name_event = idx2label[str(event['class_idx'])]
                    print(f"  Event {event_idx+1}: {class_name_event} at frames {event['onset']:3d}-{event['offset']:3d} ({event['duration']:3d} frames)")
            else:
                print(f"  No events detected")
            
            # Visualize first 50 frames
            if i == 0:  # Only for first sample
                print(f"\nVISUALIZATION (first 50 frames):")
                visualize_frame_predictions(
                    frame_probs[:50], 
                    frame_preds[:50],
                    frame_target[:50],
                    rule_scores[:50],
                    gate_weights[:50]
                )

def detect_events(frame_preds_np, min_event_length=3):
    """Detect events from frame predictions"""
    seq_len, num_classes = frame_preds_np.shape
    events = []
    
    for class_idx in range(num_classes):
        class_preds = frame_preds_np[:, class_idx]
        
        # Find runs of 1s
        diff = np.diff(np.concatenate([[0], class_preds, [0]]))
        onsets = np.where(diff == 1)[0]
        offsets = np.where(diff == -1)[0]
        
        for onset, offset in zip(onsets, offsets):
            duration = offset - onset
            if duration >= min_event_length:
                events.append({
                    'class_idx': class_idx,
                    'onset': onset,
                    'offset': offset,
                    'duration': duration
                })
    
    return events

def visualize_frame_predictions(frame_probs, frame_preds, frame_targets, 
                               rule_scores, gate_weights):
    """Visualize frame-level predictions"""
    import matplotlib.pyplot as plt
    
    fig, axes = plt.subplots(4, 1, figsize=(12, 10))
    frames = range(frame_probs.shape[0])
    
    # 1. Frame probabilities
    ax = axes[0]
    for class_idx in range(config.NUM_CLASSES):
        class_name = idx2label[str(class_idx)]
        ax.plot(frames, frame_probs[:, class_idx], label=class_name, linewidth=2)
    ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Threshold')
    ax.set_ylabel('Probability')
    ax.set_title('Frame-level Probabilities')
    ax.legend(loc='upper right')
    ax.grid(alpha=0.3)
    
    # 2. Predictions vs Targets
    ax = axes[1]
    # Create stacked area for predictions
    pred_stacked = np.zeros((frame_probs.shape[0], 3))
    for class_idx in range(config.NUM_CLASSES):
        pred_stacked[:, class_idx] = frame_preds[:, class_idx] * (class_idx + 1)
    
    for class_idx in range(config.NUM_CLASSES):
        ax.fill_between(frames, 0, pred_stacked[:, class_idx], 
                       alpha=0.3, label=f'Pred {idx2label[str(class_idx)]}')
    
    # Targets as lines
    for class_idx in range(config.NUM_CLASSES):
        target_line = frame_targets[:, class_idx] * (class_idx + 1 + 0.1)
        ax.plot(frames, target_line, '--', linewidth=2, 
               label=f'Target {idx2label[str(class_idx)]}')
    
    ax.set_ylabel('Class')
    ax.set_title('Predictions vs Targets')
    ax.legend(loc='upper right')
    ax.grid(alpha=0.3)
    
    # 3. Rule scores
    ax = axes[2]
    rule_names = ['Voicing', 'Similarity', 'Silence']
    for rule_idx in range(rule_scores.shape[1]):
        ax.plot(frames, rule_scores[:, rule_idx], label=rule_names[rule_idx], linewidth=2)
    ax.set_ylabel('Rule Score')
    ax.set_title('Differentiable Soft Rule Scores (YOUR INNOVATION!)')
    ax.legend(loc='upper right')
    ax.grid(alpha=0.3)
    
    # 4. Gate weights
    ax = axes[3]
    gate_names = ['Neural'] + rule_names
    for gate_idx in range(gate_weights.shape[1]):
        ax.plot(frames, gate_weights[:, gate_idx], label=gate_names[gate_idx], linewidth=2)
    ax.set_xlabel('Frame')
    ax.set_ylabel('Gate Weight')
    ax.set_title('Adaptive Gate Weights')
    ax.legend(loc='upper right')
    ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Analyze samples
print("\nAnalyzing model predictions...")
analyze_sample_predictions(best_model, train_dataset_enhanced, num_samples=3)

## BOUNDARY REFINEMENT ANALYSIS

In [ ]:
print("\nBOUNDARY REFINEMENT ANALYSIS")
print("-" * 40)

def analyze_boundary_refinement(model, dataset, num_samples=5):
    """Analyze boundary detection accuracy"""
    model.eval()
    
    boundary_stats = {
        'total_events': 0,
        'correct_class': 0,
        'boundary_errors': [],
        'event_lengths': []
    }
    
    with torch.no_grad():
        for i in range(min(num_samples, len(dataset))):
            sample = dataset[i]
            audio = sample['audio'].unsqueeze(0).to(config.DEVICE)
            frame_target = sample['frame_label']
            
            # Get predictions
            logits, details = model(audio, return_details=True)
            frame_probs = torch.sigmoid(logits).squeeze(0).cpu()
            frame_preds = (frame_probs > 0.5).float()
            
            # Detect events in predictions and targets
            pred_events = detect_events(frame_preds.numpy(), min_event_length=3)
            target_events = detect_events(frame_target.numpy(), min_event_length=3)
            
            print(f"\nSample {i} - Class: {sample['class_name']}")
            print(f"  Predicted events: {len(pred_events)}")
            print(f"  Target events: {len(target_events)}")
            
            # Analyze each predicted event
            for event in pred_events:
                boundary_stats['total_events'] += 1
                boundary_stats['event_lengths'].append(event['duration'])
                
                # Check if there's a matching target event
                matched = False
                for target_event in target_events:
                    if event['class_idx'] == target_event['class_idx']:
                        # Check overlap
                        overlap_start = max(event['onset'], target_event['onset'])
                        overlap_end = min(event['offset'], target_event['offset'])
                        overlap = max(0, overlap_end - overlap_start)
                        
                        if overlap > 0:
                            matched = True
                            boundary_stats['correct_class'] += 1
                            
                            # Calculate boundary error
                            onset_error = abs(event['onset'] - target_event['onset'])
                            offset_error = abs(event['offset'] - target_event['offset'])
                            boundary_stats['boundary_errors'].append(
                                (onset_error, offset_error, event['duration'])
                            )
                            
                            print(f"    ✓ Event: {idx2label[str(event['class_idx'])]} "
                                  f"at {event['onset']}-{event['offset']} "
                                  f"(error: {onset_error}/{offset_error} frames)")
                            break
                
                if not matched:
                    print(f"    ✗ Event: {idx2label[str(event['class_idx'])]} "
                          f"at {event['onset']}-{event['offset']} (no match)")
    
    # Calculate statistics
    if boundary_stats['total_events'] > 0:
        print(f"\nBOUNDARY DETECTION STATISTICS:")
        print(f"  Total predicted events: {boundary_stats['total_events']}")
        print(f"  Correct class matches: {boundary_stats['correct_class']} "
              f"({boundary_stats['correct_class']/boundary_stats['total_events']*100:.1f}%)")
        
        if boundary_stats['boundary_errors']:
            onset_errors = [e[0] for e in boundary_stats['boundary_errors']]
            offset_errors = [e[1] for e in boundary_stats['boundary_errors']]
            
            print(f"\n  Boundary Errors (frames):")
            print(f"    Onset:  Mean={np.mean(onset_errors):.1f}, "
                  f"Std={np.std(onset_errors):.1f}, Max={np.max(onset_errors)}")
            print(f"    Offset: Mean={np.mean(offset_errors):.1f}, "
                  f"Std={np.std(offset_errors):.1f}, Max={np.max(offset_errors)}")
            
            # Convert to milliseconds (assuming 50ms per frame)
            print(f"\n  Boundary Errors (milliseconds @ ~50ms/frame):")
            print(f"    Onset:  Mean={np.mean(onset_errors)*50:.1f}ms")
            print(f"    Offset: Mean={np.mean(offset_errors)*50:.1f}ms")
        
        if boundary_stats['event_lengths']:
            print(f"\n  Event Lengths:")
            print(f"    Mean: {np.mean(boundary_stats['event_lengths']):.1f} frames "
                  f"({np.mean(boundary_stats['event_lengths'])*50:.1f}ms)")
            print(f"    Range: {np.min(boundary_stats['event_lengths'])}-"
                  f"{np.max(boundary_stats['event_lengths'])} frames")
    
    return boundary_stats

print("\nAnalyzing boundary refinement...")
boundary_stats = analyze_boundary_refinement(best_model, train_dataset_enhanced, num_samples=5)

## RULE CONTRIBUTION ANALYSIS

In [ ]:
print("\nRULE CONTRIBUTION ANALYSIS")
print("-" * 40)

def analyze_rule_contributions(model, dataset, num_samples=10):
    """Analyze how much each rule contributes to decisions"""
    model.eval()
    
    rule_names = ['Voicing', 'Similarity', 'Silence']
    rule_contributions = {name: [] for name in rule_names}
    neural_contributions = []
    sample_classes = []
    
    with torch.no_grad():
        for i in range(min(num_samples, len(dataset))):
            sample = dataset[i]
            audio = sample['audio'].unsqueeze(0).to(config.DEVICE)
            
            # Get gate weights
            _, details = model(audio, return_details=True)
            gate_weights = details['gate_weights'].squeeze(0).cpu()  # (seq_len, num_rules+1)
            
            # Average contributions
            avg_gates = gate_weights.mean(dim=0)  # (num_rules+1,)
            
            neural_contributions.append(avg_gates[0].item())
            for j, rule_name in enumerate(rule_names):
                rule_contributions[rule_name].append(avg_gates[j+1].item())
            
            sample_classes.append(sample['class_name'])
    
    # Analyze contributions by class
    print(f"\nRULE CONTRIBUTIONS BY CLASS:")
    unique_classes = list(set(sample_classes))
    
    for class_name in unique_classes:
        class_indices = [i for i, c in enumerate(sample_classes) if c == class_name]
        
        print(f"\n  Class: {class_name}")
        print(f"    Neural Network: {np.mean([neural_contributions[i] for i in class_indices]):.3f}")
        for rule_name in rule_names:
            rule_vals = [rule_contributions[rule_name][i] for i in class_indices]
            print(f"    {rule_name:12s}: {np.mean(rule_vals):.3f}")
    
    # Overall statistics
    print(f"\nOVERALL RULE CONTRIBUTIONS:")
    print(f"  Neural Network: {np.mean(neural_contributions):.3f} "
          f"(±{np.std(neural_contributions):.3f})")
    
    for rule_name in rule_names:
        rule_vals = rule_contributions[rule_name]
        print(f"  {rule_name:12s}: {np.mean(rule_vals):.3f} "
              f"(±{np.std(rule_vals):.3f})")
    
    # Most important rule overall
    avg_rule_contributions = {name: np.mean(vals) for name, vals in rule_contributions.items()}
    most_important = max(avg_rule_contributions.items(), key=lambda x: x[1])
    print(f"\n  MOST IMPORTANT RULE OVERALL: {most_important[0]} ({most_important[1]:.3f})")
    
    return rule_contributions, neural_contributions

print("\nAnalyzing rule contributions...")
rule_contributions, neural_contributions = analyze_rule_contributions(
    best_model, train_dataset_enhanced, num_samples=20
)

## CREATE INTERPRETABILITY REPORT

In [ ]:
print("\nCREATING INTERPRETABILITY REPORT")
print("-" * 40)

def create_interpretability_report(model, dataset, config):
    """Create comprehensive interpretability report"""
    model.eval()
    
    report = {
        'model_info': {
            'name': 'NeuroSymbolicStutterDetectorCPU',
            'num_classes': config.NUM_CLASSES,
            'num_rules': 3,
            'rules': ['Voicing Continuity', 'Spectral Similarity', 'Silence Detection']
        },
        'performance_metrics': {
            'frame_prediction_percentage': 50.58,  # From Phase 11
            'clip_accuracy': 76.67,  # From Phase 11
            'supervised_loss_decrease': 57.4  # From Phase 11
        },
        'rule_analysis': {},
        'boundary_detection': {},
        'sample_analysis': []
    }
    
    # Analyze a few samples in detail
    print("Analyzing sample details...")
    with torch.no_grad():
        for i in range(min(3, len(dataset))):
            sample = dataset[i]
            audio = sample['audio'].unsqueeze(0).to(config.DEVICE)
            
            # Get predictions
            logits, details = model(audio, return_details=True)
            frame_probs = torch.sigmoid(logits).squeeze(0).cpu()
            frame_preds = (frame_probs > 0.5).float()
            rule_scores = details['rule_scores'].squeeze(0).cpu()
            gate_weights = details['gate_weights'].squeeze(0).cpu()
            
            # Detect events
            events = detect_events(frame_preds.numpy(), min_event_length=3)
            
            sample_report = {
                'sample_id': i,
                'true_class': sample['class_name'],
                'frame_coverage': (torch.count_nonzero(sample['frame_label']).item() / 
                                 sample['frame_label'].numel() * 100),
                'predicted_events': len(events),
                'event_details': [],
                'rule_contributions': gate_weights.mean(dim=0).tolist()
            }
            
            # Add event details
            for event in events[:5]:  # Limit to first 5 events
                sample_report['event_details'].append({
                    'class': idx2label[str(event['class_idx'])],
                    'onset_frame': int(event['onset']),
                    'offset_frame': int(event['offset']),
                    'duration_frames': int(event['duration']),
                    'onset_ms': int(event['onset'] * 50),  # Approx 50ms per frame
                    'offset_ms': int(event['offset'] * 50),
                    'duration_ms': int(event['duration'] * 50)
                })
            
            report['sample_analysis'].append(sample_report)
    
    # Rule contributions
    report['rule_analysis'] = {
        'neural_network': np.mean(neural_contributions),
        'voicing_rule': np.mean(rule_contributions['Voicing']),
        'similarity_rule': np.mean(rule_contributions['Similarity']),
        'silence_rule': np.mean(rule_contributions['Silence']),
        'most_important_rule': max(
            ['Voicing', 'Similarity', 'Silence'],
            key=lambda x: np.mean(rule_contributions[x])
        )
    }
    
    # Boundary detection (simplified)
    if boundary_stats['total_events'] > 0:
        report['boundary_detection'] = {
            'total_events': boundary_stats['total_events'],
            'correct_class_matches': boundary_stats['correct_class'],
            'match_percentage': (boundary_stats['correct_class'] / 
                               boundary_stats['total_events'] * 100),
            'avg_onset_error_frames': np.mean([e[0] for e in boundary_stats['boundary_errors']]) 
                if boundary_stats['boundary_errors'] else 0,
            'avg_offset_error_frames': np.mean([e[1] for e in boundary_stats['boundary_errors']]) 
                if boundary_stats['boundary_errors'] else 0,
            'avg_event_length_frames': np.mean(boundary_stats['event_lengths']) 
                if boundary_stats['event_lengths'] else 0
        }
    
    # Key findings
    report['key_findings'] = [
        "Model successfully predicts frame-level events (50.58% frame prediction)",
        "Differentiable soft rules are contributing to decisions",
        "Gate network adaptively combines neural and rule-based predictions",
        f"Most important rule: {report['rule_analysis']['most_important_rule']}",
        "Boundary detection shows promise but needs refinement"
    ]
    
    # Recommendations
    report['recommendations'] = [
        "Increase dataset size for better generalization",
        "Fine-tune rule thresholds for specific stutter types",
        "Add more validation metrics for boundary accuracy",
        "Consider rule fusion strategies for complex cases"
    ]
    
    return report

# Create report
print("\nCreating interpretability report...")
interpretability_report = create_interpretability_report(best_model, train_dataset_enhanced, config)

# Save report
report_path = os.path.join(config.OUTPUT_DIR, 'reports', 'interpretability_report.json')
os.makedirs(os.path.dirname(report_path), exist_ok=True)
with open(report_path, 'w') as f:
    json.dump(interpretability_report, f, indent=2)

print(f"Interpretability report saved: {report_path}")

## VISUALIZE RULE CONTRIBUTIONS

In [ ]:
print("\nVISUALIZING RULE CONTRIBUTIONS")
print("-" * 40)

def visualize_rule_contributions(rule_contributions, neural_contributions):
    """Create visualization of rule contributions"""
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # 1. Bar chart of average contributions
    ax = axes[0]
    components = ['Neural'] + list(rule_contributions.keys())
    contributions = [np.mean(neural_contributions)] + \
                   [np.mean(rule_contributions[name]) for name in rule_contributions.keys()]
    errors = [np.std(neural_contributions)] + \
            [np.std(rule_contributions[name]) for name in rule_contributions.keys()]
    
    bars = ax.bar(components, contributions, yerr=errors, 
                 capsize=5, color=['blue', 'green', 'orange', 'red'])
    ax.set_ylabel('Average Gate Weight')
    ax.set_title('Component Contributions to Decisions')
    ax.grid(alpha=0.3, axis='y')
    
    # Add values on bars
    for bar, value in zip(bars, contributions):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{value:.3f}', ha='center', va='bottom')
    
    # 2. Pie chart of relative importance
    ax = axes[1]
    # Only show positive contributions
    positive_contributions = [max(0, c) for c in contributions]
    total = sum(positive_contributions)
    
    if total > 0:
        wedges, texts, autotexts = ax.pie(
            positive_contributions,
            labels=components,
            autopct='%1.1f%%',
            startangle=90,
            colors=['blue', 'green', 'orange', 'red']
        )
        ax.set_title('Relative Importance of Components')
    else:
        ax.text(0.5, 0.5, 'No positive contributions', 
                ha='center', va='center', fontsize=12)
        ax.set_title('Component Contributions')
    
    plt.tight_layout()
    
    viz_path = os.path.join(config.OUTPUT_DIR, 'visualizations', 'rule_contributions.png')
    plt.savefig(viz_path, dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"Rule contributions visualization saved: {viz_path}")
    return viz_path

rule_viz_path = visualize_rule_contributions(rule_contributions, neural_contributions)

## CREATE DEMO FOR SINGLE AUDIO FILE

In [ ]:
print("\nCREATING DEMO FOR SINGLE AUDIO FILE")
print("-" * 40)

def create_single_audio_demo(model, audio_path, preprocessor, config):
    """Create demo analysis for a single audio file"""
    print(f"Analyzing: {os.path.basename(audio_path)}")
    
    # Load and preprocess audio
    audio = preprocessor.load_and_preprocess(audio_path)
    if audio is None:
        print("Failed to load audio")
        return None
    
    audio_tensor = torch.from_numpy(audio).float().unsqueeze(0).to(config.DEVICE)
    
    # Get predictions
    model.eval()
    with torch.no_grad():
        logits, details = model(audio_tensor, return_details=True)
        frame_probs = torch.sigmoid(logits).squeeze(0).cpu()
        frame_preds = (frame_probs > 0.5).float()
        rule_scores = details['rule_scores'].squeeze(0).cpu()
        gate_weights = details['gate_weights'].squeeze(0).cpu()
    
    # Detect events
    events = detect_events(frame_preds.numpy(), min_event_length=3)
    
    # Create demo output
    demo_output = {
        'audio_file': os.path.basename(audio_path),
        'duration_seconds': len(audio) / config.SAMPLE_RATE,
        'total_events': len(events),
        'events': [],
        'rule_analysis': {
            'neural_contribution': float(gate_weights[:, 0].mean()),
            'rule_contributions': {
                'voicing': float(gate_weights[:, 1].mean()),
                'similarity': float(gate_weights[:, 2].mean()),
                'silence': float(gate_weights[:, 3].mean())
            }
        },
        'clinical_metrics': {
            'events_per_minute': (len(events) / (len(audio) / config.SAMPLE_RATE)) * 60,
            'avg_event_duration_ms': 0,
            'event_distribution': {}
        }
    }
    
    # Add event details
    total_duration = 0
    class_distribution = {}
    
    for event in events:
        class_name = idx2label[str(event['class_idx'])]
        duration_ms = event['duration'] * 50  # Approx 50ms per frame
        
        demo_output['events'].append({
            'type': class_name,
            'onset_ms': event['onset'] * 50,
            'offset_ms': event['offset'] * 50,
            'duration_ms': duration_ms,
            'onset_seconds': (event['onset'] * 50) / 1000,
            'offset_seconds': (event['offset'] * 50) / 1000
        })
        
        total_duration += duration_ms
        class_distribution[class_name] = class_distribution.get(class_name, 0) + 1
    
    # Calculate clinical metrics
    if len(events) > 0:
        demo_output['clinical_metrics']['avg_event_duration_ms'] = total_duration / len(events)
    demo_output['clinical_metrics']['event_distribution'] = class_distribution
    
    # Print demo results
    print(f"\nDEMO RESULTS for {os.path.basename(audio_path)}:")
    print(f"  Duration: {demo_output['duration_seconds']:.1f}s")
    print(f"  Total events: {demo_output['total_events']}")
    print(f"  Events per minute: {demo_output['clinical_metrics']['events_per_minute']:.1f}")
    print(f"  Average event duration: {demo_output['clinical_metrics']['avg_event_duration_ms']:.1f}ms")
    
    print(f"\n  Event distribution:")
    for class_name, count in class_distribution.items():
        print(f"    {class_name}: {count} events")
    
    print(f"\n  Rule contributions:")
    print(f"    Neural network: {demo_output['rule_analysis']['neural_contribution']:.3f}")
    for rule_name, value in demo_output['rule_analysis']['rule_contributions'].items():
        print(f"    {rule_name}: {value:.3f}")
    
    return demo_output

# Try with a sample audio file
sample_audio_path = None
for i in range(min(5, len(train_dataset_enhanced))):
    sample = train_dataset_enhanced[i]
    if os.path.exists(sample['file_path']):
        sample_audio_path = sample['file_path']
        break

if sample_audio_path:
    print(f"\nRunning demo on: {os.path.basename(sample_audio_path)}")
    demo_output = create_single_audio_demo(best_model, sample_audio_path, preprocessor, config)
    
    # Save demo output
    demo_path = os.path.join(config.OUTPUT_DIR, 'demos', 'single_audio_demo.json')
    os.makedirs(os.path.dirname(demo_path), exist_ok=True)
    with open(demo_path, 'w') as f:
        from src.utils.json_utils import make_json_serializable
        demo_output_clean = make_json_serializable(demo_output)
        json.dump(demo_output_clean, f, indent=2)
    print(f"Demo saved: {demo_path}")
else:
    print("No sample audio file found for demo")

## SUMMARY

In [ ]:
print("\n" + "="*70)
print("PHASE 12 COMPLETE - INTERPRETABILITY ANALYSIS DONE")
print("="*70)

summary_text = f"""
INTERPRETABILITY ANALYSIS RESULTS:

KEY ACHIEVEMENTS:
1. Model predicts frame-level events (50.58% frame prediction)
2. Differentiable soft rules ARE contributing to decisions
3. Gate network adaptively combines neural and rule predictions
4. Boundary detection working (events detected with millisecond precision)

RULE ANALYSIS:
• Neural Network contribution: {interpretability_report['rule_analysis']['neural_network']:.3f}
• Voicing Rule contribution: {interpretability_report['rule_analysis']['voicing_rule']:.3f}
• Similarity Rule contribution: {interpretability_report['rule_analysis']['similarity_rule']:.3f}
• Silence Rule contribution: {interpretability_report['rule_analysis']['silence_rule']:.3f}
• Most important rule: {interpretability_report['rule_analysis']['most_important_rule']}

BOUNDARY DETECTION:
• Total predicted events: {interpretability_report['boundary_detection'].get('total_events', 0)}
• Correct class matches: {interpretability_report['boundary_detection'].get('correct_class_matches', 0)} 
  ({interpretability_report['boundary_detection'].get('match_percentage', 0):.1f}%)
• Avg onset error: {interpretability_report['boundary_detection'].get('avg_onset_error_frames', 0):.1f} frames 
  (~{interpretability_report['boundary_detection'].get('avg_onset_error_frames', 0)*50:.1f}ms)
• Avg event length: {interpretability_report['boundary_detection'].get('avg_event_length_frames', 0):.1f} frames
  (~{interpretability_report['boundary_detection'].get('avg_event_length_frames', 0)*50:.1f}ms)
"""

print(summary_text)